In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'llm.py': '"""Greedy chat generation shared by the screening runs: vLLM (LLM_BACKEND=vllm, default) or transformers (LLM_BACKEND=hf).\n\n    m = LLM("Qwen/Qwen3-4B-Instruct-2507", max_len=16384, tp=1)\n    texts = m.gen([[{"role": "system", "content": ...}, {"role": "user", "content": ...}], ...], max_tokens=512)\n\nOne process loads one repo; trying the next candidate (fp16 -> AWQ, vllm -> hf) is multirun.py\'s job, in a new process.\nPrompts are tokenized here (chat template, enable_thinking off unless think=True) and passed as token ids, so no double BOS.\nA prompt longer than max_len - max_tokens comes back as None and is counted in m.overflow (never crashes the batch).\nText after </think> is kept when a model still thinks. m.ntok_in / m.ntok_out count tokens of every call.\n"""\nimport os\nimport time\n\n# ChatML, for tokenizers that ship no chat template (vilm/vinallama-2.7b-chat): the format its model card describes\nCHATML = ("{% for m in messages %}{{ \'<|im_start|>\' + m[\'role\'] + \'\\n\' + m[\'content\'] + \'<|im_end|>\\n\' }}{% endfor %}"\n          "{% if add_generation_prompt %}{{ \'<|im_start|>assistant\\n\' }}{% endif %}")\n\n\nclass LLM:\n    def __init__(self, repo, max_len=16384, tp=1, think=False, **kw):\n        self.repo, self.max_len, self.think = repo, max_len, think\n        self.overflow = self.ntok_in = self.ntok_out = 0\n        self.backend = os.environ.get("LLM_BACKEND", "vllm")\n        t0 = time.time()\n        if self.backend == "vllm":\n            from vllm import LLM as V\n            self.eng = V(model=repo, dtype="float16", max_model_len=max_len, tensor_parallel_size=tp, seed=0,\n                         gpu_memory_utilization=float(os.environ.get("GPU_UTIL", "0.90")), trust_remote_code=True, **kw)\n            self.tok = self.eng.get_tokenizer()\n        else:\n            import torch\n            import transformers\n            from transformers import AutoModelForCausalLM, AutoTokenizer\n            import transformers.integrations.sdpa_attention as sd\n            if hasattr(sd, "use_gqa_in_sdpa"):  # T4: enable_gqa only runs in the math kernel (heads x L x L scores, OOM): repeat K/V instead\n                sd.use_gqa_in_sdpa = lambda *a, **k: False\n            dkw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"\n            self.tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)\n            cuda = torch.cuda.is_available()\n            load = lambda remote: AutoModelForCausalLM.from_pretrained(repo, **{dkw: torch.float16 if cuda else torch.float32},\n                                                                       device_map="auto" if cuda else None, trust_remote_code=remote).eval()\n            try:\n                self.model = load(True)\n            except ImportError as e:  # repo code written for an older transformers (Phi-4-mini: LossKwargs); the built-in class works\n                print(f"[llm] remote code failed ({e}); loading the built-in transformers class", flush=True)\n                self.model = load(False)\n        if not getattr(self.tok, "chat_template", None):\n            self.tok.chat_template = CHATML\n            print(f"[llm] {repo}: no chat template, using ChatML", flush=True)\n        print(f"[llm] {repo} loaded with {self.backend} in {time.time() - t0:.0f}s", flush=True)\n\n    def ids(self, msgs):\n        kw = dict(tokenize=False, add_generation_prompt=True, enable_thinking=self.think)\n        try:\n            text = self.tok.apply_chat_template(msgs, **kw)\n        except Exception:  # templates without a system role: fold it into the first user turn\n            if msgs[0]["role"] != "system":\n                raise\n            text = self.tok.apply_chat_template([dict(msgs[1], content=msgs[0]["content"] + "\\n\\n" + msgs[1]["content"])] + msgs[2:], **kw)\n        if os.environ.get("STRIP_EMPTY_THINK") == "1":  # models fine-tuned by PIPE-T1-04 train.py on Qwen3.5 (no empty think block)\n            text = text.replace("<think>\\n\\n</think>\\n\\n", "")\n        return self.tok(text, add_special_tokens=False).input_ids\n\n    def gen(self, batch, max_tokens):\n        ids = [self.ids(m) for m in batch]\n        ok = [i for i, x in enumerate(ids) if len(x) <= self.max_len - max_tokens]\n        self.overflow += len(ids) - len(ok)\n        self.ntok_in += sum(len(ids[i]) for i in ok)\n        out = [None] * len(batch)\n        if self.backend == "vllm":\n            from vllm import SamplingParams\n            sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, seed=0)\n            res = self.eng.generate([{"prompt_token_ids": ids[i]} for i in ok], sp, use_tqdm=False)\n            for i, r in zip(ok, res):\n                out[i] = r.outputs[0].text\n                self.ntok_out += len(r.outputs[0].token_ids)\n        else:\n            import torch\n            bs = int(os.environ.get("HF_BATCH", "4"))\n            pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id\n            dev = next(self.model.parameters()).device\n            order = sorted(ok, key=lambda i: len(ids[i]))\n            for k in range(0, len(order), bs):\n                part = order[k:k + bs]\n                n = max(len(ids[i]) for i in part)\n                x = torch.tensor([[pad] * (n - len(ids[i])) + ids[i] for i in part], device=dev)\n                mask = torch.tensor([[0] * (n - len(ids[i])) + [1] * len(ids[i]) for i in part], device=dev)\n                with torch.no_grad():\n                    y = self.model.generate(input_ids=x, attention_mask=mask, max_new_tokens=max_tokens, do_sample=False, pad_token_id=pad)\n                for i, row in zip(part, y[:, n:].tolist()):\n                    row = [t for t in row if t != pad]\n                    self.ntok_out += len(row)\n                    out[i] = self.tok.decode(row, skip_special_tokens=True)\n        return [o.split("</think>")[-1].strip() if o is not None else None for o in out]\n',
    'multirun.py': '"""Run one worker process per model over the GPUs of a Kaggle T4 x2 box.\n\n    ONLY=tag,tag python -u multirun.py jobs.json worker.py [-- extra worker args]      (ONLY empty or unset: every job)\n\njobs.json: [{"tag": "q3-4b", "tp": 1, "env": {"HARNESS": "4"}, "cands": [{"repo": "...", "backend": "vllm", ...}, ...], ...}, ...]\n(env: extra environment of that job\'s worker processes)\nEach candidate runs as `python -u worker.py --job \'<job + candidate as json>\' extra...` in its own process (a failed vLLM\nengine leaves a dirty process); exit code 0 ends the job, anything else tries the next candidate (fp16 -> AWQ, vllm -> hf).\ntp 2 jobs run one at a time on both GPUs first; tp 1 jobs then run two at a time, one per GPU. No GPU: one at a time.\nAfter a candidate its Hugging Face cache folder is deleted (Kaggle disk) unless the next candidate uses the same repo, or KEEP_HF=1.\nEvery worker line is echoed with a [tag] prefix; a summary of which candidate won is written to multirun_summary.json.\n"""\nimport json\nimport os\nimport queue\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nLOCK = threading.Lock()\n\n\ndef log(msg):\n    with LOCK:\n        print(msg, flush=True)\n\n\ndef ngpu():\n    try:\n        return len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())\n    except FileNotFoundError:\n        return 0\n\n\ndef run_job(job, worker, extra, gpus):\n    for k, cand in enumerate(job["cands"]):\n        spec = dict(job, **cand, cand_index=k)\n        spec.pop("cands")\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus, PYTHONUNBUFFERED="1", LLM_BACKEND=cand.get("backend", "vllm"), **job.get("env", {}))\n        t0 = time.time()\n        log(f"[{job[\'tag\']}] start candidate {k}: {cand[\'repo\']} ({env[\'LLM_BACKEND\']}) on GPU \'{gpus}\'")\n        p = subprocess.Popen([sys.executable, "-u", worker, "--job", json.dumps(spec, ensure_ascii=False)] + extra,\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)\n        tail = []\n        for line in p.stdout:\n            line = line.rstrip()\n            tail = (tail + [line])[-15:]\n            if "it/s]" not in line and "s/it]" not in line:  # drop tqdm bars\n                log(f"[{job[\'tag\']}] {line}")\n        rc = p.wait()\n        nxt = job["cands"][k + 1]["repo"] if k + 1 < len(job["cands"]) else None\n        if os.environ.get("KEEP_HF") != "1" and (rc == 0 or nxt != cand["repo"]):  # keep the weights for a retry of the same repo\n            hub = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface")) / "hub"\n            shutil.rmtree(hub / ("models--" + cand["repo"].replace("/", "--")), ignore_errors=True)\n        log(f"[{job[\'tag\']}] candidate {k} rc {rc} after {time.time() - t0:.0f}s")\n        if rc == 0:\n            return dict(tag=job["tag"], repo=cand["repo"], cand_index=k, ok=True, sec=round(time.time() - t0))\n    return dict(tag=job["tag"], ok=False, tail=tail)\n\n\ndef main():\n    argv = sys.argv[1:]\n    extra = argv[argv.index("--") + 1:] if "--" in argv else []\n    argv = argv[:argv.index("--")] if "--" in argv else argv\n    jobs = json.load(open(argv[0], encoding="utf-8"))\n    worker = argv[1]\n    if os.environ.get("ONLY"):\n        keep = os.environ["ONLY"].split(",")\n        jobs = [j for j in jobs if j["tag"] in keep]\n    n = ngpu()\n    log(f"[multirun] {len(jobs)} jobs, {n} GPU(s): {[j[\'tag\'] for j in jobs]}")\n    done = []\n    if n < 2:\n        done = [run_job(j, worker, extra, "0" if n else "") for j in jobs]\n    else:\n        done = [run_job(j, worker, extra, "0,1") for j in jobs if j.get("tp", 1) > 1]\n        q = queue.Queue()\n        for j in jobs:\n            if j.get("tp", 1) == 1:\n                q.put(j)\n\n        def drain(g):\n            while True:\n                try:\n                    j = q.get_nowait()\n                except queue.Empty:\n                    return\n                done.append(run_job(j, worker, extra, g))\n\n        ts = [threading.Thread(target=drain, args=(str(g),)) for g in range(2)]\n        [t.start() for t in ts]\n        [t.join() for t in ts]\n    json.dump(done, open("multirun_summary.json", "w"), indent=1)\n    log("[multirun] summary: " + json.dumps([(d["tag"], d.get("repo"), d["ok"]) for d in done]))\n\n\nif __name__ == "__main__":\n    main()\n',
    'issue_worker.py': '"""GS-Issue worker: one local LLM finds the issues of each claim and proposes edits (same prompt for every model).\n\n    INPUTS=layer_a.jsonl,d_edit.jsonl python -u issue_worker.py --job \'{"tag": "q3-4b", "repo": "Qwen/Qwen3-4B-Instruct-2507", "tp": 1}\'\n\nRuns inside a Kaggle notebook built by tools/kaggle/build_nb.py with llm.py and multirun.py of experiments/AG-1.\nGreedy, thinking off. Input rows: {id, claim, evidence}. Output: out/<tag>__<input stem>.jsonl, one row per input row:\n{id, raw, parsed (or null)}; out/<tag>_summary.json. Env: LIMIT=n (first n rows of each input, smoke), MAX_TOKENS (900).\nThe prompt is the D-EDIT labelling guide (benchmark/gs_eval_v2/LABELING_GUIDE.md section 2), shortened; the taxonomy\nis benchmark/gs_issue/taxonomy.py (embedded next to this file).\n"""\nimport argparse\nimport json\nimport os\nimport re\nimport time\nfrom pathlib import Path\n\nfrom taxonomy import CODES, KINDS\n\nSYSTEM = "Bạn là kiểm toán viên soát xét tuyên bố môi trường trong báo cáo của doanh nghiệp Việt Nam. Bạn chỉ trả lời bằng JSON."\nPROMPT = """Đọc TUYÊN BỐ (và BẰNG CHỨNG nếu có). Tìm các vấn đề khiến tuyên bố có thể gây hiểu nhầm, rồi đề xuất sửa.\n\nMã vấn đề (chỉ dùng các mã này):\n{codes}\n\nBốn loại sửa được phép:\n{kinds}\n\nQuy tắc:\n- Chỉ nêu vấn đề có thật trong câu; câu đã đủ chỉ số, số, kỳ, năm gốc, phạm vi, nguồn và không nói quá thì "issues": [].\n- Câu không phải tuyên bố môi trường (tiêu đề, mục lục, câu tài chính, định nghĩa) thì "is_green_claim": false và không sửa.\n- BẰNG CHỨNG chỉ để hiểu bối cảnh (ví dụ: tổng phát thải thật ra tăng, số của công ty mẹ khác số hợp nhất, mục tiêu chưa đạt).\n- Không thêm bất kỳ con số nào không có trong tuyên bố. Thông tin chưa biết thì viết chỗ trống "[bổ sung: ...]".\n- Không đổi nghĩa, không biến mục tiêu thành thành tích, không xoá thông tin đúng.\n- "span" và "target_span" chép NGUYÊN VĂN một đoạn của tuyên bố.\n\nTUYÊN BỐ: {claim}\nBẰNG CHỨNG: {evidence}\n\nTrả lời đúng một đối tượng JSON:\n{{"is_green_claim": true, "issues": [{{"code": "...", "span": "..."}}], "edits": [{{"kind": "...", "rule_code": "...", "target_span": "...", "replacement": "..."}}], "rewritten_claim": "tuyên bố sau khi áp mọi sửa"}}"""\n\n\ndef parse(text):\n    if not text:\n        return None\n    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)\n    start = text.find("{")\n    while start != -1:\n        depth, in_str, esc = 0, False, False\n        for i in range(start, len(text)):\n            ch = text[i]\n            if in_str:\n                esc = (ch == "\\\\" and not esc)\n                if ch == \'"\' and not esc:\n                    in_str = False\n                continue\n            if ch == \'"\':\n                in_str = True\n            depth += {"{": 1, "}": -1}.get(ch, 0)\n            if depth == 0:\n                try:\n                    obj = json.loads(text[start:i + 1])\n                    return obj if isinstance(obj, dict) else None\n                except json.JSONDecodeError:\n                    break\n        start = text.find("{", start + 1)\n    return None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--job")\n    job = json.loads(ap.parse_args().job)\n    tag = job["tag"]\n    inputs = [p for p in os.environ.get("INPUTS", "").split(",") if p]\n    rows = []\n    for path in inputs:\n        part = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]\n        if os.environ.get("LIMIT"):\n            part = part[: int(os.environ["LIMIT"])]\n        rows += [(Path(path).stem, r) for r in part]\n    print(f"[{tag}] {len(rows)} rows from {inputs}, repo {job[\'repo\']}", flush=True)\n    from llm import LLM\n    t0 = time.time()\n    m = LLM(job["repo"], max_len=job.get("max_len", 4096), tp=job.get("tp", 1), **job.get("kw", {}))\n    load_s = time.time() - t0\n    codes = "\\n".join(f"- {k}: {v}" for k, v in CODES.items())\n    kinds = "\\n".join(f"- {k}: {v}" for k, v in KINDS.items())\n    prompts = [[{"role": "system", "content": SYSTEM},\n                {"role": "user", "content": PROMPT.format(codes=codes, kinds=kinds, claim=r["claim"],\n                                                          evidence=(r.get("evidence") or "(không có)")[:900])}] for _, r in rows]\n    t1 = time.time()\n    raw = []\n    for i in range(0, len(prompts), 64):\n        raw += m.gen(prompts[i:i + 64], max_tokens=int(os.environ.get("MAX_TOKENS", "900")))\n        print(f"[{tag}] generated {len(raw)}/{len(prompts)} ({time.time() - t1:.0f}s)", flush=True)\n    gen_s = time.time() - t1\n    Path("out").mkdir(exist_ok=True)\n    bad, files = 0, {}\n    for (stem, r), text in zip(rows, raw):\n        obj = parse(text)\n        bad += obj is None\n        files.setdefault(stem, []).append({"id": r["id"], "raw": text, "parsed": obj})\n    for stem, out in files.items():\n        with open(f"out/{tag}__{stem}.jsonl", "w", encoding="utf-8") as fh:\n            for o in out:\n                fh.write(json.dumps(o, ensure_ascii=False) + "\\n")\n    summary = {"tag": tag, "repo": job["repo"], "rows": len(rows), "unparsed": bad, "load_s": round(load_s), "gen_s": round(gen_s),\n               "ntok_in": getattr(m, "ntok_in", None), "ntok_out": getattr(m, "ntok_out", None), "overflow": getattr(m, "overflow", None)}\n    json.dump(summary, open(f"out/{tag}_summary.json", "w"), ensure_ascii=False)\n    print(f"[{tag}] SUMMARY {json.dumps(summary, ensure_ascii=False)}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    'taxonomy.py': '"""GS-Issue taxonomy: issue codes, families, edit kinds, and the fix oracle (shared by generator, scorer and worker).\n\nThe codes are those of benchmark/gs_eval_v2/LABELING_GUIDE.md section 2 (D-EDIT), unchanged.\nA family groups codes the guide itself lets overlap (MISSING_SCOPE says "công ty mẹ hay tập đoàn",\nENTITY_SCOPE_AMBIGUOUS says the same): scoring reports strict codes and families.\n"""\nimport re\nimport unicodedata\n\nCODES = {\n    "MISSING_PERIOD": "Thiếu kỳ báo cáo / năm của số liệu",\n    "MISSING_BASELINE": "Tuyên bố thay đổi (giảm, tăng, tiết kiệm) không có năm gốc",\n    "MISSING_SCOPE": "Thiếu ranh giới (nhà máy nào, công ty mẹ hay tập đoàn, phạm vi phát thải 1/2/3)",\n    "NO_FIGURE": "Tuyên bố hiệu quả môi trường mà không có số liệu",\n    "VAGUE_TERM": "Từ mơ hồ hoặc quảng cáo: xanh, thân thiện môi trường, bền vững, không ngừng",\n    "ABSOLUTE_CLAIM": "Tuyệt đối hoá: 100 %, hoàn toàn, không phát thải, tốt nhất",\n    "OFFSET_NEUTRALITY": "Trung hòa / net zero dựa vào bù trừ, không nêu tỷ lệ bù trừ",\n    "FUTURE_AS_ACHIEVED": "Cam kết / mục tiêu diễn đạt như đã đạt",\n    "UNVERIFIED_LABEL_OR_CERT": "Nhắc nhãn, chứng nhận, giải thưởng mà không nêu tổ chức cấp, phạm vi, hiệu lực",\n    "NO_EVIDENCE_REF": "Tuyên bố định lượng không dẫn nguồn / phương pháp",\n    "HIDDEN_TRADEOFF": "Nêu một mặt tốt, che mặt xấu lớn hơn liên quan trực tiếp",\n    "IRRELEVANT_CLAIM": "Khoe điều hiển nhiên hoặc bắt buộc theo luật như một thành tích",\n    "INTENSITY_VS_ABSOLUTE": "Dùng cường độ giảm để ngụ ý tổng phát thải giảm",\n    "ENTITY_SCOPE_AMBIGUOUS": "Không rõ số của công ty mẹ hay tập đoàn",\n}\nFAMILY = {"MISSING_SCOPE": "SCOPE", "ENTITY_SCOPE_AMBIGUOUS": "SCOPE",\n          "VAGUE_TERM": "VAGUE", "NO_FIGURE": "VAGUE"}\n\n\ndef family(code: str) -> str:\n    return FAMILY.get(code, code)\n\n\nKINDS = {\n    "add_scope": "thêm kỳ, năm gốc, ranh giới, phạm vi còn thiếu, dạng chỗ trống [bổ sung: ...]",\n    "placeholder": "thay cụm mơ hồ / câu không có số bằng chỗ trống [bổ sung: chỉ số và giá trị]",\n    "evidence_ref": "thêm dẫn chiếu nguồn: [bổ sung: nguồn xác minh]",\n    "soften": "bỏ hoặc làm nhẹ lời nói quá, nói lại đúng mức",\n}\n# The kind the guide prescribes for each code (several are acceptable for some).\nKIND_OK = {\n    "MISSING_PERIOD": {"add_scope"}, "MISSING_BASELINE": {"add_scope"}, "MISSING_SCOPE": {"add_scope"},\n    "ENTITY_SCOPE_AMBIGUOUS": {"add_scope"}, "NO_FIGURE": {"placeholder"}, "VAGUE_TERM": {"placeholder", "soften"},\n    "ABSOLUTE_CLAIM": {"soften"}, "OFFSET_NEUTRALITY": {"soften"}, "FUTURE_AS_ACHIEVED": {"soften"},\n    "UNVERIFIED_LABEL_OR_CERT": {"add_scope", "evidence_ref"}, "NO_EVIDENCE_REF": {"evidence_ref"},\n    "HIDDEN_TRADEOFF": {"add_scope", "evidence_ref", "soften"}, "IRRELEVANT_CLAIM": {"soften"},\n    "INTENSITY_VS_ABSOLUTE": {"soften", "add_scope"},\n}\n\nPH = re.compile(r"[\\[(][^\\])]*(?:bổ sung|cần nêu|cần bổ sung|cần ghi)[^\\])]*[\\])]", re.I)\n# What a placeholder must mention to fill each missing slot.\nSLOT_WORDS = {\n    "period": r"kỳ|năm(?! gốc)|thời gian|giai đoạn",\n    "baseline": r"gốc|so sánh|mốc",\n    "scope": r"phạm vi|ranh giới|nhà máy|cơ sở|công ty mẹ|hợp nhất|tập đoàn|đơn vị",\n    "figure": r"số|giá trị|chỉ số|mức|định lượng",\n    "source": r"nguồn|trang|tài liệu|báo cáo|phương pháp|xác minh|kiểm toán|bảo đảm",\n    "issuer": r"tổ chức|cấp|hiệu lực|phạm vi|đơn vị",\n    "context": r"tổng|toàn bộ|diễn biến|bối cảnh|cùng kỳ|tuyệt đối|phát thải",\n    "offset": r"tỷ lệ|bù trừ|tín chỉ|sổ đăng ký",\n}\n\n\ndef fold(s: str) -> str:\n    s = unicodedata.normalize("NFC", s or "").lower()\n    return re.sub(r"\\s+", " ", s).strip()\n\n\nNUM = re.compile(r"\\d+(?:[.,]\\d+)*")\n\n\ndef numbers(s: str) -> set[str]:\n    return {re.sub(r"[.,]", "", x) for x in NUM.findall(s or "")}\n\n\ndef placeholders(text: str) -> list[str]:\n    return [fold(m.group(0)) for m in PH.finditer(text or "")]\n\n\ndef slot_filled(before: str, after: str, slot: str) -> bool:\n    """A new placeholder (not in the text before) names the slot."""\n    old = placeholders(before)\n    new = [p for p in placeholders(after) if p not in old]\n    return any(re.search(SLOT_WORDS[slot], p) for p in new)\n\n\ndef oracle(case: dict, before: str, after: str) -> dict:\n    """Did the edited text fix this case\'s planted issue, and keep everything else?\n\n    resolves: the slot is asked for (drop mutations) or the planted phrase is gone or softened (inject mutations).\n    keeps_numbers: every number of the text before is still there. new_numbers: numbers the edit invented.\n    """\n    o = case["oracle"]\n    a = fold(after)\n    if o["type"] == "all":  # several planted issues: every one must be fixed\n        parts = [oracle(dict(case, oracle=p), before, after) for p in o["parts"]]\n        return {"resolves": all(p["resolves"] for p in parts), "keeps_numbers": all(p["keeps_numbers"] for p in parts),\n                "new_numbers": sorted(set().union(*(p["new_numbers"] for p in parts)))}\n    if o["type"] == "slot":\n        res = slot_filled(before, after, o["slot"])\n    elif o["type"] == "phrase":\n        gone = fold(o["phrase"]) not in a\n        asked = any(re.search(SLOT_WORDS[o.get("slot", "context")], p) for p in placeholders(after) if p not in placeholders(before))\n        res = gone or asked\n        if o.get("forbid"):  # e.g. FUTURE_AS_ACHIEVED: "đã giảm" must not remain\n            res = res and not re.search(o["forbid"], a)\n        if o.get("require"):\n            res = res and bool(re.search(o["require"], a))\n        if o.get("alt"):  # another acceptable fix, e.g. "theo quy định" for IRRELEVANT_CLAIM\n            res = res or bool(re.search(o["alt"], a))\n    else:\n        res = False\n    before_n, after_n = numbers(before), numbers(after)\n    droppable = numbers(o.get("phrase", ""))  # softening "100 %" may drop the 100\n    return {"resolves": bool(res), "keeps_numbers": (before_n - droppable) <= after_n,\n            "new_numbers": sorted(after_n - before_n)}\n',
    'jobs_small.json': '[\n {\n  "tag": "q3-8b",\n  "tp": 2,\n  "max_len": 4096,\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-8B"\n   },\n   {\n    "repo": "Qwen/Qwen3-8B",\n    "backend": "hf"\n   }\n  ]\n },\n {\n  "tag": "q3-4b",\n  "tp": 1,\n  "max_len": 4096,\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-4B-Instruct-2507"\n   },\n   {\n    "repo": "Qwen/Qwen3-4B-Instruct-2507",\n    "backend": "hf"\n   }\n  ]\n }\n]',
    'gsi_layer_a.jsonl': '{"id": "A-missing_peri-1111-000", "claim": "Điện năng tiêu thụ tại Tòa văn phòng An Cư Tower của Công ty Cổ phần Địa ốc An Cư giảm 8% so với năm gốc 2021 (Báo cáo thường niên, trang 40; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-001", "claim": "Tổng Công ty Dệt may Hồng Lĩnh ghi nhận điện năng tiêu thụ tại Nhà máy nhuộm Phố Nối giảm 21% so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 5.6; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_peri-1111-002", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận phát thải khí nhà kính phạm vi 1 và 2 của Trung tâm phân phối Long Hậu giảm 12% so với năm gốc 2021, theo Báo cáo thường niên, trang 89.", "evidence": ""}\n{"id": "A-missing_peri-1111-003", "claim": "So với năm gốc 2019, Tổng Công ty Điện lực Sông Mây đã giảm 24% lượng nước thải xả ra môi trường tại Nhà máy nhiệt điện Vĩnh Hảo 3 (Phụ lục số liệu môi trường, mục 4.4; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-004", "claim": "So với năm gốc 2020, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 11% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy nhuộm Phố Nối (Phụ lục số liệu môi trường, mục 3.3).", "evidence": ""}\n{"id": "A-missing_peri-1111-005", "claim": "Công ty Cổ phần Xi măng Đồng Lâm ghi nhận điện năng tiêu thụ tại Nhà máy xi măng Lương Sơn giảm 8% so với năm gốc 2019, theo Phụ lục số liệu môi trường, mục 9.4.", "evidence": ""}\n{"id": "A-missing_peri-1111-006", "claim": "Công ty Cổ phần Sữa Đồng Xanh ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy sữa Bình Minh giảm 6% so với năm gốc 2021, theo Báo cáo phát triển bền vững, trang 65.", "evidence": ""}\n{"id": "A-missing_peri-1111-007", "claim": "So với năm gốc 2021, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 4,5% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy phân bón Lâm Thao Mới (Báo cáo thường niên, trang 84; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-008", "claim": "So với năm gốc 2020, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 21% phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy xút clo Việt Trì Nam (Báo cáo kiểm kê khí nhà kính, bảng 4.5; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-009", "claim": "Tổng Công ty Điện lực Sông Mây ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy thủy điện Đắk Rình giảm 7,2% so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 122; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_peri-1111-010", "claim": "So với năm gốc 2020, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 7,2% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy xút clo Việt Trì Nam (Báo cáo phát triển bền vững, trang 105).", "evidence": ""}\n{"id": "A-missing_peri-1111-011", "claim": "So với năm gốc 2020, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 21% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy xút clo Việt Trì Nam (Báo cáo kiểm kê khí nhà kính, bảng 3.4).", "evidence": ""}\n{"id": "A-missing_peri-1111-012", "claim": "Công ty Cổ phần Sữa Đồng Xanh ghi nhận phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy sữa Bình Minh giảm 21% so với năm gốc 2019, theo Báo cáo phát triển bền vững, trang 98.", "evidence": ""}\n{"id": "A-missing_peri-1111-013", "claim": "Điện năng tiêu thụ tại Nhà máy sữa Bình Minh của Công ty Cổ phần Sữa Đồng Xanh giảm 24% so với năm gốc 2020 (Phụ lục số liệu môi trường, mục 7.2).", "evidence": ""}\n{"id": "A-missing_peri-1111-014", "claim": "So với năm gốc 2020, Công ty Cổ phần Địa ốc An Cư đã giảm 7,2% lượng nước thải xả ra môi trường tại Khu đô thị Sông Hậu (Phụ lục số liệu môi trường, mục 6.6; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-015", "claim": "So với năm gốc 2018, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 4,5% điện năng tiêu thụ tại Nhà máy xút clo Việt Trì Nam (Báo cáo thường niên, trang 55; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-016", "claim": "Lượng nước tiêu thụ tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 8% so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 6.1).", "evidence": ""}\n{"id": "A-missing_peri-1111-017", "claim": "Ngân hàng TMCP Phù Sa ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Hội sở chính tại Hà Nội giảm 12% so với năm gốc 2019, theo Báo cáo thường niên, trang 62.", "evidence": ""}\n{"id": "A-missing_peri-1111-018", "claim": "Điện năng tiêu thụ tại Tòa văn phòng An Cư Tower của Công ty Cổ phần Địa ốc An Cư giảm 9,5% so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 99).", "evidence": ""}\n{"id": "A-missing_peri-1111-019", "claim": "Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng nước thải xả ra môi trường tại Nhà máy nhuộm Phố Nối giảm 8% so với năm gốc 2019, theo Báo cáo kiểm kê khí nhà kính, bảng 9.4.", "evidence": ""}\n{"id": "A-missing_peri-1111-020", "claim": "Điện năng tiêu thụ tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 4,5% so với năm gốc 2019 (Báo cáo thường niên, trang 84).", "evidence": ""}\n{"id": "A-missing_peri-1111-021", "claim": "So với năm gốc 2021, Tổng Công ty Điện lực Sông Mây đã giảm 24% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy thủy điện Đắk Rình (Báo cáo thường niên, trang 117; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-022", "claim": "Ngân hàng TMCP Phù Sa ghi nhận điện năng tiêu thụ tại khối chi nhánh phía Nam giảm 18% so với năm gốc 2019, theo Phụ lục số liệu môi trường, mục 9.3; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_peri-1111-023", "claim": "Công ty Cổ phần Thép Phúc An ghi nhận lượng nước tiêu thụ tại Khu liên hợp gang thép Kỳ Long giảm 8% so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 34; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_peri-1111-024", "claim": "So với năm gốc 2021, Tổng Công ty Điện lực Sông Mây đã giảm 3% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 (Báo cáo kiểm kê khí nhà kính, bảng 9.5).", "evidence": ""}\n{"id": "A-missing_peri-1111-025", "claim": "Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng chất thải rắn đưa đi chôn lấp tại cảng cạn Tân Cảng Bắc giảm 7,2% so với năm gốc 2018, theo Báo cáo thường niên, trang 50; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_peri-1111-026", "claim": "Lượng nhiên liệu hóa thạch sử dụng tại Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 24% so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 9.4; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-027", "claim": "So với năm gốc 2021, Công ty Cổ phần Vận tải Cánh Cò đã giảm 21% lượng chất thải rắn đưa đi chôn lấp tại cảng cạn Tân Cảng Bắc (Báo cáo phát triển bền vững, trang 81).", "evidence": ""}\n{"id": "A-missing_peri-1111-028", "claim": "So với năm gốc 2018, Tổng Công ty Điện lực Sông Mây đã giảm 12% lượng nước tiêu thụ tại Nhà máy thủy điện Đắk Rình (Phụ lục số liệu môi trường, mục 3.5).", "evidence": ""}\n{"id": "A-missing_peri-1111-029", "claim": "So với năm gốc 2018, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 8% lượng nước tiêu thụ tại Trạm nghiền Hiệp Phước (Báo cáo thường niên, trang 65).", "evidence": ""}\n{"id": "A-missing_peri-1111-030", "claim": "Lượng nước tiêu thụ tại Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh giảm 18% so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 8.4; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-031", "claim": "So với năm gốc 2021, Công ty Cổ phần Vận tải Cánh Cò đã giảm 3% lượng chất thải rắn đưa đi chôn lấp tại đội xe tải Bắc – Nam (Báo cáo thường niên, trang 132; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-032", "claim": "Lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xút clo Việt Trì Nam của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 12% so với năm gốc 2021 (Báo cáo phát triển bền vững, trang 61).", "evidence": ""}\n{"id": "A-missing_peri-1111-033", "claim": "So với năm gốc 2021, Công ty Cổ phần Địa ốc An Cư đã giảm 3% điện năng tiêu thụ tại Khu đô thị Sông Hậu (Báo cáo phát triển bền vững, trang 116).", "evidence": ""}\n{"id": "A-missing_peri-1111-034", "claim": "So với năm gốc 2021, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 8% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xút clo Việt Trì Nam (Báo cáo kiểm kê khí nhà kính, bảng 4.2; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_peri-1111-035", "claim": "So với năm gốc 2018, Công ty Cổ phần Sữa Đồng Xanh đã giảm 3% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy sữa Bình Minh (Báo cáo thường niên, trang 106).", "evidence": ""}\n{"id": "A-missing_peri-1111-036", "claim": "So với năm gốc 2020, Công ty Cổ phần Vận tải Cánh Cò đã giảm 9,5% lượng nước thải xả ra môi trường tại đội xe tải Bắc – Nam (Báo cáo thường niên, trang 57).", "evidence": ""}\n{"id": "A-missing_peri-1111-037", "claim": "So với năm gốc 2021, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 6% lượng nước thải xả ra môi trường tại Nhà máy may Hòa Khánh (Báo cáo phát triển bền vững, trang 47).", "evidence": ""}\n{"id": "A-missing_peri-1111-038", "claim": "Tổng Công ty Điện lực Sông Mây ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 8% so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 9.2; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_peri-1111-039", "claim": "Công ty Cổ phần Xi măng Đồng Lâm ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Trạm nghiền Hiệp Phước giảm 4,5% so với năm gốc 2020, theo Báo cáo thường niên, trang 43.", "evidence": ""}\n{"id": "A-missing_base-1111-000", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 7,2% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xút clo Việt Trì Nam trong năm 2025 (Báo cáo phát triển bền vững, trang 22).", "evidence": ""}\n{"id": "A-missing_base-1111-001", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây giảm 15% (Báo cáo thường niên, trang 113).", "evidence": ""}\n{"id": "A-missing_base-1111-002", "claim": "Tổng Công ty Điện lực Sông Mây đã giảm 15% lượng nước tiêu thụ tại Nhà máy thủy điện Đắk Rình trong năm 2024 (Báo cáo phát triển bền vững, trang 94).", "evidence": ""}\n{"id": "A-missing_base-1111-003", "claim": "Trong năm 2023, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng nước tiêu thụ tại Nhà máy xút clo Việt Trì Nam giảm 8%, theo Báo cáo kiểm kê khí nhà kính, bảng 6.2; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-004", "claim": "Công ty Cổ phần Sữa Đồng Xanh đã giảm 3% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy sữa Bình Minh trong năm 2025 (Báo cáo phát triển bền vững, trang 107; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_base-1111-005", "claim": "Năm 2025, điện năng tiêu thụ tại Nhà máy sữa Bình Minh của Công ty Cổ phần Sữa Đồng Xanh giảm 7,2% (Báo cáo kiểm kê khí nhà kính, bảng 3.4).", "evidence": ""}\n{"id": "A-missing_base-1111-006", "claim": "Công ty Cổ phần Thép Phúc An đã giảm 21% phát thải khí nhà kính phạm vi 1, 2 và 3 của Khu liên hợp gang thép Kỳ Long trong năm 2024 (Báo cáo thường niên, trang 90; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_base-1111-007", "claim": "Công ty Cổ phần Địa ốc An Cư đã giảm 15% điện năng tiêu thụ tại Khu đô thị Sông Hậu trong năm 2025 (Báo cáo phát triển bền vững, trang 101; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_base-1111-008", "claim": "Năm 2023, phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh giảm 9,5% (Phụ lục số liệu môi trường, mục 6.4; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_base-1111-009", "claim": "Năm 2023, phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy xút clo Việt Trì Nam của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 7,2% (Báo cáo phát triển bền vững, trang 56).", "evidence": ""}\n{"id": "A-missing_base-1111-010", "claim": "Trong năm 2025, Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng nước tiêu thụ tại Nhà máy may Hòa Khánh giảm 11%, theo Báo cáo phát triển bền vững, trang 13; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-011", "claim": "Trong năm 2023, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy phân bón Lâm Thao Mới giảm 7,2%, theo Phụ lục số liệu môi trường, mục 5.4; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-012", "claim": "Năm 2025, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 21% (Báo cáo kiểm kê khí nhà kính, bảng 7.4).", "evidence": ""}\n{"id": "A-missing_base-1111-013", "claim": "Năm 2024, phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh giảm 7,2% (Phụ lục số liệu môi trường, mục 4.5; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_base-1111-014", "claim": "Trong năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng nước thải xả ra môi trường tại Nhà máy phân bón Lâm Thao Mới giảm 7,2%, theo Báo cáo thường niên, trang 139.", "evidence": ""}\n{"id": "A-missing_base-1111-015", "claim": "Năm 2024, lượng chất thải rắn đưa đi chôn lấp tại khối chi nhánh phía Nam của Ngân hàng TMCP Phù Sa giảm 21% (Báo cáo thường niên, trang 23).", "evidence": ""}\n{"id": "A-missing_base-1111-016", "claim": "Trong năm 2023, Tổng Công ty Điện lực Sông Mây ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 4,5%, theo Báo cáo phát triển bền vững, trang 108.", "evidence": ""}\n{"id": "A-missing_base-1111-017", "claim": "Trong năm 2024, Ngân hàng TMCP Phù Sa ghi nhận lượng chất thải rắn đưa đi chôn lấp tại khối chi nhánh phía Nam giảm 24%, theo Báo cáo phát triển bền vững, trang 86; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-018", "claim": "Năm 2024, điện năng tiêu thụ tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 6% (Báo cáo kiểm kê khí nhà kính, bảng 4.6).", "evidence": ""}\n{"id": "A-missing_base-1111-019", "claim": "Trong năm 2023, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng nước thải xả ra môi trường tại Nhà máy xút clo Việt Trì Nam giảm 9,5%, theo Báo cáo phát triển bền vững, trang 107; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-020", "claim": "Trong năm 2023, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Trạm nghiền Hiệp Phước giảm 18%, theo Báo cáo thường niên, trang 124.", "evidence": ""}\n{"id": "A-missing_base-1111-021", "claim": "Trong năm 2023, Công ty Cổ phần Địa ốc An Cư ghi nhận điện năng tiêu thụ tại Khu đô thị Sông Hậu giảm 3%, theo Báo cáo kiểm kê khí nhà kính, bảng 4.1; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-022", "claim": "Trong năm 2023, Công ty Cổ phần Vận tải Cánh Cò ghi nhận điện năng tiêu thụ tại cảng cạn Tân Cảng Bắc giảm 4,5%, theo Báo cáo thường niên, trang 20.", "evidence": ""}\n{"id": "A-missing_base-1111-023", "claim": "Trong năm 2023, Công ty Cổ phần Sữa Đồng Xanh ghi nhận lượng nước tiêu thụ tại Nhà máy sữa Bình Minh giảm 12%, theo Báo cáo thường niên, trang 113.", "evidence": ""}\n{"id": "A-missing_base-1111-024", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 11% lượng nhiên liệu hóa thạch sử dụng tại Trung tâm phân phối Long Hậu trong năm 2025 (Báo cáo phát triển bền vững, trang 126).", "evidence": ""}\n{"id": "A-missing_base-1111-025", "claim": "Trong năm 2025, Công ty Cổ phần Địa ốc An Cư ghi nhận lượng nước thải xả ra môi trường tại Tòa văn phòng An Cư Tower giảm 15%, theo Báo cáo thường niên, trang 54; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-026", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại Nhà máy thủy điện Đắk Rình của Tổng Công ty Điện lực Sông Mây giảm 15% (Báo cáo thường niên, trang 110; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_base-1111-027", "claim": "Trong năm 2023, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 7,2%, theo Báo cáo kiểm kê khí nhà kính, bảng 5.5.", "evidence": ""}\n{"id": "A-missing_base-1111-028", "claim": "Năm 2024, điện năng tiêu thụ tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 21% (Báo cáo phát triển bền vững, trang 121; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_base-1111-029", "claim": "Năm 2025, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây giảm 12% (Báo cáo kiểm kê khí nhà kính, bảng 7.1).", "evidence": ""}\n{"id": "A-missing_base-1111-030", "claim": "Công ty Cổ phần Địa ốc An Cư đã giảm 15% lượng nước thải xả ra môi trường tại Tòa văn phòng An Cư Tower trong năm 2025 (Báo cáo phát triển bền vững, trang 47).", "evidence": ""}\n{"id": "A-missing_base-1111-031", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An giảm 9,5% (Báo cáo thường niên, trang 39).", "evidence": ""}\n{"id": "A-missing_base-1111-032", "claim": "Trong năm 2025, Tổng Công ty Điện lực Sông Mây ghi nhận điện năng tiêu thụ tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 3%, theo Phụ lục số liệu môi trường, mục 2.1; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-033", "claim": "Công ty Cổ phần Sữa Đồng Xanh đã giảm 15% điện năng tiêu thụ tại Trang trại bò sữa Mộc Châu Xanh trong năm 2023 (Báo cáo phát triển bền vững, trang 84).", "evidence": ""}\n{"id": "A-missing_base-1111-034", "claim": "Trong năm 2025, Công ty Cổ phần Sữa Đồng Xanh ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy sữa Bình Minh giảm 15%, theo Báo cáo thường niên, trang 45; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-035", "claim": "Trong năm 2025, Công ty Cổ phần Vận tải Cánh Cò ghi nhận điện năng tiêu thụ tại cảng cạn Tân Cảng Bắc giảm 6%, theo Báo cáo kiểm kê khí nhà kính, bảng 9.1.", "evidence": ""}\n{"id": "A-missing_base-1111-036", "claim": "Công ty Cổ phần Địa ốc An Cư đã giảm 11% lượng chất thải rắn đưa đi chôn lấp tại Khu đô thị Sông Hậu trong năm 2025 (Phụ lục số liệu môi trường, mục 4.1; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_base-1111-037", "claim": "Trong năm 2023, Công ty Cổ phần Sữa Đồng Xanh ghi nhận lượng nước thải xả ra môi trường tại Trang trại bò sữa Mộc Châu Xanh giảm 15%, theo Phụ lục số liệu môi trường, mục 9.3; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_base-1111-038", "claim": "Trong năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy phân bón Lâm Thao Mới giảm 8%, theo Báo cáo thường niên, trang 98.", "evidence": ""}\n{"id": "A-missing_base-1111-039", "claim": "Tổng Công ty Dệt may Hồng Lĩnh đã giảm 3% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhuộm Phố Nối trong năm 2023 (Báo cáo phát triển bền vững, trang 88; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-000", "claim": "Trong năm 2025, Công ty Cổ phần Vận tải Cánh Cò ghi nhận điện năng tiêu thụ giảm 7,2% so với năm gốc 2021, theo Phụ lục số liệu môi trường, mục 7.3; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_scop-1111-001", "claim": "Trong năm 2023, Công ty Cổ phần Sữa Đồng Xanh ghi nhận phát thải khí nhà kính giảm 18% so với năm gốc 2021, theo Báo cáo phát triển bền vững, trang 54.", "evidence": ""}\n{"id": "A-missing_scop-1111-002", "claim": "Năm 2024, điện năng tiêu thụ của Tổng Công ty Dệt may Hồng Lĩnh giảm 11% so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 5.4).", "evidence": ""}\n{"id": "A-missing_scop-1111-003", "claim": "So với năm gốc 2018, Công ty Cổ phần Thép Phúc An đã giảm 12% lượng nước thải xả ra môi trường trong năm 2025 (Phụ lục số liệu môi trường, mục 9.6; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-004", "claim": "Năm 2024, lượng nước tiêu thụ của Công ty Cổ phần Địa ốc An Cư giảm 9,5% so với năm gốc 2019 (Báo cáo thường niên, trang 135; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-005", "claim": "Trong năm 2024, Công ty Cổ phần Địa ốc An Cư ghi nhận phát thải khí nhà kính giảm 11% so với năm gốc 2018, theo Báo cáo phát triển bền vững, trang 12; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_scop-1111-006", "claim": "So với năm gốc 2020, Tổng Công ty Điện lực Sông Mây đã giảm 18% phát thải khí nhà kính trong năm 2025 (Phụ lục số liệu môi trường, mục 2.1).", "evidence": ""}\n{"id": "A-missing_scop-1111-007", "claim": "Năm 2023, phát thải khí nhà kính của Ngân hàng TMCP Phù Sa giảm 12% so với năm gốc 2021 (Báo cáo phát triển bền vững, trang 14; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-008", "claim": "Năm 2025, điện năng tiêu thụ của Công ty Cổ phần Vận tải Cánh Cò giảm 8% so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 5.4).", "evidence": ""}\n{"id": "A-missing_scop-1111-009", "claim": "Năm 2025, điện năng tiêu thụ của Công ty Cổ phần Vận tải Cánh Cò giảm 3% so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 7.6; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-010", "claim": "So với năm gốc 2018, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 4,5% lượng nhiên liệu hóa thạch sử dụng trong năm 2025 (Phụ lục số liệu môi trường, mục 5.1).", "evidence": ""}\n{"id": "A-missing_scop-1111-011", "claim": "Năm 2025, lượng chất thải rắn đưa đi chôn lấp của Công ty Cổ phần Vận tải Cánh Cò giảm 4,5% so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 4.1; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-012", "claim": "Trong năm 2023, Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng chất thải rắn đưa đi chôn lấp giảm 6% so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 5.5.", "evidence": ""}\n{"id": "A-missing_scop-1111-013", "claim": "So với năm gốc 2018, Công ty Cổ phần Thép Phúc An đã giảm 6% lượng nước tiêu thụ trong năm 2023 (Báo cáo kiểm kê khí nhà kính, bảng 6.5).", "evidence": ""}\n{"id": "A-missing_scop-1111-014", "claim": "Trong năm 2023, Công ty Cổ phần Thép Phúc An ghi nhận lượng nước tiêu thụ giảm 7,2% so với năm gốc 2018, theo Báo cáo thường niên, trang 14; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_scop-1111-015", "claim": "Trong năm 2023, Công ty Cổ phần Thép Phúc An ghi nhận lượng nhiên liệu hóa thạch sử dụng giảm 7,2% so với năm gốc 2019, theo Báo cáo kiểm kê khí nhà kính, bảng 4.5; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_scop-1111-016", "claim": "Năm 2023, lượng nhiên liệu hóa thạch sử dụng của Tổng Công ty Dệt may Hồng Lĩnh giảm 18% so với năm gốc 2021 (Phụ lục số liệu môi trường, mục 4.2; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-017", "claim": "So với năm gốc 2019, Công ty Cổ phần Sữa Đồng Xanh đã giảm 24% điện năng tiêu thụ trong năm 2023 (Báo cáo kiểm kê khí nhà kính, bảng 9.3; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-018", "claim": "Trong năm 2025, Công ty Cổ phần Địa ốc An Cư ghi nhận lượng nước tiêu thụ giảm 8% so với năm gốc 2018, theo Báo cáo phát triển bền vững, trang 100.", "evidence": ""}\n{"id": "A-missing_scop-1111-019", "claim": "Năm 2025, lượng chất thải rắn đưa đi chôn lấp của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 9,5% so với năm gốc 2021 (Báo cáo thường niên, trang 100).", "evidence": ""}\n{"id": "A-missing_scop-1111-020", "claim": "So với năm gốc 2020, Công ty Cổ phần Sữa Đồng Xanh đã giảm 11% lượng nước thải xả ra môi trường trong năm 2023 (Báo cáo phát triển bền vững, trang 68).", "evidence": ""}\n{"id": "A-missing_scop-1111-021", "claim": "Năm 2024, lượng nước tiêu thụ của Công ty Cổ phần Vận tải Cánh Cò giảm 8% so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 2.4; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-022", "claim": "Trong năm 2025, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận lượng nước tiêu thụ giảm 11% so với năm gốc 2021, theo Báo cáo thường niên, trang 129; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_scop-1111-023", "claim": "So với năm gốc 2020, Công ty Cổ phần Thép Phúc An đã giảm 4,5% điện năng tiêu thụ trong năm 2023 (Báo cáo phát triển bền vững, trang 138; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-024", "claim": "So với năm gốc 2020, Công ty Cổ phần Thép Phúc An đã giảm 6% lượng nước thải xả ra môi trường trong năm 2024 (Báo cáo phát triển bền vững, trang 113; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-025", "claim": "Năm 2024, điện năng tiêu thụ của Công ty Cổ phần Sữa Đồng Xanh giảm 8% so với năm gốc 2021 (Báo cáo thường niên, trang 60).", "evidence": ""}\n{"id": "A-missing_scop-1111-026", "claim": "Năm 2024, lượng nhiên liệu hóa thạch sử dụng của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 9,5% so với năm gốc 2021 (Báo cáo thường niên, trang 18).", "evidence": ""}\n{"id": "A-missing_scop-1111-027", "claim": "So với năm gốc 2018, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 8% lượng chất thải rắn đưa đi chôn lấp trong năm 2024 (Báo cáo kiểm kê khí nhà kính, bảng 3.5).", "evidence": ""}\n{"id": "A-missing_scop-1111-028", "claim": "So với năm gốc 2021, Công ty Cổ phần Thép Phúc An đã giảm 12% lượng nhiên liệu hóa thạch sử dụng trong năm 2024 (Phụ lục số liệu môi trường, mục 6.5).", "evidence": ""}\n{"id": "A-missing_scop-1111-029", "claim": "Năm 2024, điện năng tiêu thụ của Công ty Cổ phần Địa ốc An Cư giảm 7,2% so với năm gốc 2021 (Báo cáo thường niên, trang 74; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-030", "claim": "Trong năm 2023, Tổng Công ty Dệt may Hồng Lĩnh ghi nhận phát thải khí nhà kính giảm 9,5% so với năm gốc 2021, theo Báo cáo kiểm kê khí nhà kính, bảng 3.3; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-missing_scop-1111-031", "claim": "Năm 2024, lượng nhiên liệu hóa thạch sử dụng của Công ty Cổ phần Thép Phúc An giảm 15% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 60).", "evidence": ""}\n{"id": "A-missing_scop-1111-032", "claim": "Năm 2025, điện năng tiêu thụ của Công ty Cổ phần Địa ốc An Cư giảm 7,2% so với năm gốc 2021 (Phụ lục số liệu môi trường, mục 8.5; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-033", "claim": "Năm 2025, lượng nước thải xả ra môi trường của Công ty Cổ phần Sữa Đồng Xanh giảm 21% so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 3.4; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-034", "claim": "Năm 2025, lượng nước tiêu thụ của Công ty Cổ phần Thép Phúc An giảm 8% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 31; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-035", "claim": "So với năm gốc 2018, Công ty Cổ phần Thép Phúc An đã giảm 9,5% lượng chất thải rắn đưa đi chôn lấp trong năm 2023 (Báo cáo phát triển bền vững, trang 78).", "evidence": ""}\n{"id": "A-missing_scop-1111-036", "claim": "So với năm gốc 2021, Ngân hàng TMCP Phù Sa đã giảm 12% điện năng tiêu thụ trong năm 2025 (Báo cáo thường niên, trang 54; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-missing_scop-1111-037", "claim": "Trong năm 2025, Tổng Công ty Điện lực Sông Mây ghi nhận phát thải khí nhà kính giảm 4,5% so với năm gốc 2018, theo Báo cáo phát triển bền vững, trang 76.", "evidence": ""}\n{"id": "A-missing_scop-1111-038", "claim": "Trong năm 2023, Công ty Cổ phần Địa ốc An Cư ghi nhận phát thải khí nhà kính giảm 4,5% so với năm gốc 2021, theo Báo cáo kiểm kê khí nhà kính, bảng 6.3.", "evidence": ""}\n{"id": "A-missing_scop-1111-039", "claim": "Trong năm 2023, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận phát thải khí nhà kính giảm 8% so với năm gốc 2020, theo Báo cáo thường niên, trang 100; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-no_figure-1111-000", "claim": "Trong năm 2023, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận điện năng tiêu thụ tại hệ thống 120 cửa hàng phía Nam giảm so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 5.4.", "evidence": ""}\n{"id": "A-no_figure-1111-001", "claim": "Năm 2024, phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh đã giảm so với năm gốc 2019 (Báo cáo thường niên, trang 101).", "evidence": ""}\n{"id": "A-no_figure-1111-002", "claim": "Năm 2025, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh đã giảm so với năm gốc 2020 (Báo cáo thường niên, trang 79).", "evidence": ""}\n{"id": "A-no_figure-1111-003", "claim": "Trong năm 2023, Công ty Cổ phần Sữa Đồng Xanh ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy sữa Bình Minh giảm so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 8.1; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-no_figure-1111-004", "claim": "So với năm gốc 2020, Tổng Công ty Điện lực Sông Mây đã giảm lượng nước tiêu thụ tại Nhà máy nhiệt điện Vĩnh Hảo 3 trong năm 2024 (Báo cáo kiểm kê khí nhà kính, bảng 7.1; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-005", "claim": "Năm 2025, lượng nước tiêu thụ tại Hội sở chính tại Hà Nội của Ngân hàng TMCP Phù Sa đã giảm so với năm gốc 2019 (Báo cáo thường niên, trang 106).", "evidence": ""}\n{"id": "A-no_figure-1111-006", "claim": "So với năm gốc 2020, Tổng Công ty Điện lực Sông Mây đã giảm lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 trong năm 2024 (Phụ lục số liệu môi trường, mục 7.3; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-007", "claim": "So với năm gốc 2021, Tổng Công ty Điện lực Sông Mây đã giảm lượng nhiên liệu hóa thạch sử dụng tại Nhà máy thủy điện Đắk Rình trong năm 2024 (Báo cáo phát triển bền vững, trang 53).", "evidence": ""}\n{"id": "A-no_figure-1111-008", "claim": "Năm 2025, lượng nước tiêu thụ tại Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh đã giảm so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 135; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-009", "claim": "So với năm gốc 2019, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm lượng nước tiêu thụ tại Nhà máy xút clo Việt Trì Nam trong năm 2025 (Phụ lục số liệu môi trường, mục 6.2).", "evidence": ""}\n{"id": "A-no_figure-1111-010", "claim": "Trong năm 2023, Công ty Cổ phần Thép Phúc An ghi nhận điện năng tiêu thụ tại Nhà máy luyện thép Nghi Sơn 2 giảm so với năm gốc 2019, theo Phụ lục số liệu môi trường, mục 5.6; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-no_figure-1111-011", "claim": "Trong năm 2025, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nước tiêu thụ tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm so với năm gốc 2021, theo Báo cáo kiểm kê khí nhà kính, bảng 6.5; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-no_figure-1111-012", "claim": "Năm 2025, điện năng tiêu thụ tại Tòa văn phòng An Cư Tower của Công ty Cổ phần Địa ốc An Cư đã giảm so với năm gốc 2018 (Báo cáo thường niên, trang 68; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-013", "claim": "Năm 2025, điện năng tiêu thụ tại Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm so với năm gốc 2018 (Báo cáo thường niên, trang 73).", "evidence": ""}\n{"id": "A-no_figure-1111-014", "claim": "Trong năm 2024, Tổng Công ty Điện lực Sông Mây ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy nhiệt điện Vĩnh Hảo 3 giảm so với năm gốc 2020, theo Phụ lục số liệu môi trường, mục 2.1.", "evidence": ""}\n{"id": "A-no_figure-1111-015", "claim": "Trong năm 2023, Tổng Công ty Điện lực Sông Mây ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy thủy điện Đắk Rình giảm so với năm gốc 2018, theo Báo cáo kiểm kê khí nhà kính, bảng 6.6; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-no_figure-1111-016", "claim": "Trong năm 2024, Công ty Cổ phần Sữa Đồng Xanh ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy sữa Bình Minh giảm so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 9.5.", "evidence": ""}\n{"id": "A-no_figure-1111-017", "claim": "So với năm gốc 2020, Tổng Công ty Dệt may Hồng Lĩnh đã giảm lượng nước tiêu thụ tại Nhà máy nhuộm Phố Nối trong năm 2024 (Báo cáo thường niên, trang 98; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-018", "claim": "Năm 2024, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy sữa Bình Minh của Công ty Cổ phần Sữa Đồng Xanh đã giảm so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 3.6; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-019", "claim": "So với năm gốc 2020, Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm lượng nước tiêu thụ tại Trung tâm phân phối Long Hậu trong năm 2023 (Báo cáo kiểm kê khí nhà kính, bảng 2.5; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-020", "claim": "So với năm gốc 2018, Tổng Công ty Dệt may Hồng Lĩnh đã giảm lượng chất thải rắn đưa đi chôn lấp tại Nhà máy may Hòa Khánh trong năm 2024 (Báo cáo phát triển bền vững, trang 83; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-021", "claim": "Trong năm 2024, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng nước thải xả ra môi trường tại cảng cạn Tân Cảng Bắc giảm so với năm gốc 2019, theo Báo cáo kiểm kê khí nhà kính, bảng 2.6.", "evidence": ""}\n{"id": "A-no_figure-1111-022", "claim": "Trong năm 2025, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng nước thải xả ra môi trường tại cảng cạn Tân Cảng Bắc giảm so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 84.", "evidence": ""}\n{"id": "A-no_figure-1111-023", "claim": "Trong năm 2025, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận lượng nước tiêu thụ tại Trạm nghiền Hiệp Phước giảm so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 124.", "evidence": ""}\n{"id": "A-no_figure-1111-024", "claim": "Năm 2024, điện năng tiêu thụ tại Trạm nghiền Hiệp Phước của Công ty Cổ phần Xi măng Đồng Lâm đã giảm so với năm gốc 2020 (Phụ lục số liệu môi trường, mục 5.2; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-025", "claim": "So với năm gốc 2021, Công ty Cổ phần Sữa Đồng Xanh đã giảm lượng nước thải xả ra môi trường tại Trang trại bò sữa Mộc Châu Xanh trong năm 2024 (Báo cáo kiểm kê khí nhà kính, bảng 9.6; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-026", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại khối chi nhánh phía Nam của Ngân hàng TMCP Phù Sa đã giảm so với năm gốc 2018 (Báo cáo kiểm kê khí nhà kính, bảng 5.3).", "evidence": ""}\n{"id": "A-no_figure-1111-027", "claim": "So với năm gốc 2019, Tổng Công ty Dệt may Hồng Lĩnh đã giảm lượng nước thải xả ra môi trường tại Nhà máy may Hòa Khánh trong năm 2023 (Báo cáo thường niên, trang 46; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-028", "claim": "Năm 2024, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy sữa Bình Minh của Công ty Cổ phần Sữa Đồng Xanh đã giảm so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 6.3).", "evidence": ""}\n{"id": "A-no_figure-1111-029", "claim": "Trong năm 2024, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận phát thải khí nhà kính phạm vi 1 và 2 của hệ thống 120 cửa hàng phía Nam giảm so với năm gốc 2021, theo Phụ lục số liệu môi trường, mục 7.1; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-no_figure-1111-030", "claim": "Trong năm 2025, Công ty Cổ phần Thép Phúc An ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy luyện thép Nghi Sơn 2 giảm so với năm gốc 2021, theo Báo cáo kiểm kê khí nhà kính, bảng 5.1; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-no_figure-1111-031", "claim": "Trong năm 2023, Công ty Cổ phần Địa ốc An Cư ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Khu đô thị Sông Hậu giảm so với năm gốc 2021, theo Phụ lục số liệu môi trường, mục 7.5.", "evidence": ""}\n{"id": "A-no_figure-1111-032", "claim": "So với năm gốc 2021, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm lượng nước tiêu thụ tại Nhà máy phân bón Lâm Thao Mới trong năm 2024 (Báo cáo kiểm kê khí nhà kính, bảng 5.2).", "evidence": ""}\n{"id": "A-no_figure-1111-033", "claim": "Năm 2023, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy luyện thép Nghi Sơn 2 của Công ty Cổ phần Thép Phúc An đã giảm so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 109; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-034", "claim": "So với năm gốc 2020, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy phân bón Lâm Thao Mới trong năm 2024 (Báo cáo phát triển bền vững, trang 77; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-035", "claim": "Năm 2024, lượng nước tiêu thụ tại Trạm nghiền Hiệp Phước của Công ty Cổ phần Xi măng Đồng Lâm đã giảm so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 5.5; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-036", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại Trang trại bò sữa Mộc Châu Xanh của Công ty Cổ phần Sữa Đồng Xanh đã giảm so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 20; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-no_figure-1111-037", "claim": "Trong năm 2023, Công ty Cổ phần Địa ốc An Cư ghi nhận lượng nước tiêu thụ tại Khu đô thị Sông Hậu giảm so với năm gốc 2019, theo Báo cáo kiểm kê khí nhà kính, bảng 3.2.", "evidence": ""}\n{"id": "A-no_figure-1111-038", "claim": "So với năm gốc 2018, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xút clo Việt Trì Nam trong năm 2025 (Báo cáo kiểm kê khí nhà kính, bảng 9.2).", "evidence": ""}\n{"id": "A-no_figure-1111-039", "claim": "So với năm gốc 2020, Công ty Cổ phần Xi măng Đồng Lâm đã giảm lượng nước thải xả ra môi trường tại Nhà máy xi măng Lương Sơn trong năm 2024 (Báo cáo phát triển bền vững, trang 138).", "evidence": ""}\n{"id": "A-vague_term-1111-000", "claim": "So với năm gốc 2020, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 6% lượng nước tiêu thụ tại Trạm nghiền Hiệp Phước trong năm 2023 (Báo cáo kiểm kê khí nhà kính, bảng 6.6; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-001", "claim": "So với năm gốc 2018, Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 21% lượng nước thải xả ra môi trường tại Trung tâm phân phối Long Hậu trong năm 2025 (Báo cáo phát triển bền vững, trang 124), hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-002", "claim": "Năm 2024, lượng nước thải xả ra môi trường tại Nhà máy xút clo Việt Trì Nam của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 3% so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 4.2; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-003", "claim": "So với năm gốc 2020, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 4,5% lượng nước thải xả ra môi trường tại Nhà máy xi măng Lương Sơn trong năm 2024 (Báo cáo thường niên, trang 44), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-004", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 9,5% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 88; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-vague_term-1111-005", "claim": "Trong năm 2023, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Trạm nghiền Hiệp Phước giảm 11% so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 87; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập, hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-006", "claim": "Năm 2023, phát thải khí nhà kính phạm vi 1, 2 và 3 của Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 24% so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 68), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-007", "claim": "Năm 2023, lượng nước tiêu thụ tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 12% so với năm gốc 2019 (Báo cáo thường niên, trang 44), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-008", "claim": "Trong năm 2024, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy xi măng Lương Sơn giảm 3% so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 74; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập, khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-vague_term-1111-009", "claim": "Trong năm 2025, Công ty Cổ phần Sữa Đồng Xanh ghi nhận điện năng tiêu thụ tại Nhà máy sữa Bình Minh giảm 11% so với năm gốc 2019, theo Báo cáo phát triển bền vững, trang 31; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập, hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-010", "claim": "Năm 2024, lượng nước thải xả ra môi trường tại Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 15% so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 48), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-011", "claim": "Trong năm 2025, Tổng Công ty Điện lực Sông Mây ghi nhận điện năng tiêu thụ tại Nhà máy thủy điện Đắk Rình giảm 3% so với năm gốc 2021, theo Phụ lục số liệu môi trường, mục 9.4, góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-012", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 12% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 61), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-013", "claim": "Trong năm 2023, Ngân hàng TMCP Phù Sa ghi nhận điện năng tiêu thụ tại khối chi nhánh phía Nam giảm 3% so với năm gốc 2020, theo Báo cáo thường niên, trang 83, góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-014", "claim": "So với năm gốc 2020, Ngân hàng TMCP Phù Sa đã giảm 7,2% lượng nhiên liệu hóa thạch sử dụng tại Hội sở chính tại Hà Nội trong năm 2025 (Phụ lục số liệu môi trường, mục 9.6), khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-vague_term-1111-015", "claim": "So với năm gốc 2019, Công ty Cổ phần Sữa Đồng Xanh đã giảm 8% lượng nhiên liệu hóa thạch sử dụng tại Trang trại bò sữa Mộc Châu Xanh trong năm 2023 (Báo cáo phát triển bền vững, trang 107), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-016", "claim": "So với năm gốc 2020, Công ty Cổ phần Vận tải Cánh Cò đã giảm 8% điện năng tiêu thụ tại đội xe tải Bắc – Nam trong năm 2023 (Báo cáo phát triển bền vững, trang 109; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-017", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại Trung tâm phân phối Long Hậu của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 11% so với năm gốc 2021 (Báo cáo thường niên, trang 85), khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-vague_term-1111-018", "claim": "Trong năm 2025, Công ty Cổ phần Sữa Đồng Xanh ghi nhận điện năng tiêu thụ tại Trang trại bò sữa Mộc Châu Xanh giảm 3% so với năm gốc 2018, theo Báo cáo thường niên, trang 38, góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-019", "claim": "Năm 2024, lượng chất thải rắn đưa đi chôn lấp tại Hội sở chính tại Hà Nội của Ngân hàng TMCP Phù Sa giảm 4,5% so với năm gốc 2019 (Báo cáo thường niên, trang 50), góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-020", "claim": "Trong năm 2023, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng nước tiêu thụ tại cảng cạn Tân Cảng Bắc giảm 21% so với năm gốc 2019, theo Phụ lục số liệu môi trường, mục 2.1; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập, khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-vague_term-1111-021", "claim": "Trong năm 2024, Công ty Cổ phần Thép Phúc An ghi nhận điện năng tiêu thụ tại Khu liên hợp gang thép Kỳ Long giảm 12% so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 131; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập, hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-022", "claim": "Trong năm 2025, Tổng Công ty Dệt may Hồng Lĩnh ghi nhận điện năng tiêu thụ tại Nhà máy may Hòa Khánh giảm 6% so với năm gốc 2021, theo Phụ lục số liệu môi trường, mục 9.3; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập, khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-vague_term-1111-023", "claim": "Trong năm 2023, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xi măng Lương Sơn giảm 21% so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 127, góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-024", "claim": "So với năm gốc 2021, Công ty Cổ phần Địa ốc An Cư đã giảm 21% điện năng tiêu thụ tại Khu đô thị Sông Hậu trong năm 2023 (Báo cáo thường niên, trang 99), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-025", "claim": "Năm 2023, lượng nước tiêu thụ tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 9,5% so với năm gốc 2020 (Báo cáo thường niên, trang 20), góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-026", "claim": "Năm 2023, lượng nước thải xả ra môi trường tại Nhà máy sữa Bình Minh của Công ty Cổ phần Sữa Đồng Xanh giảm 8% so với năm gốc 2018 (Báo cáo thường niên, trang 104), góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-027", "claim": "Trong năm 2023, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng chất thải rắn đưa đi chôn lấp tại cảng cạn Tân Cảng Bắc giảm 12% so với năm gốc 2018, theo Báo cáo thường niên, trang 27, hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-028", "claim": "Trong năm 2023, Công ty Cổ phần Thép Phúc An ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Khu liên hợp gang thép Kỳ Long giảm 12% so với năm gốc 2018, theo Báo cáo kiểm kê khí nhà kính, bảng 7.3, khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-vague_term-1111-029", "claim": "So với năm gốc 2019, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 21% lượng nhiên liệu hóa thạch sử dụng tại Trạm nghiền Hiệp Phước trong năm 2025 (Báo cáo thường niên, trang 67; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-030", "claim": "Trong năm 2023, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng nước tiêu thụ tại Nhà máy phân bón Lâm Thao Mới giảm 12% so với năm gốc 2021, theo Báo cáo kiểm kê khí nhà kính, bảng 5.6; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập, thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-031", "claim": "So với năm gốc 2018, Tổng Công ty Điện lực Sông Mây đã giảm 21% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 trong năm 2024 (Báo cáo kiểm kê khí nhà kính, bảng 3.2; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-032", "claim": "So với năm gốc 2021, Công ty Cổ phần Thép Phúc An đã giảm 24% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy luyện thép Nghi Sơn 2 trong năm 2024 (Báo cáo phát triển bền vững, trang 117), hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-033", "claim": "So với năm gốc 2021, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 6% lượng nước tiêu thụ tại Trạm nghiền Hiệp Phước trong năm 2023 (Báo cáo kiểm kê khí nhà kính, bảng 2.1), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-034", "claim": "Trong năm 2024, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nước thải xả ra môi trường tại Nhà máy thủy điện Đắk Rình giảm 24% so với năm gốc 2018, theo Báo cáo phát triển bền vững, trang 44, thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-035", "claim": "Năm 2023, lượng nhiên liệu hóa thạch sử dụng tại Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An giảm 8% so với năm gốc 2020 (Phụ lục số liệu môi trường, mục 6.6; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-036", "claim": "So với năm gốc 2018, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 15% lượng nước tiêu thụ tại Nhà máy nhuộm Phố Nối trong năm 2025 (Báo cáo thường niên, trang 69), khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-vague_term-1111-037", "claim": "Năm 2024, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây giảm 9,5% so với năm gốc 2019 (Báo cáo thường niên, trang 38), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-vague_term-1111-038", "claim": "Năm 2023, điện năng tiêu thụ tại Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 6% so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 5.6), hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-vague_term-1111-039", "claim": "So với năm gốc 2020, Ngân hàng TMCP Phù Sa đã giảm 3% lượng nước thải xả ra môi trường tại Hội sở chính tại Hà Nội trong năm 2024 (Phụ lục số liệu môi trường, mục 7.4; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-absolute_cla-1111-000", "claim": "Năm 2025, điện năng tiêu thụ tại Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 12% so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 7.2), đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-001", "claim": "So với năm gốc 2019, Công ty Cổ phần Sữa Đồng Xanh đã giảm 15% điện năng tiêu thụ tại Trang trại bò sữa Mộc Châu Xanh trong năm 2025 (Phụ lục số liệu môi trường, mục 8.2; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-002", "claim": "Trong năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy phân bón Lâm Thao Mới giảm 8% so với năm gốc 2018, theo Báo cáo kiểm kê khí nhà kính, bảng 2.3, đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-003", "claim": "Trong năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy xút clo Việt Trì Nam giảm 11% so với năm gốc 2019, theo Báo cáo kiểm kê khí nhà kính, bảng 6.5, đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-004", "claim": "Trong năm 2024, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng nước tiêu thụ tại đội xe tải Bắc – Nam giảm 18% so với năm gốc 2018, theo Báo cáo kiểm kê khí nhà kính, bảng 8.4; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập, qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-absolute_cla-1111-005", "claim": "So với năm gốc 2019, Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 11% lượng chất thải rắn đưa đi chôn lấp tại hệ thống 120 cửa hàng phía Nam trong năm 2023 (Phụ lục số liệu môi trường, mục 4.3), qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-absolute_cla-1111-006", "claim": "Trong năm 2025, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy xút clo Việt Trì Nam giảm 21% so với năm gốc 2018, theo Phụ lục số liệu môi trường, mục 2.3, và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-absolute_cla-1111-007", "claim": "Trong năm 2024, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận lượng nước tiêu thụ tại Trạm nghiền Hiệp Phước giảm 15% so với năm gốc 2021, theo Báo cáo phát triển bền vững, trang 34; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập, và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-absolute_cla-1111-008", "claim": "Trong năm 2023, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận lượng nước thải xả ra môi trường tại Nhà máy xi măng Lương Sơn giảm 12% so với năm gốc 2019, theo Phụ lục số liệu môi trường, mục 9.6, qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-absolute_cla-1111-009", "claim": "Năm 2025, lượng chất thải rắn đưa đi chôn lấp tại Trung tâm phân phối Long Hậu của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 7,2% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 119), đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-010", "claim": "Năm 2024, phát thải khí nhà kính phạm vi 1 và 2 của Hội sở chính tại Hà Nội của Ngân hàng TMCP Phù Sa giảm 4,5% so với năm gốc 2020 (Phụ lục số liệu môi trường, mục 2.3; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-absolute_cla-1111-011", "claim": "Năm 2024, lượng nhiên liệu hóa thạch sử dụng tại Trạm nghiền Hiệp Phước của Công ty Cổ phần Xi măng Đồng Lâm giảm 9,5% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 39), đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-012", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 7,2% so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 6.4), qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-absolute_cla-1111-013", "claim": "Trong năm 2024, Ngân hàng TMCP Phù Sa ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Hội sở chính tại Hà Nội giảm 18% so với năm gốc 2021, theo Báo cáo phát triển bền vững, trang 31, giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-014", "claim": "Trong năm 2024, Công ty Cổ phần Địa ốc An Cư ghi nhận điện năng tiêu thụ tại Khu đô thị Sông Hậu giảm 6% so với năm gốc 2021, theo Báo cáo thường niên, trang 28, đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-015", "claim": "Trong năm 2025, Công ty Cổ phần Địa ốc An Cư ghi nhận phát thải khí nhà kính phạm vi 1 và 2 của Khu đô thị Sông Hậu giảm 8% so với năm gốc 2018, theo Phụ lục số liệu môi trường, mục 4.1, giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-016", "claim": "Trong năm 2024, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy xi măng Lương Sơn giảm 8% so với năm gốc 2020, theo Báo cáo thường niên, trang 88, và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-absolute_cla-1111-017", "claim": "Trong năm 2025, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng nước tiêu thụ tại đội xe tải Bắc – Nam giảm 4,5% so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 4.4; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập, giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-018", "claim": "Năm 2025, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy xút clo Việt Trì Nam của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 7,2% so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 2.5; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-absolute_cla-1111-019", "claim": "Trong năm 2025, Công ty Cổ phần Thép Phúc An ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy luyện thép Nghi Sơn 2 giảm 24% so với năm gốc 2018, theo Báo cáo thường niên, trang 31; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập, giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-020", "claim": "So với năm gốc 2021, Công ty Cổ phần Địa ốc An Cư đã giảm 15% phát thải khí nhà kính phạm vi 1, 2 và 3 của Tòa văn phòng An Cư Tower trong năm 2024 (Báo cáo phát triển bền vững, trang 128; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-absolute_cla-1111-021", "claim": "So với năm gốc 2019, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 21% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy nhuộm Phố Nối trong năm 2024 (Phụ lục số liệu môi trường, mục 3.4), và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-absolute_cla-1111-022", "claim": "Năm 2023, điện năng tiêu thụ tại đội xe tải Bắc – Nam của Công ty Cổ phần Vận tải Cánh Cò giảm 7,2% so với năm gốc 2020 (Báo cáo thường niên, trang 125; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-absolute_cla-1111-023", "claim": "Năm 2024, lượng nước thải xả ra môi trường tại Trang trại bò sữa Mộc Châu Xanh của Công ty Cổ phần Sữa Đồng Xanh giảm 6% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 48; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-absolute_cla-1111-024", "claim": "Năm 2023, phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy luyện thép Nghi Sơn 2 của Công ty Cổ phần Thép Phúc An giảm 9,5% so với năm gốc 2021 (Phụ lục số liệu môi trường, mục 4.1), giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-025", "claim": "Năm 2024, phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 7,2% so với năm gốc 2019 (Báo cáo thường niên, trang 53; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-026", "claim": "Năm 2023, lượng nước tiêu thụ tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 4,5% so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 80), giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-027", "claim": "So với năm gốc 2019, Tổng Công ty Điện lực Sông Mây đã giảm 8% điện năng tiêu thụ tại Nhà máy nhiệt điện Vĩnh Hảo 3 trong năm 2024 (Báo cáo phát triển bền vững, trang 18; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-absolute_cla-1111-028", "claim": "Trong năm 2025, Công ty Cổ phần Địa ốc An Cư ghi nhận lượng nước tiêu thụ tại Khu đô thị Sông Hậu giảm 4,5% so với năm gốc 2020, theo Phụ lục số liệu môi trường, mục 2.5, giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-029", "claim": "Năm 2024, phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 15% so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 5.1; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-030", "claim": "Năm 2024, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy sữa Bình Minh của Công ty Cổ phần Sữa Đồng Xanh giảm 6% so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 6.3), qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-absolute_cla-1111-031", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại Hội sở chính tại Hà Nội của Ngân hàng TMCP Phù Sa giảm 15% so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 2.5; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-032", "claim": "Trong năm 2024, Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy may Hòa Khánh giảm 7,2% so với năm gốc 2020, theo Phụ lục số liệu môi trường, mục 7.1, qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-absolute_cla-1111-033", "claim": "Năm 2023, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây giảm 12% so với năm gốc 2018 (Báo cáo thường niên, trang 87), giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-034", "claim": "Năm 2024, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh giảm 11% so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 132; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-absolute_cla-1111-035", "claim": "Năm 2024, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 18% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 42; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-absolute_cla-1111-036", "claim": "So với năm gốc 2021, Tổng Công ty Điện lực Sông Mây đã giảm 7,2% điện năng tiêu thụ tại Nhà máy thủy điện Đắk Rình trong năm 2025 (Phụ lục số liệu môi trường, mục 6.3), và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-absolute_cla-1111-037", "claim": "Trong năm 2024, Công ty Cổ phần Sữa Đồng Xanh ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy sữa Bình Minh giảm 18% so với năm gốc 2018, theo Phụ lục số liệu môi trường, mục 3.5, đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-038", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh giảm 21% so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 8.5; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-absolute_cla-1111-039", "claim": "Trong năm 2023, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận phát thải khí nhà kính phạm vi 1 và 2 của hệ thống 120 cửa hàng phía Nam giảm 7,2% so với năm gốc 2020, theo Báo cáo thường niên, trang 23, đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-offset_neutr-1111-000", "claim": "Trong năm 2025, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nước thải xả ra môi trường tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 12% so với năm gốc 2020, theo Phụ lục số liệu môi trường, mục 6.5; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập, đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-001", "claim": "Trong năm 2025, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng chất thải rắn đưa đi chôn lấp tại đội xe tải Bắc – Nam giảm 15% so với năm gốc 2018, theo Báo cáo phát triển bền vững, trang 57; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập, đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-002", "claim": "Năm 2024, phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh giảm 11% so với năm gốc 2018 (Báo cáo kiểm kê khí nhà kính, bảng 9.4; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-003", "claim": "Năm 2024, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây giảm 15% so với năm gốc 2018 (Báo cáo thường niên, trang 67), đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-004", "claim": "So với năm gốc 2021, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 6% lượng nước thải xả ra môi trường tại Nhà máy xút clo Việt Trì Nam trong năm 2025 (Báo cáo phát triển bền vững, trang 30), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-005", "claim": "Trong năm 2024, Công ty Cổ phần Thép Phúc An ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Khu liên hợp gang thép Kỳ Long giảm 12% so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 5.4, đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-006", "claim": "So với năm gốc 2020, Ngân hàng TMCP Phù Sa đã giảm 18% phát thải khí nhà kính phạm vi 1, 2 và 3 của khối chi nhánh phía Nam trong năm 2024 (Phụ lục số liệu môi trường, mục 6.2), đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-007", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Trang trại bò sữa Mộc Châu Xanh của Công ty Cổ phần Sữa Đồng Xanh giảm 4,5% so với năm gốc 2019 (Báo cáo thường niên, trang 14; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-008", "claim": "Trong năm 2023, Ngân hàng TMCP Phù Sa ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Hội sở chính tại Hà Nội giảm 7,2% so với năm gốc 2019, theo Báo cáo thường niên, trang 42, đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-009", "claim": "Trong năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy phân bón Lâm Thao Mới giảm 15% so với năm gốc 2018, theo Báo cáo phát triển bền vững, trang 100, đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-010", "claim": "Năm 2024, phát thải khí nhà kính phạm vi 1 và 2 của hệ thống 120 cửa hàng phía Nam của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 8% so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 30), đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-011", "claim": "Năm 2023, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 8% so với năm gốc 2018 (Báo cáo kiểm kê khí nhà kính, bảng 5.4; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-012", "claim": "Trong năm 2023, Công ty Cổ phần Thép Phúc An ghi nhận lượng nước tiêu thụ tại Khu liên hợp gang thép Kỳ Long giảm 3% so với năm gốc 2021, theo Báo cáo thường niên, trang 30; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập, đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-013", "claim": "So với năm gốc 2018, Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 15% điện năng tiêu thụ tại Trung tâm phân phối Long Hậu trong năm 2025 (Báo cáo phát triển bền vững, trang 62), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-014", "claim": "Trong năm 2025, Ngân hàng TMCP Phù Sa ghi nhận lượng nước thải xả ra môi trường tại khối chi nhánh phía Nam giảm 18% so với năm gốc 2019, theo Báo cáo phát triển bền vững, trang 118; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập, đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-015", "claim": "Trong năm 2023, Công ty Cổ phần Sữa Đồng Xanh ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy sữa Bình Minh giảm 12% so với năm gốc 2020, theo Báo cáo thường niên, trang 113; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập, đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-016", "claim": "Trong năm 2024, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng chất thải rắn đưa đi chôn lấp tại cảng cạn Tân Cảng Bắc giảm 11% so với năm gốc 2019, theo Báo cáo kiểm kê khí nhà kính, bảng 3.3, đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-017", "claim": "Năm 2023, lượng chất thải rắn đưa đi chôn lấp tại khối chi nhánh phía Nam của Ngân hàng TMCP Phù Sa giảm 7,2% so với năm gốc 2021 (Báo cáo phát triển bền vững, trang 135), đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-018", "claim": "So với năm gốc 2020, Công ty Cổ phần Địa ốc An Cư đã giảm 15% phát thải khí nhà kính phạm vi 1 và 2 của Khu đô thị Sông Hậu trong năm 2024 (Báo cáo thường niên, trang 94; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-019", "claim": "Năm 2023, lượng nước thải xả ra môi trường tại Trung tâm phân phối Long Hậu của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 3% so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 5.6; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-020", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 21% so với năm gốc 2019 (Báo cáo thường niên, trang 12; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-021", "claim": "Năm 2025, lượng nước tiêu thụ tại Trung tâm phân phối Long Hậu của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 18% so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 8.3; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-022", "claim": "Năm 2023, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh giảm 11% so với năm gốc 2020 (Báo cáo thường niên, trang 102; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-023", "claim": "Năm 2025, lượng nước tiêu thụ tại Nhà máy xút clo Việt Trì Nam của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 15% so với năm gốc 2020 (Phụ lục số liệu môi trường, mục 4.4; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-024", "claim": "Trong năm 2025, Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng nước thải xả ra môi trường tại Nhà máy nhuộm Phố Nối giảm 21% so với năm gốc 2021, theo Phụ lục số liệu môi trường, mục 6.2, đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-025", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của cảng cạn Tân Cảng Bắc của Công ty Cổ phần Vận tải Cánh Cò giảm 11% so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 9.3), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-026", "claim": "Trong năm 2025, Công ty Cổ phần Thép Phúc An ghi nhận điện năng tiêu thụ tại Khu liên hợp gang thép Kỳ Long giảm 11% so với năm gốc 2021, theo Phụ lục số liệu môi trường, mục 7.6; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập, đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-027", "claim": "So với năm gốc 2021, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 12% lượng nước tiêu thụ tại Nhà máy phân bón Lâm Thao Mới trong năm 2023 (Phụ lục số liệu môi trường, mục 8.5), đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-028", "claim": "Năm 2025, lượng nhiên liệu hóa thạch sử dụng tại hệ thống 120 cửa hàng phía Nam của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 18% so với năm gốc 2018 (Báo cáo kiểm kê khí nhà kính, bảng 3.6; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-029", "claim": "Năm 2025, lượng nhiên liệu hóa thạch sử dụng tại đội xe tải Bắc – Nam của Công ty Cổ phần Vận tải Cánh Cò giảm 3% so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 5.6), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-030", "claim": "So với năm gốc 2018, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 3% phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy nhuộm Phố Nối trong năm 2024 (Phụ lục số liệu môi trường, mục 6.1), đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-031", "claim": "Năm 2023, lượng nước thải xả ra môi trường tại Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây giảm 12% so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 2.3; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-032", "claim": "Năm 2023, lượng nước thải xả ra môi trường tại Trung tâm phân phối Long Hậu của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 15% so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 7.1; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-033", "claim": "So với năm gốc 2020, Tổng Công ty Điện lực Sông Mây đã giảm 15% điện năng tiêu thụ tại Nhà máy nhiệt điện Vĩnh Hảo 3 trong năm 2024 (Phụ lục số liệu môi trường, mục 4.4), đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-034", "claim": "So với năm gốc 2019, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 15% lượng nước thải xả ra môi trường tại Nhà máy may Hòa Khánh trong năm 2025 (Báo cáo thường niên, trang 25; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-035", "claim": "Trong năm 2025, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Trạm nghiền Hiệp Phước giảm 8% so với năm gốc 2018, theo Báo cáo kiểm kê khí nhà kính, bảng 2.5; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập, đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-036", "claim": "Trong năm 2024, Công ty Cổ phần Sữa Đồng Xanh ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy sữa Bình Minh giảm 3% so với năm gốc 2020, theo Báo cáo thường niên, trang 46; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập, đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-037", "claim": "Năm 2023, lượng nhiên liệu hóa thạch sử dụng tại Trang trại bò sữa Mộc Châu Xanh của Công ty Cổ phần Sữa Đồng Xanh giảm 4,5% so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 5.5), đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-038", "claim": "Trong năm 2025, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận lượng nhiên liệu hóa thạch sử dụng tại hệ thống 120 cửa hàng phía Nam giảm 12% so với năm gốc 2021, theo Báo cáo kiểm kê khí nhà kính, bảng 6.3, đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-offset_neutr-1111-039", "claim": "Năm 2024, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 7,2% so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 7.2; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-future_as_ac-1111-000", "claim": "Công ty Cổ phần Thép Phúc An đã giảm 30% điện năng tiêu thụ tại Khu liên hợp gang thép Kỳ Long, hoàn thành mục tiêu năm 2030 so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 8.3).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% điện năng tiêu thụ vào năm 2030 so với năm gốc 2019. Kết quả năm 2024: giảm 9,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-001", "claim": "Công ty Cổ phần Xi măng Đồng Lâm đã giảm 30% lượng nước tiêu thụ tại Trạm nghiền Hiệp Phước, hoàn thành mục tiêu năm 2030 so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 8.6).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nước tiêu thụ vào năm 2030 so với năm gốc 2018. Kết quả năm 2024: giảm 12,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-002", "claim": "Công ty Cổ phần Xi măng Đồng Lâm đã giảm 40% lượng nhiên liệu hóa thạch sử dụng tại Trạm nghiền Hiệp Phước, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 2.3).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% lượng nhiên liệu hóa thạch sử dụng vào năm 2035 so với năm gốc 2018. Kết quả năm 2023: giảm 16,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-003", "claim": "Tổng Công ty Điện lực Sông Mây đã giảm 20% lượng nước thải xả ra môi trường tại Nhà máy thủy điện Đắk Rình, hoàn thành mục tiêu năm 2030 so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 135).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng nước thải xả ra môi trường vào năm 2030 so với năm gốc 2019. Kết quả năm 2025: giảm 8,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-004", "claim": "Ngân hàng TMCP Phù Sa đã giảm 20% phát thải khí nhà kính phạm vi 1 và 2 của Hội sở chính tại Hà Nội, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 123).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% phát thải khí nhà kính vào năm 2035 so với năm gốc 2018. Kết quả năm 2024: giảm 4,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-005", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 40% lượng nước tiêu thụ tại Nhà máy phân bón Lâm Thao Mới, hoàn thành mục tiêu năm 2035 so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 5.3).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% lượng nước tiêu thụ vào năm 2035 so với năm gốc 2019. Kết quả năm 2023: giảm 16,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-006", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 20% phát thải khí nhà kính phạm vi 1 và 2 của hệ thống 120 cửa hàng phía Nam, hoàn thành mục tiêu năm 2030 so với năm gốc 2021 (Báo cáo phát triển bền vững, trang 136).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% phát thải khí nhà kính vào năm 2030 so với năm gốc 2021. Kết quả năm 2023: giảm 4,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-007", "claim": "Tổng Công ty Dệt may Hồng Lĩnh đã giảm 20% điện năng tiêu thụ tại Nhà máy nhuộm Phố Nối, hoàn thành mục tiêu năm 2030 so với năm gốc 2020 (Phụ lục số liệu môi trường, mục 3.3).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% điện năng tiêu thụ vào năm 2030 so với năm gốc 2020. Kết quả năm 2024: giảm 8,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-008", "claim": "Công ty Cổ phần Thép Phúc An đã giảm 25% lượng nước tiêu thụ tại Nhà máy luyện thép Nghi Sơn 2, hoàn thành mục tiêu năm 2035 so với năm gốc 2019 (Báo cáo thường niên, trang 100).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 25% lượng nước tiêu thụ vào năm 2035 so với năm gốc 2019. Kết quả năm 2024: giảm 10,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-009", "claim": "Công ty Cổ phần Vận tải Cánh Cò đã giảm 40% điện năng tiêu thụ tại đội xe tải Bắc – Nam, hoàn thành mục tiêu năm 2035 so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 27).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% điện năng tiêu thụ vào năm 2035 so với năm gốc 2020. Kết quả năm 2025: giảm 8,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-010", "claim": "Công ty Cổ phần Sữa Đồng Xanh đã giảm 30% lượng nước tiêu thụ tại Trang trại bò sữa Mộc Châu Xanh, hoàn thành mục tiêu năm 2030 so với năm gốc 2020 (Báo cáo thường niên, trang 26).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nước tiêu thụ vào năm 2030 so với năm gốc 2020. Kết quả năm 2024: giảm 12,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-011", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 25% lượng chất thải rắn đưa đi chôn lấp tại hệ thống 120 cửa hàng phía Nam, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Báo cáo kiểm kê khí nhà kính, bảng 7.5).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 25% lượng chất thải rắn đưa đi chôn lấp vào năm 2035 so với năm gốc 2018. Kết quả năm 2024: giảm 5,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-012", "claim": "Công ty Cổ phần Sữa Đồng Xanh đã giảm 40% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy sữa Bình Minh, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Báo cáo thường niên, trang 34).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% lượng nhiên liệu hóa thạch sử dụng vào năm 2035 so với năm gốc 2018. Kết quả năm 2025: giảm 8,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-013", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 40% lượng chất thải rắn đưa đi chôn lấp tại Trung tâm phân phối Long Hậu, hoàn thành mục tiêu năm 2035 so với năm gốc 2020 (Phụ lục số liệu môi trường, mục 5.4).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% lượng chất thải rắn đưa đi chôn lấp vào năm 2035 so với năm gốc 2020. Kết quả năm 2024: giảm 16,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-014", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 40% lượng nước thải xả ra môi trường tại Nhà máy phân bón Lâm Thao Mới, hoàn thành mục tiêu năm 2035 so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 9.3).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% lượng nước thải xả ra môi trường vào năm 2035 so với năm gốc 2021. Kết quả năm 2025: giảm 12,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-015", "claim": "Công ty Cổ phần Vận tải Cánh Cò đã giảm 20% lượng nước tiêu thụ tại cảng cạn Tân Cảng Bắc, hoàn thành mục tiêu năm 2030 so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 67).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng nước tiêu thụ vào năm 2030 so với năm gốc 2018. Kết quả năm 2024: giảm 8,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-016", "claim": "Công ty Cổ phần Sữa Đồng Xanh đã giảm 30% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy sữa Bình Minh, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 41).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nhiên liệu hóa thạch sử dụng vào năm 2035 so với năm gốc 2018. Kết quả năm 2024: giảm 9,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-017", "claim": "Ngân hàng TMCP Phù Sa đã giảm 30% lượng nước thải xả ra môi trường tại Hội sở chính tại Hà Nội, hoàn thành mục tiêu năm 2035 so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 3.3).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nước thải xả ra môi trường vào năm 2035 so với năm gốc 2021. Kết quả năm 2023: giảm 6,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-018", "claim": "Tổng Công ty Điện lực Sông Mây đã giảm 30% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy thủy điện Đắk Rình, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Báo cáo kiểm kê khí nhà kính, bảng 7.2).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nhiên liệu hóa thạch sử dụng vào năm 2035 so với năm gốc 2018. Kết quả năm 2023: giảm 12,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-019", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 20% phát thải khí nhà kính phạm vi 1 và 2 của Trung tâm phân phối Long Hậu, hoàn thành mục tiêu năm 2030 so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 8.6).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% phát thải khí nhà kính vào năm 2030 so với năm gốc 2019. Kết quả năm 2023: giảm 6,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-020", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 30% lượng nhiên liệu hóa thạch sử dụng tại Trung tâm phân phối Long Hậu, hoàn thành mục tiêu năm 2030 so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 8.4).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nhiên liệu hóa thạch sử dụng vào năm 2030 so với năm gốc 2020. Kết quả năm 2024: giảm 6,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-021", "claim": "Tổng Công ty Điện lực Sông Mây đã giảm 25% lượng nước tiêu thụ tại Nhà máy thủy điện Đắk Rình, hoàn thành mục tiêu năm 2035 so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 71).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 25% lượng nước tiêu thụ vào năm 2035 so với năm gốc 2020. Kết quả năm 2024: giảm 5,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-022", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 40% điện năng tiêu thụ tại Trung tâm phân phối Long Hậu, hoàn thành mục tiêu năm 2035 so với năm gốc 2019 (Báo cáo thường niên, trang 125).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% điện năng tiêu thụ vào năm 2035 so với năm gốc 2019. Kết quả năm 2025: giảm 12,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-023", "claim": "Tổng Công ty Dệt may Hồng Lĩnh đã giảm 30% lượng nước thải xả ra môi trường tại Nhà máy nhuộm Phố Nối, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 68).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nước thải xả ra môi trường vào năm 2035 so với năm gốc 2018. Kết quả năm 2025: giảm 6,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-024", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 25% lượng nhiên liệu hóa thạch sử dụng tại hệ thống 120 cửa hàng phía Nam, hoàn thành mục tiêu năm 2035 so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 3.1).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 25% lượng nhiên liệu hóa thạch sử dụng vào năm 2035 so với năm gốc 2019. Kết quả năm 2025: giảm 7,5% so với năm gốc."}\n{"id": "A-future_as_ac-1111-025", "claim": "Công ty Cổ phần Địa ốc An Cư đã giảm 40% lượng nhiên liệu hóa thạch sử dụng tại Tòa văn phòng An Cư Tower, hoàn thành mục tiêu năm 2030 so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 4.2).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% lượng nhiên liệu hóa thạch sử dụng vào năm 2030 so với năm gốc 2019. Kết quả năm 2025: giảm 8,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-026", "claim": "Công ty Cổ phần Địa ốc An Cư đã giảm 30% lượng nhiên liệu hóa thạch sử dụng tại Tòa văn phòng An Cư Tower, hoàn thành mục tiêu năm 2030 so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 60).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nhiên liệu hóa thạch sử dụng vào năm 2030 so với năm gốc 2019. Kết quả năm 2025: giảm 6,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-027", "claim": "Công ty Cổ phần Thép Phúc An đã giảm 20% lượng chất thải rắn đưa đi chôn lấp tại Khu liên hợp gang thép Kỳ Long, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 116).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng chất thải rắn đưa đi chôn lấp vào năm 2035 so với năm gốc 2018. Kết quả năm 2023: giảm 6,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-028", "claim": "Công ty Cổ phần Xi măng Đồng Lâm đã giảm 40% lượng chất thải rắn đưa đi chôn lấp tại Trạm nghiền Hiệp Phước, hoàn thành mục tiêu năm 2035 so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 97).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% lượng chất thải rắn đưa đi chôn lấp vào năm 2035 so với năm gốc 2020. Kết quả năm 2023: giảm 8,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-029", "claim": "Công ty Cổ phần Sữa Đồng Xanh đã giảm 25% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy sữa Bình Minh, hoàn thành mục tiêu năm 2035 so với năm gốc 2020 (Phụ lục số liệu môi trường, mục 5.5).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 25% phát thải khí nhà kính vào năm 2035 so với năm gốc 2020. Kết quả năm 2024: giảm 7,5% so với năm gốc."}\n{"id": "A-future_as_ac-1111-030", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 20% điện năng tiêu thụ tại Trung tâm phân phối Long Hậu, hoàn thành mục tiêu năm 2030 so với năm gốc 2021 (Báo cáo thường niên, trang 91).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% điện năng tiêu thụ vào năm 2030 so với năm gốc 2021. Kết quả năm 2023: giảm 4,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-031", "claim": "Công ty Cổ phần Thép Phúc An đã giảm 30% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy luyện thép Nghi Sơn 2, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 2.4).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nhiên liệu hóa thạch sử dụng vào năm 2035 so với năm gốc 2018. Kết quả năm 2024: giảm 12,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-032", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 20% lượng nước thải xả ra môi trường tại hệ thống 120 cửa hàng phía Nam, hoàn thành mục tiêu năm 2035 so với năm gốc 2019 (Báo cáo thường niên, trang 56).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng nước thải xả ra môi trường vào năm 2035 so với năm gốc 2019. Kết quả năm 2023: giảm 6,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-033", "claim": "Công ty Cổ phần Xi măng Đồng Lâm đã giảm 30% lượng nước thải xả ra môi trường tại Trạm nghiền Hiệp Phước, hoàn thành mục tiêu năm 2030 so với năm gốc 2021 (Báo cáo phát triển bền vững, trang 73).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng nước thải xả ra môi trường vào năm 2030 so với năm gốc 2021. Kết quả năm 2023: giảm 9,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-034", "claim": "Công ty Cổ phần Địa ốc An Cư đã giảm 20% lượng nhiên liệu hóa thạch sử dụng tại Khu đô thị Sông Hậu, hoàn thành mục tiêu năm 2030 so với năm gốc 2020 (Báo cáo thường niên, trang 26).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng nhiên liệu hóa thạch sử dụng vào năm 2030 so với năm gốc 2020. Kết quả năm 2025: giảm 6,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-035", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 20% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xút clo Việt Trì Nam, hoàn thành mục tiêu năm 2030 so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 4.6).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng chất thải rắn đưa đi chôn lấp vào năm 2030 so với năm gốc 2020. Kết quả năm 2025: giảm 4,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-036", "claim": "Công ty Cổ phần Địa ốc An Cư đã giảm 25% lượng nước thải xả ra môi trường tại Khu đô thị Sông Hậu, hoàn thành mục tiêu năm 2035 so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 83).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 25% lượng nước thải xả ra môi trường vào năm 2035 so với năm gốc 2018. Kết quả năm 2025: giảm 10,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-037", "claim": "Công ty Cổ phần Thép Phúc An đã giảm 40% lượng chất thải rắn đưa đi chôn lấp tại Khu liên hợp gang thép Kỳ Long, hoàn thành mục tiêu năm 2035 so với năm gốc 2021 (Báo cáo thường niên, trang 107).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% lượng chất thải rắn đưa đi chôn lấp vào năm 2035 so với năm gốc 2021. Kết quả năm 2024: giảm 16,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-038", "claim": "Công ty Cổ phần Xi măng Đồng Lâm đã giảm 20% phát thải khí nhà kính phạm vi 1, 2 và 3 của Trạm nghiền Hiệp Phước, hoàn thành mục tiêu năm 2035 so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 2.6).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% phát thải khí nhà kính vào năm 2035 so với năm gốc 2021. Kết quả năm 2024: giảm 6,0% so với năm gốc."}\n{"id": "A-future_as_ac-1111-039", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 25% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy xút clo Việt Trì Nam, hoàn thành mục tiêu năm 2030 so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 51).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 25% lượng nhiên liệu hóa thạch sử dụng vào năm 2030 so với năm gốc 2020. Kết quả năm 2025: giảm 7,5% so với năm gốc."}\n{"id": "A-unverified_l-1111-000", "claim": "Năm 2023, Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-001", "claim": "Năm 2023, Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh đạt chứng nhận nhãn Sinh thái Việt Nam.", "evidence": ""}\n{"id": "A-unverified_l-1111-002", "claim": "Năm 2023, Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm đạt chứng nhận ISO 14001:2015.", "evidence": ""}\n{"id": "A-unverified_l-1111-003", "claim": "Năm 2025, Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh đạt chứng nhận nhãn Sinh thái Việt Nam.", "evidence": ""}\n{"id": "A-unverified_l-1111-004", "claim": "Năm 2023, Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh đạt chứng nhận LEED Gold.", "evidence": ""}\n{"id": "A-unverified_l-1111-005", "claim": "Năm 2025, Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-006", "claim": "Năm 2023, Trang trại bò sữa Mộc Châu Xanh của Công ty Cổ phần Sữa Đồng Xanh đạt chứng nhận nhãn Sinh thái Việt Nam.", "evidence": ""}\n{"id": "A-unverified_l-1111-007", "claim": "Năm 2024, khối chi nhánh phía Nam của Ngân hàng TMCP Phù Sa đạt chứng nhận nhãn Sinh thái Việt Nam.", "evidence": ""}\n{"id": "A-unverified_l-1111-008", "claim": "Năm 2023, Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-009", "claim": "Năm 2025, Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm đạt chứng nhận EDGE Advanced.", "evidence": ""}\n{"id": "A-unverified_l-1111-010", "claim": "Năm 2025, Hội sở chính tại Hà Nội của Ngân hàng TMCP Phù Sa đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-011", "claim": "Năm 2025, Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh đạt chứng nhận LEED Gold.", "evidence": ""}\n{"id": "A-unverified_l-1111-012", "claim": "Năm 2025, Nhà máy luyện thép Nghi Sơn 2 của Công ty Cổ phần Thép Phúc An đạt chứng nhận ISO 14001:2015.", "evidence": ""}\n{"id": "A-unverified_l-1111-013", "claim": "Năm 2024, Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh đạt chứng nhận LEED Gold.", "evidence": ""}\n{"id": "A-unverified_l-1111-014", "claim": "Năm 2025, Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-015", "claim": "Năm 2023, Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-016", "claim": "Năm 2023, cảng cạn Tân Cảng Bắc của Công ty Cổ phần Vận tải Cánh Cò đạt chứng nhận LEED Gold.", "evidence": ""}\n{"id": "A-unverified_l-1111-017", "claim": "Năm 2025, hệ thống 120 cửa hàng phía Nam của Công ty Cổ phần Bán lẻ Lạc Xuân đạt chứng nhận LEED Gold.", "evidence": ""}\n{"id": "A-unverified_l-1111-018", "claim": "Năm 2025, Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư đạt chứng nhận EDGE Advanced.", "evidence": ""}\n{"id": "A-unverified_l-1111-019", "claim": "Năm 2025, Trạm nghiền Hiệp Phước của Công ty Cổ phần Xi măng Đồng Lâm đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-020", "claim": "Năm 2025, Nhà máy xút clo Việt Trì Nam của Công ty Cổ phần Hóa chất Đức Giang Nam đạt chứng nhận EDGE Advanced.", "evidence": ""}\n{"id": "A-unverified_l-1111-021", "claim": "Năm 2025, Trang trại bò sữa Mộc Châu Xanh của Công ty Cổ phần Sữa Đồng Xanh đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-022", "claim": "Năm 2024, Nhà máy sữa Bình Minh của Công ty Cổ phần Sữa Đồng Xanh đạt chứng nhận LEED Gold.", "evidence": ""}\n{"id": "A-unverified_l-1111-023", "claim": "Năm 2023, Trạm nghiền Hiệp Phước của Công ty Cổ phần Xi măng Đồng Lâm đạt chứng nhận ISO 14001:2015.", "evidence": ""}\n{"id": "A-unverified_l-1111-024", "claim": "Năm 2023, Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-025", "claim": "Năm 2024, cảng cạn Tân Cảng Bắc của Công ty Cổ phần Vận tải Cánh Cò đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-026", "claim": "Năm 2023, Nhà máy sữa Bình Minh của Công ty Cổ phần Sữa Đồng Xanh đạt chứng nhận EDGE Advanced.", "evidence": ""}\n{"id": "A-unverified_l-1111-027", "claim": "Năm 2025, Trạm nghiền Hiệp Phước của Công ty Cổ phần Xi măng Đồng Lâm đạt chứng nhận ISO 14001:2015.", "evidence": ""}\n{"id": "A-unverified_l-1111-028", "claim": "Năm 2023, Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh đạt chứng nhận EDGE Advanced.", "evidence": ""}\n{"id": "A-unverified_l-1111-029", "claim": "Năm 2025, cảng cạn Tân Cảng Bắc của Công ty Cổ phần Vận tải Cánh Cò đạt chứng nhận nhãn Sinh thái Việt Nam.", "evidence": ""}\n{"id": "A-unverified_l-1111-030", "claim": "Năm 2025, khối chi nhánh phía Nam của Ngân hàng TMCP Phù Sa đạt chứng nhận LEED Gold.", "evidence": ""}\n{"id": "A-unverified_l-1111-031", "claim": "Năm 2024, Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-032", "claim": "Năm 2024, Trung tâm phân phối Long Hậu của Công ty Cổ phần Bán lẻ Lạc Xuân đạt chứng nhận EDGE Advanced.", "evidence": ""}\n{"id": "A-unverified_l-1111-033", "claim": "Năm 2024, Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam đạt chứng nhận EDGE Advanced.", "evidence": ""}\n{"id": "A-unverified_l-1111-034", "claim": "Năm 2023, khối chi nhánh phía Nam của Ngân hàng TMCP Phù Sa đạt chứng nhận nhãn Sinh thái Việt Nam.", "evidence": ""}\n{"id": "A-unverified_l-1111-035", "claim": "Năm 2023, hệ thống 120 cửa hàng phía Nam của Công ty Cổ phần Bán lẻ Lạc Xuân đạt chứng nhận EDGE Advanced.", "evidence": ""}\n{"id": "A-unverified_l-1111-036", "claim": "Năm 2023, hệ thống 120 cửa hàng phía Nam của Công ty Cổ phần Bán lẻ Lạc Xuân đạt chứng nhận ISO 50001:2018.", "evidence": ""}\n{"id": "A-unverified_l-1111-037", "claim": "Năm 2025, Trang trại bò sữa Mộc Châu Xanh của Công ty Cổ phần Sữa Đồng Xanh đạt chứng nhận ISO 14001:2015.", "evidence": ""}\n{"id": "A-unverified_l-1111-038", "claim": "Năm 2024, Nhà máy thủy điện Đắk Rình của Tổng Công ty Điện lực Sông Mây đạt chứng nhận EDGE Advanced.", "evidence": ""}\n{"id": "A-unverified_l-1111-039", "claim": "Năm 2024, Nhà máy thủy điện Đắk Rình của Tổng Công ty Điện lực Sông Mây đạt chứng nhận LEED Gold.", "evidence": ""}\n{"id": "A-no_evidence_-1111-000", "claim": "So với năm gốc 2018, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 9,5% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy xút clo Việt Trì Nam trong năm 2024.", "evidence": ""}\n{"id": "A-no_evidence_-1111-001", "claim": "Trong năm 2023, Ngân hàng TMCP Phù Sa ghi nhận điện năng tiêu thụ tại khối chi nhánh phía Nam giảm 15% so với năm gốc 2018.", "evidence": ""}\n{"id": "A-no_evidence_-1111-002", "claim": "Năm 2023, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy luyện thép Nghi Sơn 2 của Công ty Cổ phần Thép Phúc An giảm 3% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-no_evidence_-1111-003", "claim": "Trong năm 2025, Công ty Cổ phần Địa ốc An Cư ghi nhận lượng nước thải xả ra môi trường tại Tòa văn phòng An Cư Tower giảm 15% so với năm gốc 2020.", "evidence": ""}\n{"id": "A-no_evidence_-1111-004", "claim": "Trong năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng nước tiêu thụ tại Nhà máy xút clo Việt Trì Nam giảm 12% so với năm gốc 2020.", "evidence": ""}\n{"id": "A-no_evidence_-1111-005", "claim": "So với năm gốc 2021, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 24% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xi măng Lương Sơn trong năm 2024.", "evidence": ""}\n{"id": "A-no_evidence_-1111-006", "claim": "So với năm gốc 2020, Công ty Cổ phần Thép Phúc An đã giảm 24% phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy luyện thép Nghi Sơn 2 trong năm 2023.", "evidence": ""}\n{"id": "A-no_evidence_-1111-007", "claim": "Trong năm 2023, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy thủy điện Đắk Rình giảm 6% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-no_evidence_-1111-008", "claim": "Trong năm 2023, Công ty Cổ phần Thép Phúc An ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy luyện thép Nghi Sơn 2 giảm 11% so với năm gốc 2020.", "evidence": ""}\n{"id": "A-no_evidence_-1111-009", "claim": "Năm 2023, điện năng tiêu thụ tại Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh giảm 8% so với năm gốc 2018.", "evidence": ""}\n{"id": "A-no_evidence_-1111-010", "claim": "Năm 2024, lượng nước thải xả ra môi trường tại Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 9,5% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-no_evidence_-1111-011", "claim": "So với năm gốc 2020, Công ty Cổ phần Địa ốc An Cư đã giảm 6% lượng nước thải xả ra môi trường tại Tòa văn phòng An Cư Tower trong năm 2024.", "evidence": ""}\n{"id": "A-no_evidence_-1111-012", "claim": "So với năm gốc 2020, Công ty Cổ phần Vận tải Cánh Cò đã giảm 8% lượng nước thải xả ra môi trường tại đội xe tải Bắc – Nam trong năm 2023.", "evidence": ""}\n{"id": "A-no_evidence_-1111-013", "claim": "Trong năm 2024, Ngân hàng TMCP Phù Sa ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Hội sở chính tại Hà Nội giảm 8% so với năm gốc 2020.", "evidence": ""}\n{"id": "A-no_evidence_-1111-014", "claim": "So với năm gốc 2018, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 4,5% lượng nhiên liệu hóa thạch sử dụng tại Nhà máy nhuộm Phố Nối trong năm 2024.", "evidence": ""}\n{"id": "A-no_evidence_-1111-015", "claim": "Trong năm 2023, Công ty Cổ phần Thép Phúc An ghi nhận điện năng tiêu thụ tại Nhà máy luyện thép Nghi Sơn 2 giảm 7,2% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-no_evidence_-1111-016", "claim": "Năm 2025, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh giảm 6% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-no_evidence_-1111-017", "claim": "So với năm gốc 2019, Công ty Cổ phần Vận tải Cánh Cò đã giảm 3% lượng chất thải rắn đưa đi chôn lấp tại cảng cạn Tân Cảng Bắc trong năm 2023.", "evidence": ""}\n{"id": "A-no_evidence_-1111-018", "claim": "Trong năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận điện năng tiêu thụ tại Nhà máy phân bón Lâm Thao Mới giảm 4,5% so với năm gốc 2021.", "evidence": ""}\n{"id": "A-no_evidence_-1111-019", "claim": "So với năm gốc 2020, Ngân hàng TMCP Phù Sa đã giảm 6% phát thải khí nhà kính phạm vi 1 và 2 của khối chi nhánh phía Nam trong năm 2025.", "evidence": ""}\n{"id": "A-no_evidence_-1111-020", "claim": "Trong năm 2024, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng nước tiêu thụ tại cảng cạn Tân Cảng Bắc giảm 3% so với năm gốc 2020.", "evidence": ""}\n{"id": "A-no_evidence_-1111-021", "claim": "Năm 2023, lượng nước tiêu thụ tại Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An giảm 15% so với năm gốc 2018.", "evidence": ""}\n{"id": "A-no_evidence_-1111-022", "claim": "So với năm gốc 2018, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 6% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy may Hòa Khánh trong năm 2024.", "evidence": ""}\n{"id": "A-no_evidence_-1111-023", "claim": "Năm 2024, phát thải khí nhà kính phạm vi 1, 2 và 3 của Trạm nghiền Hiệp Phước của Công ty Cổ phần Xi măng Đồng Lâm giảm 3% so với năm gốc 2020.", "evidence": ""}\n{"id": "A-no_evidence_-1111-024", "claim": "Trong năm 2024, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận điện năng tiêu thụ tại Nhà máy xi măng Lương Sơn giảm 11% so với năm gốc 2021.", "evidence": ""}\n{"id": "A-no_evidence_-1111-025", "claim": "Năm 2024, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 24% so với năm gốc 2020.", "evidence": ""}\n{"id": "A-no_evidence_-1111-026", "claim": "Trong năm 2024, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Trung tâm phân phối Long Hậu giảm 4,5% so với năm gốc 2018.", "evidence": ""}\n{"id": "A-no_evidence_-1111-027", "claim": "Trong năm 2025, Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy may Hòa Khánh giảm 24% so với năm gốc 2020.", "evidence": ""}\n{"id": "A-no_evidence_-1111-028", "claim": "Năm 2023, phát thải khí nhà kính phạm vi 1 và 2 của đội xe tải Bắc – Nam của Công ty Cổ phần Vận tải Cánh Cò giảm 15% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-no_evidence_-1111-029", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại Hội sở chính tại Hà Nội của Ngân hàng TMCP Phù Sa giảm 8% so với năm gốc 2021.", "evidence": ""}\n{"id": "A-no_evidence_-1111-030", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1, 2 và 3 của Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An giảm 12% so với năm gốc 2018.", "evidence": ""}\n{"id": "A-no_evidence_-1111-031", "claim": "So với năm gốc 2019, Công ty Cổ phần Địa ốc An Cư đã giảm 7,2% phát thải khí nhà kính phạm vi 1 và 2 của Khu đô thị Sông Hậu trong năm 2024.", "evidence": ""}\n{"id": "A-no_evidence_-1111-032", "claim": "Năm 2024, lượng nước thải xả ra môi trường tại Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An giảm 24% so với năm gốc 2021.", "evidence": ""}\n{"id": "A-no_evidence_-1111-033", "claim": "Năm 2025, lượng chất thải rắn đưa đi chôn lấp tại Trung tâm phân phối Long Hậu của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 9,5% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-no_evidence_-1111-034", "claim": "Trong năm 2025, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xút clo Việt Trì Nam giảm 15% so với năm gốc 2018.", "evidence": ""}\n{"id": "A-no_evidence_-1111-035", "claim": "Trong năm 2025, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Trung tâm phân phối Long Hậu giảm 11% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-no_evidence_-1111-036", "claim": "Trong năm 2023, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng nước thải xả ra môi trường tại đội xe tải Bắc – Nam giảm 3% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-no_evidence_-1111-037", "claim": "So với năm gốc 2018, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 6% lượng nước thải xả ra môi trường tại Nhà máy phân bón Lâm Thao Mới trong năm 2023.", "evidence": ""}\n{"id": "A-no_evidence_-1111-038", "claim": "So với năm gốc 2019, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 4,5% lượng nước tiêu thụ tại Nhà máy phân bón Lâm Thao Mới trong năm 2023.", "evidence": ""}\n{"id": "A-no_evidence_-1111-039", "claim": "Năm 2024, lượng nước tiêu thụ tại khối chi nhánh phía Nam của Ngân hàng TMCP Phù Sa giảm 3% so với năm gốc 2018.", "evidence": ""}\n{"id": "A-hidden_trade-1111-000", "claim": "Năm 2024, Công ty Cổ phần Địa ốc An Cư lắp đặt hệ thống điện mặt trời áp mái công suất 350 kWp tại văn phòng (Báo cáo phát triển bền vững, trang 75).", "evidence": "Năm 2024, điện năng tiêu thụ từ lưới điện than của Địa ốc An Cư tăng 12% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-001", "claim": "Năm 2023, Công ty Cổ phần Xi măng Đồng Lâm chuyển 20% xe nâng trong kho sang chạy điện (Báo cáo phát triển bền vững, trang 32).", "evidence": "Năm 2023, lượng dầu diesel tiêu thụ của đội xe vận tải của Xi măng Đồng Lâm tăng 12% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-002", "claim": "Năm 2025, Ngân hàng TMCP Phù Sa chuyển 15% xe nâng trong kho sang chạy điện (Phụ lục số liệu môi trường, mục 4.3).", "evidence": "Năm 2025, lượng dầu diesel tiêu thụ của đội xe vận tải của Ngân hàng Phù Sa tăng 34% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-003", "claim": "Năm 2024, Ngân hàng TMCP Phù Sa trồng 1500 cây xanh tại khu vực nhà máy (Báo cáo thường niên, trang 101).", "evidence": "Năm 2024, diện tích rừng tự nhiên bị chuyển đổi cho dự án mở rộng của Ngân hàng Phù Sa tăng 12% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-004", "claim": "Năm 2024, Tổng Công ty Điện lực Sông Mây chuyển 15% xe nâng trong kho sang chạy điện (Báo cáo kiểm kê khí nhà kính, bảng 4.2).", "evidence": "Năm 2024, lượng dầu diesel tiêu thụ của đội xe vận tải của Điện lực Sông Mây tăng 12% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-005", "claim": "Năm 2025, Công ty Cổ phần Sữa Đồng Xanh giảm 40% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Báo cáo kiểm kê khí nhà kính, bảng 3.3).", "evidence": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh tăng 25% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-006", "claim": "Năm 2023, Công ty Cổ phần Địa ốc An Cư chuyển 30% xe nâng trong kho sang chạy điện (Báo cáo kiểm kê khí nhà kính, bảng 8.3).", "evidence": "Năm 2023, lượng dầu diesel tiêu thụ của đội xe vận tải của Địa ốc An Cư tăng 18% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-007", "claim": "Năm 2023, Công ty Cổ phần Địa ốc An Cư chuyển 15% xe nâng trong kho sang chạy điện (Phụ lục số liệu môi trường, mục 8.3).", "evidence": "Năm 2023, lượng dầu diesel tiêu thụ của đội xe vận tải của Địa ốc An Cư tăng 12% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-008", "claim": "Năm 2025, Công ty Cổ phần Thép Phúc An trồng 350 cây xanh tại khu vực nhà máy (Phụ lục số liệu môi trường, mục 2.1).", "evidence": "Năm 2025, diện tích rừng tự nhiên bị chuyển đổi cho dự án mở rộng của Thép Phúc An tăng 25% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-009", "claim": "Năm 2023, Công ty Cổ phần Địa ốc An Cư giảm 40% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Phụ lục số liệu môi trường, mục 4.4).", "evidence": "Năm 2023, phát thải khí nhà kính phạm vi 1 và 2 của Địa ốc An Cư tăng 18% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-010", "claim": "Năm 2025, Công ty Cổ phần Sữa Đồng Xanh chuyển 15% xe nâng trong kho sang chạy điện (Báo cáo thường niên, trang 87).", "evidence": "Năm 2025, lượng dầu diesel tiêu thụ của đội xe vận tải của Sữa Đồng Xanh tăng 25% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-011", "claim": "Năm 2025, Ngân hàng TMCP Phù Sa giảm 15% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Báo cáo thường niên, trang 128).", "evidence": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa tăng 18% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-012", "claim": "Năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam giảm 15% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Báo cáo thường niên, trang 91).", "evidence": "Năm 2024, phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam tăng 25% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-013", "claim": "Năm 2023, Tổng Công ty Dệt may Hồng Lĩnh lắp đặt hệ thống điện mặt trời áp mái công suất 350 kWp tại văn phòng (Báo cáo kiểm kê khí nhà kính, bảng 2.6).", "evidence": "Năm 2023, điện năng tiêu thụ từ lưới điện than của Dệt may Hồng Lĩnh tăng 12% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-014", "claim": "Năm 2025, Tổng Công ty Điện lực Sông Mây trồng 800 cây xanh tại khu vực nhà máy (Báo cáo kiểm kê khí nhà kính, bảng 7.4).", "evidence": "Năm 2025, diện tích rừng tự nhiên bị chuyển đổi cho dự án mở rộng của Điện lực Sông Mây tăng 12% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-015", "claim": "Năm 2023, Ngân hàng TMCP Phù Sa lắp đặt hệ thống điện mặt trời áp mái công suất 120 kWp tại văn phòng (Báo cáo thường niên, trang 62).", "evidence": "Năm 2023, điện năng tiêu thụ từ lưới điện than của Ngân hàng Phù Sa tăng 12% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-016", "claim": "Năm 2023, Công ty Cổ phần Sữa Đồng Xanh chuyển 15% xe nâng trong kho sang chạy điện (Báo cáo kiểm kê khí nhà kính, bảng 6.1).", "evidence": "Năm 2023, lượng dầu diesel tiêu thụ của đội xe vận tải của Sữa Đồng Xanh tăng 18% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-017", "claim": "Năm 2025, Công ty Cổ phần Vận tải Cánh Cò lắp đặt hệ thống điện mặt trời áp mái công suất 1500 kWp tại văn phòng (Phụ lục số liệu môi trường, mục 2.3).", "evidence": "Năm 2025, điện năng tiêu thụ từ lưới điện than của Vận tải Cánh Cò tăng 18% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-018", "claim": "Năm 2024, Công ty Cổ phần Xi măng Đồng Lâm giảm 15% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Báo cáo phát triển bền vững, trang 62).", "evidence": "Năm 2024, phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm tăng 12% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-019", "claim": "Năm 2024, Công ty Cổ phần Thép Phúc An giảm 20% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Phụ lục số liệu môi trường, mục 3.5).", "evidence": "Năm 2024, phát thải khí nhà kính phạm vi 1 và 2 của Thép Phúc An tăng 34% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-020", "claim": "Năm 2023, Tổng Công ty Điện lực Sông Mây chuyển 15% xe nâng trong kho sang chạy điện (Báo cáo thường niên, trang 60).", "evidence": "Năm 2023, lượng dầu diesel tiêu thụ của đội xe vận tải của Điện lực Sông Mây tăng 18% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-021", "claim": "Năm 2024, Công ty Cổ phần Vận tải Cánh Cò lắp đặt hệ thống điện mặt trời áp mái công suất 800 kWp tại văn phòng (Báo cáo thường niên, trang 92).", "evidence": "Năm 2024, điện năng tiêu thụ từ lưới điện than của Vận tải Cánh Cò tăng 34% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-022", "claim": "Năm 2025, Công ty Cổ phần Thép Phúc An trồng 120 cây xanh tại khu vực nhà máy (Báo cáo kiểm kê khí nhà kính, bảng 4.4).", "evidence": "Năm 2025, diện tích rừng tự nhiên bị chuyển đổi cho dự án mở rộng của Thép Phúc An tăng 18% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-023", "claim": "Năm 2023, Công ty Cổ phần Bán lẻ Lạc Xuân giảm 40% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Báo cáo phát triển bền vững, trang 78).", "evidence": "Năm 2023, phát thải khí nhà kính phạm vi 1 và 2 của Bán lẻ Lạc Xuân tăng 25% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-024", "claim": "Năm 2025, Công ty Cổ phần Địa ốc An Cư chuyển 15% xe nâng trong kho sang chạy điện (Báo cáo kiểm kê khí nhà kính, bảng 7.5).", "evidence": "Năm 2025, lượng dầu diesel tiêu thụ của đội xe vận tải của Địa ốc An Cư tăng 18% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-025", "claim": "Năm 2023, Ngân hàng TMCP Phù Sa chuyển 15% xe nâng trong kho sang chạy điện (Báo cáo phát triển bền vững, trang 85).", "evidence": "Năm 2023, lượng dầu diesel tiêu thụ của đội xe vận tải của Ngân hàng Phù Sa tăng 18% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-026", "claim": "Năm 2023, Công ty Cổ phần Vận tải Cánh Cò chuyển 30% xe nâng trong kho sang chạy điện (Phụ lục số liệu môi trường, mục 8.4).", "evidence": "Năm 2023, lượng dầu diesel tiêu thụ của đội xe vận tải của Vận tải Cánh Cò tăng 25% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-027", "claim": "Năm 2025, Tổng Công ty Dệt may Hồng Lĩnh chuyển 15% xe nâng trong kho sang chạy điện (Phụ lục số liệu môi trường, mục 4.4).", "evidence": "Năm 2025, lượng dầu diesel tiêu thụ của đội xe vận tải của Dệt may Hồng Lĩnh tăng 34% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-028", "claim": "Năm 2024, Công ty Cổ phần Xi măng Đồng Lâm trồng 120 cây xanh tại khu vực nhà máy (Báo cáo phát triển bền vững, trang 44).", "evidence": "Năm 2024, diện tích rừng tự nhiên bị chuyển đổi cho dự án mở rộng của Xi măng Đồng Lâm tăng 34% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-029", "claim": "Năm 2023, Công ty Cổ phần Sữa Đồng Xanh chuyển 15% xe nâng trong kho sang chạy điện (Báo cáo phát triển bền vững, trang 69).", "evidence": "Năm 2023, lượng dầu diesel tiêu thụ của đội xe vận tải của Sữa Đồng Xanh tăng 25% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-030", "claim": "Năm 2023, Công ty Cổ phần Bán lẻ Lạc Xuân giảm 40% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Báo cáo thường niên, trang 77).", "evidence": "Năm 2023, phát thải khí nhà kính phạm vi 1 và 2 của Bán lẻ Lạc Xuân tăng 12% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-031", "claim": "Năm 2024, Công ty Cổ phần Thép Phúc An chuyển 20% xe nâng trong kho sang chạy điện (Báo cáo kiểm kê khí nhà kính, bảng 2.5).", "evidence": "Năm 2024, lượng dầu diesel tiêu thụ của đội xe vận tải của Thép Phúc An tăng 18% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-032", "claim": "Năm 2025, Công ty Cổ phần Xi măng Đồng Lâm chuyển 15% xe nâng trong kho sang chạy điện (Báo cáo thường niên, trang 80).", "evidence": "Năm 2025, lượng dầu diesel tiêu thụ của đội xe vận tải của Xi măng Đồng Lâm tăng 25% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-033", "claim": "Năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam lắp đặt hệ thống điện mặt trời áp mái công suất 120 kWp tại văn phòng (Phụ lục số liệu môi trường, mục 5.4).", "evidence": "Năm 2024, điện năng tiêu thụ từ lưới điện than của Hóa chất Đức Giang Nam tăng 18% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-034", "claim": "Năm 2024, Tổng Công ty Điện lực Sông Mây trồng 1500 cây xanh tại khu vực nhà máy (Báo cáo thường niên, trang 45).", "evidence": "Năm 2024, diện tích rừng tự nhiên bị chuyển đổi cho dự án mở rộng của Điện lực Sông Mây tăng 18% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-035", "claim": "Năm 2025, Tổng Công ty Điện lực Sông Mây giảm 15% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Báo cáo phát triển bền vững, trang 126).", "evidence": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây tăng 12% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-036", "claim": "Năm 2025, Công ty Cổ phần Bán lẻ Lạc Xuân chuyển 30% xe nâng trong kho sang chạy điện (Báo cáo thường niên, trang 79).", "evidence": "Năm 2025, lượng dầu diesel tiêu thụ của đội xe vận tải của Bán lẻ Lạc Xuân tăng 12% so với năm 2024 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-037", "claim": "Năm 2023, Công ty Cổ phần Vận tải Cánh Cò lắp đặt hệ thống điện mặt trời áp mái công suất 1500 kWp tại văn phòng (Báo cáo phát triển bền vững, trang 128).", "evidence": "Năm 2023, điện năng tiêu thụ từ lưới điện than của Vận tải Cánh Cò tăng 18% so với năm 2022 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-038", "claim": "Năm 2024, Công ty Cổ phần Sữa Đồng Xanh lắp đặt hệ thống điện mặt trời áp mái công suất 1500 kWp tại văn phòng (Báo cáo phát triển bền vững, trang 102).", "evidence": "Năm 2024, điện năng tiêu thụ từ lưới điện than của Sữa Đồng Xanh tăng 18% so với năm 2023 do mở rộng công suất."}\n{"id": "A-hidden_trade-1111-039", "claim": "Năm 2023, Tổng Công ty Dệt may Hồng Lĩnh lắp đặt hệ thống điện mặt trời áp mái công suất 1500 kWp tại văn phòng (Phụ lục số liệu môi trường, mục 9.5).", "evidence": "Năm 2023, điện năng tiêu thụ từ lưới điện than của Dệt may Hồng Lĩnh tăng 12% so với năm 2022 do mở rộng công suất."}\n{"id": "A-irrelevant_c-1111-000", "claim": "Công ty Cổ phần Vận tải Cánh Cò khẳng định vị thế tiên phong khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-001", "claim": "Công ty Cổ phần Địa ốc An Cư tự hào dẫn đầu ngành khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-002", "claim": "Ngân hàng TMCP Phù Sa tự hào dẫn đầu ngành khi thu gom và chuyển giao chất thải nguy hại cho đơn vị có giấy phép.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-003", "claim": "Tổng Công ty Điện lực Sông Mây tự hào dẫn đầu ngành khi không sử dụng môi chất làm lạnh CFC trong sản phẩm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-004", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân khẳng định vị thế tiên phong khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-005", "claim": "Công ty Cổ phần Vận tải Cánh Cò tự hào dẫn đầu ngành khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-006", "claim": "Công ty Cổ phần Xi măng Đồng Lâm tự hào dẫn đầu ngành khi lắp đặt hệ thống quan trắc nước thải tự động, truyền dữ liệu về Sở Tài nguyên và Môi trường.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-007", "claim": "Ngân hàng TMCP Phù Sa tự hào dẫn đầu ngành khi không sử dụng môi chất làm lạnh CFC trong sản phẩm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-008", "claim": "Tổng Công ty Điện lực Sông Mây tự hào dẫn đầu ngành khi lắp đặt hệ thống quan trắc nước thải tự động, truyền dữ liệu về Sở Tài nguyên và Môi trường.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-009", "claim": "Công ty Cổ phần Sữa Đồng Xanh tự hào là doanh nghiệp tiên phong khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-010", "claim": "Công ty Cổ phần Thép Phúc An khẳng định vị thế tiên phong khi lắp đặt hệ thống quan trắc nước thải tự động, truyền dữ liệu về Sở Tài nguyên và Môi trường.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-011", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân tự hào dẫn đầu ngành khi thu gom và chuyển giao chất thải nguy hại cho đơn vị có giấy phép.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-012", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam khẳng định vị thế tiên phong khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-013", "claim": "Công ty Cổ phần Sữa Đồng Xanh tự hào là doanh nghiệp tiên phong khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-014", "claim": "Tổng Công ty Dệt may Hồng Lĩnh khẳng định vị thế tiên phong khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-015", "claim": "Tổng Công ty Điện lực Sông Mây khẳng định vị thế tiên phong khi không sử dụng môi chất làm lạnh CFC trong sản phẩm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-016", "claim": "Ngân hàng TMCP Phù Sa tự hào dẫn đầu ngành khi lắp đặt hệ thống quan trắc nước thải tự động, truyền dữ liệu về Sở Tài nguyên và Môi trường.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-017", "claim": "Công ty Cổ phần Thép Phúc An tự hào dẫn đầu ngành khi không sử dụng môi chất làm lạnh CFC trong sản phẩm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-018", "claim": "Công ty Cổ phần Vận tải Cánh Cò tự hào là doanh nghiệp tiên phong khi lắp đặt hệ thống quan trắc nước thải tự động, truyền dữ liệu về Sở Tài nguyên và Môi trường.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-019", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân tự hào dẫn đầu ngành khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-020", "claim": "Ngân hàng TMCP Phù Sa tự hào là doanh nghiệp tiên phong khi thu gom và chuyển giao chất thải nguy hại cho đơn vị có giấy phép.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-021", "claim": "Ngân hàng TMCP Phù Sa tự hào dẫn đầu ngành khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-022", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam tự hào là doanh nghiệp tiên phong khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-023", "claim": "Công ty Cổ phần Sữa Đồng Xanh tự hào là doanh nghiệp tiên phong khi lắp đặt hệ thống quan trắc nước thải tự động, truyền dữ liệu về Sở Tài nguyên và Môi trường.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-024", "claim": "Công ty Cổ phần Vận tải Cánh Cò tự hào dẫn đầu ngành khi lắp đặt hệ thống quan trắc nước thải tự động, truyền dữ liệu về Sở Tài nguyên và Môi trường.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-025", "claim": "Công ty Cổ phần Địa ốc An Cư tự hào dẫn đầu ngành khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-026", "claim": "Công ty Cổ phần Thép Phúc An khẳng định vị thế tiên phong khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-027", "claim": "Tổng Công ty Điện lực Sông Mây tự hào là doanh nghiệp tiên phong khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-028", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân tự hào dẫn đầu ngành khi không sử dụng môi chất làm lạnh CFC trong sản phẩm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-029", "claim": "Công ty Cổ phần Địa ốc An Cư khẳng định vị thế tiên phong khi thu gom và chuyển giao chất thải nguy hại cho đơn vị có giấy phép.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-030", "claim": "Ngân hàng TMCP Phù Sa tự hào là doanh nghiệp tiên phong khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-031", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam khẳng định vị thế tiên phong khi thu gom và chuyển giao chất thải nguy hại cho đơn vị có giấy phép.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-032", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân khẳng định vị thế tiên phong khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-033", "claim": "Tổng Công ty Dệt may Hồng Lĩnh khẳng định vị thế tiên phong khi thu gom và chuyển giao chất thải nguy hại cho đơn vị có giấy phép.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-034", "claim": "Công ty Cổ phần Địa ốc An Cư tự hào là doanh nghiệp tiên phong khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-035", "claim": "Tổng Công ty Dệt may Hồng Lĩnh tự hào là doanh nghiệp tiên phong khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-036", "claim": "Công ty Cổ phần Địa ốc An Cư tự hào là doanh nghiệp tiên phong khi không sử dụng môi chất làm lạnh CFC trong sản phẩm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-037", "claim": "Tổng Công ty Điện lực Sông Mây tự hào dẫn đầu ngành khi nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-038", "claim": "Công ty Cổ phần Sữa Đồng Xanh khẳng định vị thế tiên phong khi thu gom và chuyển giao chất thải nguy hại cho đơn vị có giấy phép.", "evidence": ""}\n{"id": "A-irrelevant_c-1111-039", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam khẳng định vị thế tiên phong khi thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới.", "evidence": ""}\n{"id": "A-intensity_vs-1111-000", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi thép thô của Công ty Cổ phần Thép Phúc An giảm 9% so với năm gốc 2018, qua đó cắt giảm tổng lượng phát thải của Thép Phúc An (Báo cáo kiểm kê khí nhà kính, bảng 5.3).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Thép Phúc An năm 2023 tăng 22% so với năm gốc 2018 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-001", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi tấn sản phẩm của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 5,5% so với năm gốc 2019, qua đó cắt giảm tổng lượng phát thải của Hóa chất Đức Giang Nam (Báo cáo phát triển bền vững, trang 81).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam năm 2025 tăng 22% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-002", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi tấn-km của Công ty Cổ phần Vận tải Cánh Cò giảm 4% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Vận tải Cánh Cò (Phụ lục số liệu môi trường, mục 3.3).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò năm 2024 tăng 15% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-003", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi MWh điện của Tổng Công ty Điện lực Sông Mây giảm 5,5% so với năm gốc 2019, qua đó cắt giảm tổng lượng phát thải của Điện lực Sông Mây (Báo cáo thường niên, trang 96).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây năm 2024 tăng 22% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-004", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi MWh điện của Tổng Công ty Điện lực Sông Mây giảm 9% so với năm gốc 2020, qua đó cắt giảm tổng lượng phát thải của Điện lực Sông Mây (Báo cáo kiểm kê khí nhà kính, bảng 7.2).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây năm 2025 tăng 22% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-005", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi clinker của Công ty Cổ phần Xi măng Đồng Lâm giảm 7% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Xi măng Đồng Lâm (Báo cáo phát triển bền vững, trang 108).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm năm 2023 tăng 15% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-006", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi nhân viên của Ngân hàng TMCP Phù Sa giảm 7% so với năm gốc 2019, qua đó cắt giảm tổng lượng phát thải của Ngân hàng Phù Sa (Phụ lục số liệu môi trường, mục 5.3).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa năm 2023 tăng 22% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-007", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi tấn sản phẩm của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 7% so với năm gốc 2020, qua đó cắt giảm tổng lượng phát thải của Hóa chất Đức Giang Nam (Báo cáo phát triển bền vững, trang 106).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam năm 2024 tăng 15% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-008", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi tấn-km của Công ty Cổ phần Vận tải Cánh Cò giảm 9% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Vận tải Cánh Cò (Báo cáo phát triển bền vững, trang 82).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò năm 2025 tăng 11% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-009", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi clinker của Công ty Cổ phần Xi măng Đồng Lâm giảm 5,5% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Xi măng Đồng Lâm (Phụ lục số liệu môi trường, mục 8.3).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm năm 2025 tăng 6% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-010", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi nhân viên của Ngân hàng TMCP Phù Sa giảm 9% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Ngân hàng Phù Sa (Báo cáo thường niên, trang 127).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa năm 2023 tăng 22% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-011", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi MWh điện của Tổng Công ty Điện lực Sông Mây giảm 9% so với năm gốc 2020, qua đó cắt giảm tổng lượng phát thải của Điện lực Sông Mây (Báo cáo kiểm kê khí nhà kính, bảng 5.5).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây năm 2024 tăng 15% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-012", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi nhân viên của Ngân hàng TMCP Phù Sa giảm 5,5% so với năm gốc 2020, qua đó cắt giảm tổng lượng phát thải của Ngân hàng Phù Sa (Báo cáo kiểm kê khí nhà kính, bảng 6.2).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa năm 2023 tăng 22% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-013", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi tấn sữa thành phẩm của Công ty Cổ phần Sữa Đồng Xanh giảm 7% so với năm gốc 2019, qua đó cắt giảm tổng lượng phát thải của Sữa Đồng Xanh (Báo cáo phát triển bền vững, trang 50).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh năm 2025 tăng 6% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-014", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi tấn-km của Công ty Cổ phần Vận tải Cánh Cò giảm 9% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Vận tải Cánh Cò (Báo cáo kiểm kê khí nhà kính, bảng 7.5).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò năm 2023 tăng 11% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-015", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi tấn sữa thành phẩm của Công ty Cổ phần Sữa Đồng Xanh giảm 4% so với năm gốc 2019, qua đó cắt giảm tổng lượng phát thải của Sữa Đồng Xanh (Báo cáo kiểm kê khí nhà kính, bảng 8.1).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh năm 2025 tăng 11% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-016", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi m2 sàn của Công ty Cổ phần Địa ốc An Cư giảm 9% so với năm gốc 2018, qua đó cắt giảm tổng lượng phát thải của Địa ốc An Cư (Báo cáo phát triển bền vững, trang 30).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Địa ốc An Cư năm 2024 tăng 15% so với năm gốc 2018 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-017", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi nhân viên của Ngân hàng TMCP Phù Sa giảm 4% so với năm gốc 2020, qua đó cắt giảm tổng lượng phát thải của Ngân hàng Phù Sa (Báo cáo phát triển bền vững, trang 23).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa năm 2023 tăng 22% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-018", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi tấn-km của Công ty Cổ phần Vận tải Cánh Cò giảm 5,5% so với năm gốc 2018, qua đó cắt giảm tổng lượng phát thải của Vận tải Cánh Cò (Báo cáo phát triển bền vững, trang 103).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò năm 2025 tăng 11% so với năm gốc 2018 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-019", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi MWh điện của Tổng Công ty Điện lực Sông Mây giảm 4% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Điện lực Sông Mây (Phụ lục số liệu môi trường, mục 9.2).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây năm 2024 tăng 22% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-020", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi tấn-km của Công ty Cổ phần Vận tải Cánh Cò giảm 4% so với năm gốc 2018, qua đó cắt giảm tổng lượng phát thải của Vận tải Cánh Cò (Báo cáo phát triển bền vững, trang 29).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò năm 2023 tăng 22% so với năm gốc 2018 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-021", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi nghìn sản phẩm của Tổng Công ty Dệt may Hồng Lĩnh giảm 4% so với năm gốc 2020, qua đó cắt giảm tổng lượng phát thải của Dệt may Hồng Lĩnh (Phụ lục số liệu môi trường, mục 7.6).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Dệt may Hồng Lĩnh năm 2024 tăng 15% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-022", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi tấn sữa thành phẩm của Công ty Cổ phần Sữa Đồng Xanh giảm 5,5% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Sữa Đồng Xanh (Báo cáo phát triển bền vững, trang 122).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh năm 2024 tăng 15% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-023", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi clinker của Công ty Cổ phần Xi măng Đồng Lâm giảm 7% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Xi măng Đồng Lâm (Phụ lục số liệu môi trường, mục 8.5).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm năm 2024 tăng 22% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-024", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi tấn sản phẩm của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 5,5% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Hóa chất Đức Giang Nam (Báo cáo kiểm kê khí nhà kính, bảng 2.1).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam năm 2025 tăng 6% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-025", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi tấn-km của Công ty Cổ phần Vận tải Cánh Cò giảm 9% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Vận tải Cánh Cò (Báo cáo phát triển bền vững, trang 134).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò năm 2024 tăng 22% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-026", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi MWh điện của Tổng Công ty Điện lực Sông Mây giảm 4% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Điện lực Sông Mây (Báo cáo kiểm kê khí nhà kính, bảng 5.2).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây năm 2025 tăng 15% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-027", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi nhân viên của Ngân hàng TMCP Phù Sa giảm 5,5% so với năm gốc 2020, qua đó cắt giảm tổng lượng phát thải của Ngân hàng Phù Sa (Báo cáo thường niên, trang 64).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa năm 2025 tăng 15% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-028", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi nhân viên của Ngân hàng TMCP Phù Sa giảm 7% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Ngân hàng Phù Sa (Báo cáo kiểm kê khí nhà kính, bảng 8.6).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa năm 2025 tăng 11% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-029", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi tấn-km của Công ty Cổ phần Vận tải Cánh Cò giảm 5,5% so với năm gốc 2019, qua đó cắt giảm tổng lượng phát thải của Vận tải Cánh Cò (Báo cáo kiểm kê khí nhà kính, bảng 3.3).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò năm 2024 tăng 11% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-030", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi MWh điện của Tổng Công ty Điện lực Sông Mây giảm 7% so với năm gốc 2018, qua đó cắt giảm tổng lượng phát thải của Điện lực Sông Mây (Báo cáo phát triển bền vững, trang 59).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây năm 2025 tăng 22% so với năm gốc 2018 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-031", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi thép thô của Công ty Cổ phần Thép Phúc An giảm 4% so với năm gốc 2020, qua đó cắt giảm tổng lượng phát thải của Thép Phúc An (Báo cáo kiểm kê khí nhà kính, bảng 8.5).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Thép Phúc An năm 2025 tăng 11% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-032", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi clinker của Công ty Cổ phần Xi măng Đồng Lâm giảm 7% so với năm gốc 2019, qua đó cắt giảm tổng lượng phát thải của Xi măng Đồng Lâm (Báo cáo phát triển bền vững, trang 76).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm năm 2023 tăng 6% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-033", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi nhân viên của Ngân hàng TMCP Phù Sa giảm 9% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Ngân hàng Phù Sa (Báo cáo kiểm kê khí nhà kính, bảng 6.6).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa năm 2025 tăng 11% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-034", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi nghìn sản phẩm của Tổng Công ty Dệt may Hồng Lĩnh giảm 4% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Dệt may Hồng Lĩnh (Báo cáo thường niên, trang 73).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Dệt may Hồng Lĩnh năm 2025 tăng 22% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-035", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi nhân viên của Ngân hàng TMCP Phù Sa giảm 9% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Ngân hàng Phù Sa (Báo cáo thường niên, trang 96).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa năm 2023 tăng 11% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-036", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi thép thô của Công ty Cổ phần Thép Phúc An giảm 7% so với năm gốc 2019, qua đó cắt giảm tổng lượng phát thải của Thép Phúc An (Báo cáo thường niên, trang 51).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Thép Phúc An năm 2025 tăng 22% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-037", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi tỷ đồng doanh thu của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 7% so với năm gốc 2021, qua đó cắt giảm tổng lượng phát thải của Bán lẻ Lạc Xuân (Báo cáo phát triển bền vững, trang 25).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Bán lẻ Lạc Xuân năm 2025 tăng 15% so với năm gốc 2021 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-038", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi nhân viên của Ngân hàng TMCP Phù Sa giảm 7% so với năm gốc 2019, qua đó cắt giảm tổng lượng phát thải của Ngân hàng Phù Sa (Báo cáo thường niên, trang 14).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa năm 2025 tăng 6% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-intensity_vs-1111-039", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi thép thô của Công ty Cổ phần Thép Phúc An giảm 5,5% so với năm gốc 2018, qua đó cắt giảm tổng lượng phát thải của Thép Phúc An (Báo cáo kiểm kê khí nhà kính, bảng 9.3).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Thép Phúc An năm 2025 tăng 22% so với năm gốc 2018 do sản lượng tăng."}\n{"id": "A-entity_scope-1111-000", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm là 297.473 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 9.6).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 297.473; Hợp nhất toàn tập đoàn: 922.166."}\n{"id": "A-entity_scope-1111-001", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh là 954.901 tCO2e (Báo cáo thường niên, trang 60).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 954.901; Hợp nhất toàn tập đoàn: 2.196.272."}\n{"id": "A-entity_scope-1111-002", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm là 167.427 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 8.4).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 167.427; Hợp nhất toàn tập đoàn: 519.023."}\n{"id": "A-entity_scope-1111-003", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò là 483.274 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 8.5).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 483.274; Hợp nhất toàn tập đoàn: 1.111.530."}\n{"id": "A-entity_scope-1111-004", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Bán lẻ Lạc Xuân là 254.154 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 3.3).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 254.154; Hợp nhất toàn tập đoàn: 787.877."}\n{"id": "A-entity_scope-1111-005", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa là 448.134 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 8.2).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 448.134; Hợp nhất toàn tập đoàn: 717.014."}\n{"id": "A-entity_scope-1111-006", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Thép Phúc An là 953.963 tCO2e (Báo cáo thường niên, trang 140).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 953.963; Hợp nhất toàn tập đoàn: 1.526.340."}\n{"id": "A-entity_scope-1111-007", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò là 529.715 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 3.3).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 529.715; Hợp nhất toàn tập đoàn: 1.218.344."}\n{"id": "A-entity_scope-1111-008", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa là 702.797 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 3.6).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 702.797; Hợp nhất toàn tập đoàn: 2.178.670."}\n{"id": "A-entity_scope-1111-009", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam là 560.567 tCO2e (Phụ lục số liệu môi trường, mục 3.3).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 560.567; Hợp nhất toàn tập đoàn: 1.289.304."}\n{"id": "A-entity_scope-1111-010", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây là 297.895 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 4.2).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 297.895; Hợp nhất toàn tập đoàn: 476.632."}\n{"id": "A-entity_scope-1111-011", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò là 261.667 tCO2e (Báo cáo thường niên, trang 118).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 261.667; Hợp nhất toàn tập đoàn: 811.167."}\n{"id": "A-entity_scope-1111-012", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Thép Phúc An là 466.237 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 2.2).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 466.237; Hợp nhất toàn tập đoàn: 1.072.345."}\n{"id": "A-entity_scope-1111-013", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa là 424.344 tCO2e (Phụ lục số liệu môi trường, mục 9.4).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 424.344; Hợp nhất toàn tập đoàn: 975.991."}\n{"id": "A-entity_scope-1111-014", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Địa ốc An Cư là 388.455 tCO2e (Báo cáo thường niên, trang 41).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 388.455; Hợp nhất toàn tập đoàn: 621.528."}\n{"id": "A-entity_scope-1111-015", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh là 534.984 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 3.2).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 534.984; Hợp nhất toàn tập đoàn: 1.230.463."}\n{"id": "A-entity_scope-1111-016", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm là 754.944 tCO2e (Báo cáo thường niên, trang 16).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 754.944; Hợp nhất toàn tập đoàn: 1.736.371."}\n{"id": "A-entity_scope-1111-017", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm là 342.774 tCO2e (Báo cáo phát triển bền vững, trang 78).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 342.774; Hợp nhất toàn tập đoàn: 788.380."}\n{"id": "A-entity_scope-1111-018", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Bán lẻ Lạc Xuân là 588.328 tCO2e (Phụ lục số liệu môi trường, mục 7.3).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 588.328; Hợp nhất toàn tập đoàn: 1.353.154."}\n{"id": "A-entity_scope-1111-019", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Địa ốc An Cư là 740.808 tCO2e (Báo cáo thường niên, trang 12).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 740.808; Hợp nhất toàn tập đoàn: 1.185.292."}\n{"id": "A-entity_scope-1111-020", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh là 427.634 tCO2e (Báo cáo thường niên, trang 53).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 427.634; Hợp nhất toàn tập đoàn: 684.214."}\n{"id": "A-entity_scope-1111-021", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Bán lẻ Lạc Xuân là 593.609 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 6.4).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 593.609; Hợp nhất toàn tập đoàn: 1.365.300."}\n{"id": "A-entity_scope-1111-022", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam là 781.500 tCO2e (Báo cáo phát triển bền vững, trang 56).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 781.500; Hợp nhất toàn tập đoàn: 1.797.449."}\n{"id": "A-entity_scope-1111-023", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Bán lẻ Lạc Xuân là 747.406 tCO2e (Báo cáo phát triển bền vững, trang 88).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 747.406; Hợp nhất toàn tập đoàn: 2.316.958."}\n{"id": "A-entity_scope-1111-024", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Địa ốc An Cư là 268.940 tCO2e (Báo cáo phát triển bền vững, trang 50).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 268.940; Hợp nhất toàn tập đoàn: 618.562."}\n{"id": "A-entity_scope-1111-025", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh là 311.950 tCO2e (Báo cáo thường niên, trang 88).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 311.950; Hợp nhất toàn tập đoàn: 499.120."}\n{"id": "A-entity_scope-1111-026", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Dệt may Hồng Lĩnh là 127.160 tCO2e (Báo cáo phát triển bền vững, trang 84).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 127.160; Hợp nhất toàn tập đoàn: 394.196."}\n{"id": "A-entity_scope-1111-027", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam là 710.768 tCO2e (Báo cáo phát triển bền vững, trang 90).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 710.768; Hợp nhất toàn tập đoàn: 1.634.766."}\n{"id": "A-entity_scope-1111-028", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam là 694.417 tCO2e (Báo cáo phát triển bền vững, trang 82).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 694.417; Hợp nhất toàn tập đoàn: 1.111.067."}\n{"id": "A-entity_scope-1111-029", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm là 443.595 tCO2e (Báo cáo thường niên, trang 59).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 443.595; Hợp nhất toàn tập đoàn: 1.375.144."}\n{"id": "A-entity_scope-1111-030", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây là 413.381 tCO2e (Phụ lục số liệu môi trường, mục 5.1).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 413.381; Hợp nhất toàn tập đoàn: 661.409."}\n{"id": "A-entity_scope-1111-031", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Dệt may Hồng Lĩnh là 665.565 tCO2e (Báo cáo phát triển bền vững, trang 14).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 665.565; Hợp nhất toàn tập đoàn: 2.063.251."}\n{"id": "A-entity_scope-1111-032", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò là 357.795 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 7.5).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 357.795; Hợp nhất toàn tập đoàn: 572.472."}\n{"id": "A-entity_scope-1111-033", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam là 513.353 tCO2e (Phụ lục số liệu môi trường, mục 8.2).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 513.353; Hợp nhất toàn tập đoàn: 1.180.711."}\n{"id": "A-entity_scope-1111-034", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Bán lẻ Lạc Xuân là 304.523 tCO2e (Báo cáo thường niên, trang 90).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 304.523; Hợp nhất toàn tập đoàn: 944.021."}\n{"id": "A-entity_scope-1111-035", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Bán lẻ Lạc Xuân là 396.369 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 9.4).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 396.369; Hợp nhất toàn tập đoàn: 1.228.743."}\n{"id": "A-entity_scope-1111-036", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm là 325.802 tCO2e (Phụ lục số liệu môi trường, mục 4.4).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 325.802; Hợp nhất toàn tập đoàn: 1.009.986."}\n{"id": "A-entity_scope-1111-037", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Ngân hàng Phù Sa là 439.555 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 3.3).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 439.555; Hợp nhất toàn tập đoàn: 1.362.620."}\n{"id": "A-entity_scope-1111-038", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh là 183.590 tCO2e (Phụ lục số liệu môi trường, mục 3.3).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 183.590; Hợp nhất toàn tập đoàn: 422.256."}\n{"id": "A-entity_scope-1111-039", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò là 872.273 tCO2e (Báo cáo thường niên, trang 42).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 872.273; Hợp nhất toàn tập đoàn: 2.006.227."}\n{"id": "A-clean-1111-000", "claim": "So với năm gốc 2021, Công ty Cổ phần Thép Phúc An đã giảm 18% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy luyện thép Nghi Sơn 2 trong năm 2023 (Báo cáo kiểm kê khí nhà kính, bảng 7.2; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-001", "claim": "So với năm gốc 2019, Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 8% lượng chất thải rắn đưa đi chôn lấp tại Trung tâm phân phối Long Hậu trong năm 2025 (Báo cáo thường niên, trang 87).", "evidence": ""}\n{"id": "A-clean-1111-002", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy thủy điện Đắk Rình của Tổng Công ty Điện lực Sông Mây giảm 11% so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 94).", "evidence": ""}\n{"id": "A-clean-1111-003", "claim": "Trong năm 2024, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng nước thải xả ra môi trường tại Nhà máy phân bón Lâm Thao Mới giảm 24% so với năm gốc 2021, theo Báo cáo kiểm kê khí nhà kính, bảng 6.5.", "evidence": ""}\n{"id": "A-clean-1111-004", "claim": "Năm 2025, điện năng tiêu thụ tại Trung tâm phân phối Long Hậu của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 9,5% so với năm gốc 2021 (Báo cáo thường niên, trang 60).", "evidence": ""}\n{"id": "A-clean-1111-005", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1, 2 và 3 của đội xe tải Bắc – Nam của Công ty Cổ phần Vận tải Cánh Cò giảm 21% so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 7.4; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-006", "claim": "Trong năm 2023, Công ty Cổ phần Thép Phúc An ghi nhận lượng nước thải xả ra môi trường tại Khu liên hợp gang thép Kỳ Long giảm 3% so với năm gốc 2018, theo Báo cáo phát triển bền vững, trang 36.", "evidence": ""}\n{"id": "A-clean-1111-007", "claim": "Trong năm 2024, Tổng Công ty Điện lực Sông Mây ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 24% so với năm gốc 2020, theo Phụ lục số liệu môi trường, mục 6.3.", "evidence": ""}\n{"id": "A-clean-1111-008", "claim": "Trong năm 2023, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Trung tâm phân phối Long Hậu giảm 11% so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 4.4.", "evidence": ""}\n{"id": "A-clean-1111-009", "claim": "Năm 2024, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy luyện thép Nghi Sơn 2 của Công ty Cổ phần Thép Phúc An giảm 9,5% so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 2.3; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-010", "claim": "So với năm gốc 2019, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 11% điện năng tiêu thụ tại Nhà máy xi măng Lương Sơn trong năm 2024 (Phụ lục số liệu môi trường, mục 9.1; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-011", "claim": "Trong năm 2024, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Trung tâm phân phối Long Hậu giảm 15% so với năm gốc 2018, theo Phụ lục số liệu môi trường, mục 9.3.", "evidence": ""}\n{"id": "A-clean-1111-012", "claim": "So với năm gốc 2020, Ngân hàng TMCP Phù Sa đã giảm 21% điện năng tiêu thụ tại Hội sở chính tại Hà Nội trong năm 2025 (Báo cáo phát triển bền vững, trang 57).", "evidence": ""}\n{"id": "A-clean-1111-013", "claim": "So với năm gốc 2021, Công ty Cổ phần Địa ốc An Cư đã giảm 4,5% điện năng tiêu thụ tại Khu đô thị Sông Hậu trong năm 2023 (Phụ lục số liệu môi trường, mục 4.3).", "evidence": ""}\n{"id": "A-clean-1111-014", "claim": "So với năm gốc 2020, Công ty Cổ phần Thép Phúc An đã giảm 21% phát thải khí nhà kính phạm vi 1 và 2 của Khu liên hợp gang thép Kỳ Long trong năm 2025 (Phụ lục số liệu môi trường, mục 6.1).", "evidence": ""}\n{"id": "A-clean-1111-015", "claim": "Trong năm 2024, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nước tiêu thụ tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 8% so với năm gốc 2021, theo Phụ lục số liệu môi trường, mục 4.3; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-clean-1111-016", "claim": "So với năm gốc 2020, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 24% điện năng tiêu thụ tại Nhà máy xi măng Lương Sơn trong năm 2024 (Phụ lục số liệu môi trường, mục 3.3; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-017", "claim": "So với năm gốc 2019, Công ty Cổ phần Sữa Đồng Xanh đã giảm 18% điện năng tiêu thụ tại Trang trại bò sữa Mộc Châu Xanh trong năm 2025 (Báo cáo phát triển bền vững, trang 126).", "evidence": ""}\n{"id": "A-clean-1111-018", "claim": "Năm 2025, lượng chất thải rắn đưa đi chôn lấp tại hệ thống 120 cửa hàng phía Nam của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 18% so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 30; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-019", "claim": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy sữa Bình Minh của Công ty Cổ phần Sữa Đồng Xanh giảm 6% so với năm gốc 2021 (Báo cáo thường niên, trang 71; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-020", "claim": "Trong năm 2025, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận lượng nhiên liệu hóa thạch sử dụng tại hệ thống 120 cửa hàng phía Nam giảm 8% so với năm gốc 2019, theo Báo cáo phát triển bền vững, trang 40; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-clean-1111-021", "claim": "Năm 2025, lượng chất thải rắn đưa đi chôn lấp tại hệ thống 120 cửa hàng phía Nam của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 7,2% so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 5.3; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-022", "claim": "Năm 2023, lượng nước tiêu thụ tại Nhà máy thủy điện Đắk Rình của Tổng Công ty Điện lực Sông Mây giảm 11% so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 2.5; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-023", "claim": "So với năm gốc 2019, Công ty Cổ phần Địa ốc An Cư đã giảm 24% lượng nhiên liệu hóa thạch sử dụng tại Tòa văn phòng An Cư Tower trong năm 2023 (Báo cáo phát triển bền vững, trang 73).", "evidence": ""}\n{"id": "A-clean-1111-024", "claim": "So với năm gốc 2020, Công ty Cổ phần Xi măng Đồng Lâm đã giảm 6% điện năng tiêu thụ tại Trạm nghiền Hiệp Phước trong năm 2024 (Báo cáo phát triển bền vững, trang 134).", "evidence": ""}\n{"id": "A-clean-1111-025", "claim": "Trong năm 2025, Ngân hàng TMCP Phù Sa ghi nhận lượng nước thải xả ra môi trường tại khối chi nhánh phía Nam giảm 12% so với năm gốc 2020, theo Báo cáo kiểm kê khí nhà kính, bảng 5.3.", "evidence": ""}\n{"id": "A-clean-1111-026", "claim": "Năm 2023, điện năng tiêu thụ tại cảng cạn Tân Cảng Bắc của Công ty Cổ phần Vận tải Cánh Cò giảm 4,5% so với năm gốc 2021 (Báo cáo thường niên, trang 91; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-027", "claim": "Trong năm 2025, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nước thải xả ra môi trường tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 9,5% so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 122.", "evidence": ""}\n{"id": "A-clean-1111-028", "claim": "Năm 2025, lượng nước thải xả ra môi trường tại Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh giảm 7,2% so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 27).", "evidence": ""}\n{"id": "A-clean-1111-029", "claim": "So với năm gốc 2021, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 18% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy xút clo Việt Trì Nam trong năm 2025 (Báo cáo kiểm kê khí nhà kính, bảng 4.3).", "evidence": ""}\n{"id": "A-clean-1111-030", "claim": "So với năm gốc 2019, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 18% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhuộm Phố Nối trong năm 2023 (Báo cáo kiểm kê khí nhà kính, bảng 6.4; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-031", "claim": "Năm 2024, phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 7,2% so với năm gốc 2018 (Báo cáo kiểm kê khí nhà kính, bảng 3.2).", "evidence": ""}\n{"id": "A-clean-1111-032", "claim": "So với năm gốc 2020, Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 8% điện năng tiêu thụ tại Trung tâm phân phối Long Hậu trong năm 2023 (Báo cáo phát triển bền vững, trang 45; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-033", "claim": "So với năm gốc 2020, Công ty Cổ phần Thép Phúc An đã giảm 18% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy luyện thép Nghi Sơn 2 trong năm 2024 (Báo cáo thường niên, trang 112).", "evidence": ""}\n{"id": "A-clean-1111-034", "claim": "So với năm gốc 2020, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 8% lượng nước thải xả ra môi trường tại Nhà máy xút clo Việt Trì Nam trong năm 2023 (Phụ lục số liệu môi trường, mục 9.4).", "evidence": ""}\n{"id": "A-clean-1111-035", "claim": "So với năm gốc 2020, Công ty Cổ phần Địa ốc An Cư đã giảm 12% lượng nước tiêu thụ tại Tòa văn phòng An Cư Tower trong năm 2024 (Báo cáo thường niên, trang 58).", "evidence": ""}\n{"id": "A-clean-1111-036", "claim": "So với năm gốc 2021, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 12% lượng nước tiêu thụ tại Nhà máy nhuộm Phố Nối trong năm 2025 (Báo cáo kiểm kê khí nhà kính, bảng 8.3; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-037", "claim": "So với năm gốc 2020, Công ty Cổ phần Vận tải Cánh Cò đã giảm 9,5% lượng chất thải rắn đưa đi chôn lấp tại đội xe tải Bắc – Nam trong năm 2023 (Phụ lục số liệu môi trường, mục 7.5; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-038", "claim": "Năm 2024, lượng nhiên liệu hóa thạch sử dụng tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 6% so với năm gốc 2021 (Báo cáo kiểm kê khí nhà kính, bảng 9.4; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-039", "claim": "So với năm gốc 2018, Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 15% lượng nước tiêu thụ tại hệ thống 120 cửa hàng phía Nam trong năm 2023 (Báo cáo thường niên, trang 38; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-040", "claim": "Trong năm 2024, Công ty Cổ phần Vận tải Cánh Cò ghi nhận điện năng tiêu thụ tại cảng cạn Tân Cảng Bắc giảm 6% so với năm gốc 2021, theo Báo cáo phát triển bền vững, trang 110; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-clean-1111-041", "claim": "So với năm gốc 2021, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 4,5% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy xút clo Việt Trì Nam trong năm 2025 (Phụ lục số liệu môi trường, mục 7.3).", "evidence": ""}\n{"id": "A-clean-1111-042", "claim": "Trong năm 2023, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nước tiêu thụ tại Nhà máy thủy điện Đắk Rình giảm 9,5% so với năm gốc 2021, theo Báo cáo kiểm kê khí nhà kính, bảng 3.2; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-clean-1111-043", "claim": "Năm 2025, lượng nước tiêu thụ tại Trung tâm phân phối Long Hậu của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 7,2% so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 9.6).", "evidence": ""}\n{"id": "A-clean-1111-044", "claim": "Trong năm 2023, Công ty Cổ phần Vận tải Cánh Cò ghi nhận điện năng tiêu thụ tại cảng cạn Tân Cảng Bắc giảm 11% so với năm gốc 2020, theo Phụ lục số liệu môi trường, mục 8.3.", "evidence": ""}\n{"id": "A-clean-1111-045", "claim": "So với năm gốc 2020, Ngân hàng TMCP Phù Sa đã giảm 15% điện năng tiêu thụ tại Hội sở chính tại Hà Nội trong năm 2024 (Phụ lục số liệu môi trường, mục 2.2; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-046", "claim": "Năm 2025, điện năng tiêu thụ tại Trạm nghiền Hiệp Phước của Công ty Cổ phần Xi măng Đồng Lâm giảm 4,5% so với năm gốc 2021 (Báo cáo thường niên, trang 13).", "evidence": ""}\n{"id": "A-clean-1111-047", "claim": "Năm 2023, lượng nước thải xả ra môi trường tại khối chi nhánh phía Nam của Ngân hàng TMCP Phù Sa giảm 18% so với năm gốc 2021 (Phụ lục số liệu môi trường, mục 6.3; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-048", "claim": "So với năm gốc 2020, Ngân hàng TMCP Phù Sa đã giảm 6% lượng nước thải xả ra môi trường tại Hội sở chính tại Hà Nội trong năm 2025 (Báo cáo thường niên, trang 28).", "evidence": ""}\n{"id": "A-clean-1111-049", "claim": "So với năm gốc 2018, Công ty Cổ phần Thép Phúc An đã giảm 11% phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy luyện thép Nghi Sơn 2 trong năm 2025 (Phụ lục số liệu môi trường, mục 5.5; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-050", "claim": "Trong năm 2024, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận lượng chất thải rắn đưa đi chôn lấp tại hệ thống 120 cửa hàng phía Nam giảm 15% so với năm gốc 2019, theo Báo cáo phát triển bền vững, trang 45; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-clean-1111-051", "claim": "So với năm gốc 2020, Tổng Công ty Điện lực Sông Mây đã giảm 21% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 trong năm 2024 (Phụ lục số liệu môi trường, mục 5.6).", "evidence": ""}\n{"id": "A-clean-1111-052", "claim": "So với năm gốc 2019, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 7,2% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy nhuộm Phố Nối trong năm 2023 (Báo cáo phát triển bền vững, trang 136).", "evidence": ""}\n{"id": "A-clean-1111-053", "claim": "Trong năm 2024, Công ty Cổ phần Thép Phúc An ghi nhận lượng nước thải xả ra môi trường tại Khu liên hợp gang thép Kỳ Long giảm 11% so với năm gốc 2020, theo Báo cáo phát triển bền vững, trang 17; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-clean-1111-054", "claim": "Năm 2023, lượng chất thải rắn đưa đi chôn lấp tại Khu đô thị Sông Hậu của Công ty Cổ phần Địa ốc An Cư giảm 3% so với năm gốc 2021 (Báo cáo phát triển bền vững, trang 48).", "evidence": ""}\n{"id": "A-clean-1111-055", "claim": "So với năm gốc 2019, Tổng Công ty Điện lực Sông Mây đã giảm 8% điện năng tiêu thụ tại Nhà máy nhiệt điện Vĩnh Hảo 3 trong năm 2023 (Báo cáo kiểm kê khí nhà kính, bảng 7.5; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-056", "claim": "So với năm gốc 2018, Công ty Cổ phần Sữa Đồng Xanh đã giảm 3% phát thải khí nhà kính phạm vi 1 và 2 của Nhà máy sữa Bình Minh trong năm 2024 (Báo cáo thường niên, trang 21).", "evidence": ""}\n{"id": "A-clean-1111-057", "claim": "Năm 2025, lượng nhiên liệu hóa thạch sử dụng tại hệ thống 120 cửa hàng phía Nam của Công ty Cổ phần Bán lẻ Lạc Xuân giảm 6% so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 8.3).", "evidence": ""}\n{"id": "A-clean-1111-058", "claim": "Năm 2023, lượng nước thải xả ra môi trường tại Nhà máy luyện thép Nghi Sơn 2 của Công ty Cổ phần Thép Phúc An giảm 3% so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 9.2; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-clean-1111-059", "claim": "So với năm gốc 2020, Ngân hàng TMCP Phù Sa đã giảm 21% lượng chất thải rắn đưa đi chôn lấp tại Hội sở chính tại Hà Nội trong năm 2024 (Báo cáo phát triển bền vững, trang 70).", "evidence": ""}\n{"id": "A-clean-future_a-1111-000", "claim": "Tổng Công ty Điện lực Sông Mây đặt mục tiêu giảm 25% lượng nước thải xả ra môi trường tại Nhà máy thủy điện Đắk Rình vào năm 2035 so với năm gốc 2021 (Báo cáo phát triển bền vững, trang 113).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 25% lượng nước thải xả ra môi trường vào năm 2035 so với năm gốc 2021. Kết quả năm 2023: giảm 5,0% so với năm gốc."}\n{"id": "A-clean-future_a-1111-001", "claim": "Tổng Công ty Dệt may Hồng Lĩnh đặt mục tiêu giảm 40% điện năng tiêu thụ tại Nhà máy may Hòa Khánh vào năm 2030 so với năm gốc 2021 (Phụ lục số liệu môi trường, mục 8.6).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 40% điện năng tiêu thụ vào năm 2030 so với năm gốc 2021. Kết quả năm 2023: giảm 8,0% so với năm gốc."}\n{"id": "A-clean-future_a-1111-002", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam đặt mục tiêu giảm 30% phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy xút clo Việt Trì Nam vào năm 2030 so với năm gốc 2021 (Báo cáo phát triển bền vững, trang 40).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% phát thải khí nhà kính vào năm 2030 so với năm gốc 2021. Kết quả năm 2023: giảm 12,0% so với năm gốc."}\n{"id": "A-clean-future_a-1111-003", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đặt mục tiêu giảm 20% lượng nước tiêu thụ tại hệ thống 120 cửa hàng phía Nam vào năm 2030 so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 86).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng nước tiêu thụ vào năm 2030 so với năm gốc 2020. Kết quả năm 2023: giảm 6,0% so với năm gốc."}\n{"id": "A-clean-future_a-1111-004", "claim": "Công ty Cổ phần Thép Phúc An đặt mục tiêu giảm 20% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy luyện thép Nghi Sơn 2 vào năm 2035 so với năm gốc 2019 (Báo cáo kiểm kê khí nhà kính, bảng 5.1).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng chất thải rắn đưa đi chôn lấp vào năm 2035 so với năm gốc 2019. Kết quả năm 2025: giảm 6,0% so với năm gốc."}\n{"id": "A-clean-future_a-1111-005", "claim": "Công ty Cổ phần Xi măng Đồng Lâm đặt mục tiêu giảm 25% lượng nước tiêu thụ tại Nhà máy xi măng Lương Sơn vào năm 2030 so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 83).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 25% lượng nước tiêu thụ vào năm 2030 so với năm gốc 2019. Kết quả năm 2024: giảm 5,0% so với năm gốc."}\n{"id": "A-clean-future_a-1111-006", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam đặt mục tiêu giảm 20% lượng nước tiêu thụ tại Nhà máy phân bón Lâm Thao Mới vào năm 2030 so với năm gốc 2021 (Báo cáo thường niên, trang 53).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng nước tiêu thụ vào năm 2030 so với năm gốc 2021. Kết quả năm 2023: giảm 6,0% so với năm gốc."}\n{"id": "A-clean-future_a-1111-007", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam đặt mục tiêu giảm 30% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy phân bón Lâm Thao Mới vào năm 2030 so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 67).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng chất thải rắn đưa đi chôn lấp vào năm 2030 so với năm gốc 2020. Kết quả năm 2025: giảm 12,0% so với năm gốc."}\n{"id": "A-clean-future_a-1111-008", "claim": "Tổng Công ty Điện lực Sông Mây đặt mục tiêu giảm 20% lượng nước thải xả ra môi trường tại Nhà máy nhiệt điện Vĩnh Hảo 3 vào năm 2030 so với năm gốc 2019 (Báo cáo thường niên, trang 14).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 20% lượng nước thải xả ra môi trường vào năm 2030 so với năm gốc 2019. Kết quả năm 2025: giảm 6,0% so với năm gốc."}\n{"id": "A-clean-future_a-1111-009", "claim": "Công ty Cổ phần Sữa Đồng Xanh đặt mục tiêu giảm 30% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy sữa Bình Minh vào năm 2035 so với năm gốc 2018 (Báo cáo thường niên, trang 121).", "evidence": "Lộ trình giảm phát thải: mục tiêu giảm 30% lượng chất thải rắn đưa đi chôn lấp vào năm 2035 so với năm gốc 2018. Kết quả năm 2025: giảm 9,0% so với năm gốc."}\n{"id": "A-clean-unverifi-1111-000", "claim": "Năm 2023, Tòa văn phòng An Cư Tower của Công ty Cổ phần Địa ốc An Cư được Hội đồng Công trình Xanh Đông Dương cấp chứng nhận ISO 50001:2018 cho hệ thống quản lý năng lượng, hiệu lực đến năm 2026.", "evidence": ""}\n{"id": "A-clean-unverifi-1111-001", "claim": "Năm 2024, Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An được Hội đồng Công trình Xanh Đông Dương cấp chứng nhận nhãn Sinh thái Việt Nam cho dòng sản phẩm chủ lực, hiệu lực đến năm 2027.", "evidence": ""}\n{"id": "A-clean-unverifi-1111-002", "claim": "Năm 2023, đội xe tải Bắc – Nam của Công ty Cổ phần Vận tải Cánh Cò được Hội đồng Công trình Xanh Đông Dương cấp chứng nhận ISO 14001:2015 cho hệ thống quản lý môi trường, hiệu lực đến năm 2026.", "evidence": ""}\n{"id": "A-clean-unverifi-1111-003", "claim": "Năm 2024, Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh được Hội đồng Công trình Xanh Đông Dương cấp chứng nhận EDGE Advanced cho công trình, hiệu lực đến năm 2027.", "evidence": ""}\n{"id": "A-clean-unverifi-1111-004", "claim": "Năm 2023, khối chi nhánh phía Nam của Ngân hàng TMCP Phù Sa được Hội đồng Công trình Xanh Đông Dương cấp chứng nhận ISO 50001:2018 cho hệ thống quản lý năng lượng, hiệu lực đến năm 2026.", "evidence": ""}\n{"id": "A-clean-unverifi-1111-005", "claim": "Năm 2023, Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây được Trung tâm Chứng nhận Phù Hợp Bắc Hải cấp chứng nhận EDGE Advanced cho công trình, hiệu lực đến năm 2026.", "evidence": ""}\n{"id": "A-clean-unverifi-1111-006", "claim": "Năm 2024, Nhà máy thủy điện Đắk Rình của Tổng Công ty Điện lực Sông Mây được Trung tâm Chứng nhận Phù Hợp Bắc Hải cấp chứng nhận EDGE Advanced cho công trình, hiệu lực đến năm 2027.", "evidence": ""}\n{"id": "A-clean-unverifi-1111-007", "claim": "Năm 2025, Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh được Hội đồng Công trình Xanh Đông Dương cấp chứng nhận ISO 14001:2015 cho hệ thống quản lý môi trường, hiệu lực đến năm 2028.", "evidence": ""}\n{"id": "A-clean-unverifi-1111-008", "claim": "Năm 2024, Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm được Tổ chức Chứng nhận Quốc tế Sao Mai cấp chứng nhận ISO 14001:2015 cho hệ thống quản lý môi trường, hiệu lực đến năm 2027.", "evidence": ""}\n{"id": "A-clean-unverifi-1111-009", "claim": "Năm 2025, Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm được Trung tâm Chứng nhận Phù Hợp Bắc Hải cấp chứng nhận LEED Gold cho công trình, hiệu lực đến năm 2028.", "evidence": ""}\n{"id": "A-clean-hidden_t-1111-000", "claim": "Năm 2023, Công ty Cổ phần Vận tải Cánh Cò lắp đặt hệ thống điện mặt trời áp mái công suất 1500 kWp tại văn phòng (Báo cáo thường niên, trang 123).", "evidence": "Năm 2023, điện năng tiêu thụ từ lưới điện than của Vận tải Cánh Cò giảm 12% so với năm 2022."}\n{"id": "A-clean-hidden_t-1111-001", "claim": "Năm 2023, Công ty Cổ phần Xi măng Đồng Lâm lắp đặt hệ thống điện mặt trời áp mái công suất 350 kWp tại văn phòng (Phụ lục số liệu môi trường, mục 9.2).", "evidence": "Năm 2023, điện năng tiêu thụ từ lưới điện than của Xi măng Đồng Lâm giảm 25% so với năm 2022."}\n{"id": "A-clean-hidden_t-1111-002", "claim": "Năm 2024, Công ty Cổ phần Thép Phúc An trồng 1500 cây xanh tại khu vực nhà máy (Báo cáo phát triển bền vững, trang 58).", "evidence": "Năm 2024, diện tích rừng tự nhiên bị chuyển đổi cho dự án mở rộng của Thép Phúc An giảm 34% so với năm 2023."}\n{"id": "A-clean-hidden_t-1111-003", "claim": "Năm 2024, Công ty Cổ phần Thép Phúc An chuyển 30% xe nâng trong kho sang chạy điện (Báo cáo kiểm kê khí nhà kính, bảng 9.3).", "evidence": "Năm 2024, lượng dầu diesel tiêu thụ của đội xe vận tải của Thép Phúc An giảm 12% so với năm 2023."}\n{"id": "A-clean-hidden_t-1111-004", "claim": "Năm 2025, Công ty Cổ phần Địa ốc An Cư lắp đặt hệ thống điện mặt trời áp mái công suất 350 kWp tại văn phòng (Phụ lục số liệu môi trường, mục 8.6).", "evidence": "Năm 2025, điện năng tiêu thụ từ lưới điện than của Địa ốc An Cư giảm 34% so với năm 2024."}\n{"id": "A-clean-hidden_t-1111-005", "claim": "Năm 2023, Tổng Công ty Điện lực Sông Mây chuyển 40% xe nâng trong kho sang chạy điện (Báo cáo kiểm kê khí nhà kính, bảng 7.6).", "evidence": "Năm 2023, lượng dầu diesel tiêu thụ của đội xe vận tải của Điện lực Sông Mây giảm 34% so với năm 2022."}\n{"id": "A-clean-hidden_t-1111-006", "claim": "Năm 2025, Công ty Cổ phần Địa ốc An Cư giảm 15% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Báo cáo thường niên, trang 57).", "evidence": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Địa ốc An Cư giảm 18% so với năm 2024."}\n{"id": "A-clean-hidden_t-1111-007", "claim": "Năm 2023, Công ty Cổ phần Vận tải Cánh Cò trồng 800 cây xanh tại khu vực nhà máy (Phụ lục số liệu môi trường, mục 8.4).", "evidence": "Năm 2023, diện tích rừng tự nhiên bị chuyển đổi cho dự án mở rộng của Vận tải Cánh Cò giảm 25% so với năm 2022."}\n{"id": "A-clean-hidden_t-1111-008", "claim": "Năm 2023, Công ty Cổ phần Hóa chất Đức Giang Nam trồng 120 cây xanh tại khu vực nhà máy (Báo cáo kiểm kê khí nhà kính, bảng 4.1).", "evidence": "Năm 2023, diện tích rừng tự nhiên bị chuyển đổi cho dự án mở rộng của Hóa chất Đức Giang Nam giảm 25% so với năm 2022."}\n{"id": "A-clean-hidden_t-1111-009", "claim": "Năm 2025, Công ty Cổ phần Xi măng Đồng Lâm giảm 15% lượng túi ni-lông phát cho khách tại hệ thống cửa hàng (Phụ lục số liệu môi trường, mục 8.3).", "evidence": "Năm 2025, phát thải khí nhà kính phạm vi 1 và 2 của Xi măng Đồng Lâm giảm 34% so với năm 2024."}\n{"id": "A-clean-irreleva-1111-000", "claim": "Công ty Cổ phần Sữa Đồng Xanh thu gom và chuyển giao chất thải nguy hại cho đơn vị có giấy phép theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-irreleva-1111-001", "claim": "Tổng Công ty Dệt may Hồng Lĩnh nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-irreleva-1111-002", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-irreleva-1111-003", "claim": "Công ty Cổ phần Sữa Đồng Xanh nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-irreleva-1111-004", "claim": "Công ty Cổ phần Địa ốc An Cư thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-irreleva-1111-005", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-irreleva-1111-006", "claim": "Công ty Cổ phần Vận tải Cánh Cò không sử dụng môi chất làm lạnh CFC trong sản phẩm theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-irreleva-1111-007", "claim": "Công ty Cổ phần Vận tải Cánh Cò nộp báo cáo công tác bảo vệ môi trường định kỳ hằng năm theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-irreleva-1111-008", "claim": "Ngân hàng TMCP Phù Sa thực hiện đánh giá tác động môi trường trước khi xây dựng dự án mới theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-irreleva-1111-009", "claim": "Công ty Cổ phần Địa ốc An Cư lắp đặt hệ thống quan trắc nước thải tự động, truyền dữ liệu về Sở Tài nguyên và Môi trường theo quy định của Luật Bảo vệ môi trường 2020.", "evidence": ""}\n{"id": "A-clean-intensit-1111-000", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi nghìn sản phẩm của Tổng Công ty Dệt may Hồng Lĩnh giảm 4% so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 9.4).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Dệt may Hồng Lĩnh năm 2024 tăng 11% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-clean-intensit-1111-001", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi tấn sản phẩm của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 4% so với năm gốc 2018 (Báo cáo thường niên, trang 116).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam năm 2023 tăng 15% so với năm gốc 2018 do sản lượng tăng."}\n{"id": "A-clean-intensit-1111-002", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi tấn sản phẩm của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 9% so với năm gốc 2019 (Báo cáo phát triển bền vững, trang 40).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam năm 2025 tăng 11% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-clean-intensit-1111-003", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi MWh điện của Tổng Công ty Điện lực Sông Mây giảm 5,5% so với năm gốc 2018 (Báo cáo phát triển bền vững, trang 101).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây năm 2025 tăng 6% so với năm gốc 2018 do sản lượng tăng."}\n{"id": "A-clean-intensit-1111-004", "claim": "Năm 2025, cường độ phát thải khí nhà kính trên mỗi MWh điện của Tổng Công ty Điện lực Sông Mây giảm 4% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 27).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Điện lực Sông Mây năm 2025 tăng 15% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-clean-intensit-1111-005", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi tấn-km của Công ty Cổ phần Vận tải Cánh Cò giảm 5,5% so với năm gốc 2019 (Báo cáo thường niên, trang 31).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Vận tải Cánh Cò năm 2024 tăng 6% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-clean-intensit-1111-006", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi tấn sữa thành phẩm của Công ty Cổ phần Sữa Đồng Xanh giảm 9% so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 5.2).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh năm 2024 tăng 6% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-clean-intensit-1111-007", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi tấn sữa thành phẩm của Công ty Cổ phần Sữa Đồng Xanh giảm 5,5% so với năm gốc 2020 (Báo cáo thường niên, trang 51).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Sữa Đồng Xanh năm 2023 tăng 15% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-clean-intensit-1111-008", "claim": "Năm 2023, cường độ phát thải khí nhà kính trên mỗi nghìn sản phẩm của Tổng Công ty Dệt may Hồng Lĩnh giảm 9% so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 3.3).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Dệt may Hồng Lĩnh năm 2023 tăng 22% so với năm gốc 2019 do sản lượng tăng."}\n{"id": "A-clean-intensit-1111-009", "claim": "Năm 2024, cường độ phát thải khí nhà kính trên mỗi tấn sản phẩm của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 5,5% so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 2.3).", "evidence": "Tổng phát thải khí nhà kính phạm vi 1 và 2 của Hóa chất Đức Giang Nam năm 2024 tăng 6% so với năm gốc 2020 do sản lượng tăng."}\n{"id": "A-clean-entity_s-1111-000", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Hóa chất Đức Giang Nam là 152.305 tCO2e (Báo cáo phát triển bền vững, trang 96).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 152.305; Hợp nhất toàn tập đoàn: 243.688."}\n{"id": "A-clean-entity_s-1111-001", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Địa ốc An Cư là 952.597 tCO2e (Báo cáo thường niên, trang 80).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 952.597; Hợp nhất toàn tập đoàn: 2.953.050."}\n{"id": "A-clean-entity_s-1111-002", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Vận tải Cánh Cò là 296.758 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 3.4).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 296.758; Hợp nhất toàn tập đoàn: 919.949."}\n{"id": "A-clean-entity_s-1111-003", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Địa ốc An Cư là 358.713 tCO2e (Phụ lục số liệu môi trường, mục 5.4).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 358.713; Hợp nhất toàn tập đoàn: 573.940."}\n{"id": "A-clean-entity_s-1111-004", "claim": "Năm 2023, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Điện lực Sông Mây là 295.603 tCO2e (Báo cáo thường niên, trang 63).", "evidence": "Bảng phát thải năm 2023 (tCO2e): Công ty mẹ: 295.603; Hợp nhất toàn tập đoàn: 472.964."}\n{"id": "A-clean-entity_s-1111-005", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Dệt may Hồng Lĩnh là 235.816 tCO2e (Báo cáo phát triển bền vững, trang 33).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 235.816; Hợp nhất toàn tập đoàn: 731.029."}\n{"id": "A-clean-entity_s-1111-006", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Xi măng Đồng Lâm là 425.123 tCO2e (Báo cáo phát triển bền vững, trang 94).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 425.123; Hợp nhất toàn tập đoàn: 1.317.881."}\n{"id": "A-clean-entity_s-1111-007", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Địa ốc An Cư là 150.642 tCO2e (Phụ lục số liệu môi trường, mục 8.3).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 150.642; Hợp nhất toàn tập đoàn: 346.476."}\n{"id": "A-clean-entity_s-1111-008", "claim": "Năm 2025, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Điện lực Sông Mây là 185.929 tCO2e (Phụ lục số liệu môi trường, mục 8.1).", "evidence": "Bảng phát thải năm 2025 (tCO2e): Công ty mẹ: 185.929; Hợp nhất toàn tập đoàn: 427.636."}\n{"id": "A-clean-entity_s-1111-009", "claim": "Năm 2024, tổng phát thải khí nhà kính phạm vi 1 và 2 của Công ty mẹ Dệt may Hồng Lĩnh là 593.113 tCO2e (Báo cáo kiểm kê khí nhà kính, bảng 8.5).", "evidence": "Bảng phát thải năm 2024 (tCO2e): Công ty mẹ: 593.113; Hợp nhất toàn tập đoàn: 1.364.159."}\n{"id": "A-multi-1111-000", "claim": "Lượng nhiên liệu hóa thạch sử dụng tại Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây giảm 21% (Báo cáo kiểm kê khí nhà kính, bảng 9.3; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-multi-1111-001", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 11% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy xút clo Việt Trì Nam (Báo cáo phát triển bền vững, trang 35; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-multi-1111-002", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 12% phát thải khí nhà kính phạm vi 1 và 2 của hệ thống 120 cửa hàng phía Nam (Báo cáo phát triển bền vững, trang 12; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-multi-1111-003", "claim": "Ngân hàng TMCP Phù Sa ghi nhận lượng nước thải xả ra môi trường tại khối chi nhánh phía Nam giảm 9,5%, theo Báo cáo phát triển bền vững, trang 53; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-multi-1111-004", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy xút clo Việt Trì Nam giảm 9,5%, theo Báo cáo kiểm kê khí nhà kính, bảng 6.3; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập.", "evidence": ""}\n{"id": "A-multi-1111-005", "claim": "Công ty Cổ phần Thép Phúc An ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Khu liên hợp gang thép Kỳ Long giảm 8%, theo Báo cáo phát triển bền vững, trang 107; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-multi-1111-006", "claim": "Tổng Công ty Điện lực Sông Mây ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 3%, theo Báo cáo thường niên, trang 99; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập.", "evidence": ""}\n{"id": "A-multi-1111-007", "claim": "Công ty Cổ phần Thép Phúc An ghi nhận phát thải khí nhà kính phạm vi 1, 2 và 3 của Nhà máy luyện thép Nghi Sơn 2 giảm 8%, theo Báo cáo thường niên, trang 92; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n{"id": "A-multi-1111-008", "claim": "Công ty Cổ phần Thép Phúc An đã giảm 9,5% lượng nước thải xả ra môi trường tại Khu liên hợp gang thép Kỳ Long (Báo cáo phát triển bền vững, trang 133).", "evidence": ""}\n{"id": "A-multi-1111-009", "claim": "Phát thải khí nhà kính phạm vi 1 và 2 của Trạm nghiền Hiệp Phước của Công ty Cổ phần Xi măng Đồng Lâm giảm 24% (Phụ lục số liệu môi trường, mục 4.4).", "evidence": ""}\n{"id": "A-multi-1111-100", "claim": "So với năm gốc 2018, Tổng Công ty Điện lực Sông Mây đã giảm 7,2% phát thải khí nhà kính trong năm 2023.", "evidence": ""}\n{"id": "A-multi-1111-101", "claim": "Trong năm 2024, Công ty Cổ phần Vận tải Cánh Cò ghi nhận lượng nước thải xả ra môi trường giảm 4,5% so với năm gốc 2018.", "evidence": ""}\n{"id": "A-multi-1111-102", "claim": "So với năm gốc 2021, Công ty Cổ phần Thép Phúc An đã giảm 15% lượng nhiên liệu hóa thạch sử dụng trong năm 2025.", "evidence": ""}\n{"id": "A-multi-1111-103", "claim": "Trong năm 2023, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nước tiêu thụ giảm 8% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-multi-1111-104", "claim": "Trong năm 2025, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận điện năng tiêu thụ giảm 4,5% so với năm gốc 2021.", "evidence": ""}\n{"id": "A-multi-1111-105", "claim": "So với năm gốc 2019, Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 24% lượng chất thải rắn đưa đi chôn lấp trong năm 2024.", "evidence": ""}\n{"id": "A-multi-1111-106", "claim": "Trong năm 2025, Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng nước tiêu thụ giảm 11% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-multi-1111-107", "claim": "Trong năm 2025, Công ty Cổ phần Xi măng Đồng Lâm ghi nhận điện năng tiêu thụ giảm 4,5% so với năm gốc 2019.", "evidence": ""}\n{"id": "A-multi-1111-108", "claim": "So với năm gốc 2021, Công ty Cổ phần Địa ốc An Cư đã giảm 3% lượng nhiên liệu hóa thạch sử dụng trong năm 2023.", "evidence": ""}\n{"id": "A-multi-1111-109", "claim": "Trong năm 2023, Công ty Cổ phần Thép Phúc An ghi nhận phát thải khí nhà kính giảm 12% so với năm gốc 2018.", "evidence": ""}\n{"id": "A-multi-1111-200", "claim": "Năm 2023, lượng nước tiêu thụ tại Trang trại bò sữa Mộc Châu Xanh của Công ty Cổ phần Sữa Đồng Xanh giảm 11% (Phụ lục số liệu môi trường, mục 4.4; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập), hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-multi-1111-201", "claim": "Năm 2024, lượng nước thải xả ra môi trường tại Hội sở chính tại Hà Nội của Ngân hàng TMCP Phù Sa giảm 18% (Báo cáo thường niên, trang 14), hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-multi-1111-202", "claim": "Trong năm 2023, Tổng Công ty Điện lực Sông Mây ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 11%, theo Phụ lục số liệu môi trường, mục 5.4; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập, hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-multi-1111-203", "claim": "Trong năm 2024, Công ty Cổ phần Sữa Đồng Xanh ghi nhận lượng nước tiêu thụ tại Trang trại bò sữa Mộc Châu Xanh giảm 11%, theo Phụ lục số liệu môi trường, mục 5.1; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập, hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-multi-1111-204", "claim": "Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 18% phát thải khí nhà kính phạm vi 1 và 2 của hệ thống 120 cửa hàng phía Nam trong năm 2024 (Báo cáo thường niên, trang 48), thể hiện nỗ lực không ngừng vì một tương lai xanh.", "evidence": ""}\n{"id": "A-multi-1111-205", "claim": "Trong năm 2023, Tổng Công ty Điện lực Sông Mây ghi nhận điện năng tiêu thụ tại Nhà máy nhiệt điện Vĩnh Hảo 3 giảm 9,5%, theo Báo cáo thường niên, trang 63, góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-multi-1111-206", "claim": "Năm 2025, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy nhuộm Phố Nối của Tổng Công ty Dệt may Hồng Lĩnh giảm 7,2% (Báo cáo thường niên, trang 126), hướng tới mô hình sản xuất thân thiện với môi trường.", "evidence": ""}\n{"id": "A-multi-1111-207", "claim": "Năm 2023, lượng nước tiêu thụ tại Nhà máy may Hòa Khánh của Tổng Công ty Dệt may Hồng Lĩnh giảm 21% (Phụ lục số liệu môi trường, mục 9.1; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-multi-1111-208", "claim": "Trong năm 2025, Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng nước tiêu thụ tại Nhà máy xút clo Việt Trì Nam giảm 24%, theo Phụ lục số liệu môi trường, mục 2.2, khẳng định cam kết phát triển xanh, sạch và bền vững.", "evidence": ""}\n{"id": "A-multi-1111-209", "claim": "Năm 2025, lượng nhiên liệu hóa thạch sử dụng tại Khu liên hợp gang thép Kỳ Long của Công ty Cổ phần Thép Phúc An giảm 7,2% (Phụ lục số liệu môi trường, mục 8.6), góp phần xây dựng hình ảnh doanh nghiệp xanh và thân thiện với môi trường.", "evidence": ""}\n{"id": "A-multi-1111-300", "claim": "Lượng chất thải rắn đưa đi chôn lấp tại Hội sở chính tại Hà Nội của Ngân hàng TMCP Phù Sa giảm 3% so với năm gốc 2020 (Báo cáo thường niên, trang 90; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-multi-1111-301", "claim": "Lượng nước tiêu thụ tại Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 11% so với năm gốc 2019 (Phụ lục số liệu môi trường, mục 7.4; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-multi-1111-302", "claim": "Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng nước tiêu thụ tại Nhà máy nhuộm Phố Nối giảm 6% so với năm gốc 2018, theo Báo cáo kiểm kê khí nhà kính, bảng 6.3; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập, đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-multi-1111-303", "claim": "So với năm gốc 2020, Tổng Công ty Dệt may Hồng Lĩnh đã giảm 18% lượng nước thải xả ra môi trường tại Nhà máy may Hòa Khánh (Báo cáo kiểm kê khí nhà kính, bảng 2.6), giúp sản phẩm của Công ty sạch 100 %.", "evidence": ""}\n{"id": "A-multi-1111-304", "claim": "So với năm gốc 2021, Tổng Công ty Điện lực Sông Mây đã giảm 4,5% lượng nước tiêu thụ tại Nhà máy thủy điện Đắk Rình (Báo cáo phát triển bền vững, trang 83; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-multi-1111-305", "claim": "Lượng nhiên liệu hóa thạch sử dụng tại Nhà máy xút clo Việt Trì Nam của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 8% so với năm gốc 2020 (Báo cáo thường niên, trang 94; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), qua đó loại bỏ hoàn toàn rủi ro ô nhiễm.", "evidence": ""}\n{"id": "A-multi-1111-306", "claim": "Lượng nước thải xả ra môi trường tại Nhà máy xi măng Lương Sơn của Công ty Cổ phần Xi măng Đồng Lâm giảm 7,2% so với năm gốc 2018 (Phụ lục số liệu môi trường, mục 7.3; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập), đạt mức hiệu quả môi trường tốt nhất toàn ngành.", "evidence": ""}\n{"id": "A-multi-1111-307", "claim": "Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhuộm Phố Nối giảm 4,5% so với năm gốc 2021, theo Phụ lục số liệu môi trường, mục 2.4, và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-multi-1111-308", "claim": "Lượng nhiên liệu hóa thạch sử dụng tại Trang trại bò sữa Mộc Châu Xanh của Công ty Cổ phần Sữa Đồng Xanh giảm 6% so với năm gốc 2021 (Phụ lục số liệu môi trường, mục 9.2), và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-multi-1111-309", "claim": "So với năm gốc 2020, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 6% lượng chất thải rắn đưa đi chôn lấp tại Nhà máy phân bón Lâm Thao Mới (Phụ lục số liệu môi trường, mục 6.3; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập), và hoàn toàn không gây tác động tiêu cực nào đến môi trường.", "evidence": ""}\n{"id": "A-multi-1111-400", "claim": "So với năm gốc 2018, Công ty Cổ phần Địa ốc An Cư đã giảm 8% lượng nước thải xả ra môi trường tại Khu đô thị Sông Hậu trong năm 2024, đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-401", "claim": "So với năm gốc 2021, Công ty Cổ phần Địa ốc An Cư đã giảm 21% lượng chất thải rắn đưa đi chôn lấp tại Tòa văn phòng An Cư Tower trong năm 2023, đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-402", "claim": "So với năm gốc 2021, Công ty Cổ phần Bán lẻ Lạc Xuân đã giảm 24% lượng nhiên liệu hóa thạch sử dụng tại hệ thống 120 cửa hàng phía Nam trong năm 2023, đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-403", "claim": "So với năm gốc 2020, Công ty Cổ phần Địa ốc An Cư đã giảm 11% điện năng tiêu thụ tại Khu đô thị Sông Hậu trong năm 2024, đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-404", "claim": "Trong năm 2024, Tổng Công ty Dệt may Hồng Lĩnh ghi nhận lượng nhiên liệu hóa thạch sử dụng tại Nhà máy nhuộm Phố Nối giảm 12% so với năm gốc 2020, đồng thời đạt trung hòa carbon trong năm 2024 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-405", "claim": "Năm 2023, lượng chất thải rắn đưa đi chôn lấp tại Nhà máy nhiệt điện Vĩnh Hảo 3 của Tổng Công ty Điện lực Sông Mây giảm 4,5% so với năm gốc 2018, đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-406", "claim": "Trong năm 2023, Công ty Cổ phần Bán lẻ Lạc Xuân ghi nhận lượng nước thải xả ra môi trường tại Trung tâm phân phối Long Hậu giảm 24% so với năm gốc 2019, đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-407", "claim": "So với năm gốc 2021, Công ty Cổ phần Sữa Đồng Xanh đã giảm 8% lượng chất thải rắn đưa đi chôn lấp tại Trang trại bò sữa Mộc Châu Xanh trong năm 2025, đồng thời đạt trung hòa carbon trong năm 2025 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-408", "claim": "So với năm gốc 2019, Tổng Công ty Điện lực Sông Mây đã giảm 4,5% lượng nước thải xả ra môi trường tại Nhà máy thủy điện Đắk Rình trong năm 2023, đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-409", "claim": "Năm 2023, lượng nhiên liệu hóa thạch sử dụng tại Nhà máy phân bón Lâm Thao Mới của Công ty Cổ phần Hóa chất Đức Giang Nam giảm 9,5% so với năm gốc 2018, đồng thời đạt trung hòa carbon trong năm 2023 nhờ mua tín chỉ carbon.", "evidence": ""}\n{"id": "A-multi-1111-500", "claim": "Điện năng tiêu thụ của Công ty Cổ phần Xi măng Đồng Lâm giảm 6% so với năm gốc 2019 (Báo cáo thường niên, trang 106; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập).", "evidence": ""}\n{"id": "A-multi-1111-501", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận điện năng tiêu thụ giảm 4,5% so với năm gốc 2018, theo Báo cáo thường niên, trang 89.", "evidence": ""}\n{"id": "A-multi-1111-502", "claim": "Lượng nhiên liệu hóa thạch sử dụng của Công ty Cổ phần Sữa Đồng Xanh giảm 11% so với năm gốc 2021 (Phụ lục số liệu môi trường, mục 9.1; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-multi-1111-503", "claim": "Điện năng tiêu thụ của Ngân hàng TMCP Phù Sa giảm 21% so với năm gốc 2020 (Báo cáo kiểm kê khí nhà kính, bảng 6.5).", "evidence": ""}\n{"id": "A-multi-1111-504", "claim": "Công ty Cổ phần Sữa Đồng Xanh ghi nhận lượng nước tiêu thụ giảm 4,5% so với năm gốc 2018, theo Báo cáo phát triển bền vững, trang 43.", "evidence": ""}\n{"id": "A-multi-1111-505", "claim": "So với năm gốc 2019, Công ty Cổ phần Hóa chất Đức Giang Nam đã giảm 7,2% điện năng tiêu thụ (Báo cáo phát triển bền vững, trang 65; số liệu đã được Công ty TNHH Đảm bảo Bền vững Hạ Long bảo đảm độc lập).", "evidence": ""}\n{"id": "A-multi-1111-506", "claim": "Ngân hàng TMCP Phù Sa ghi nhận điện năng tiêu thụ giảm 24% so với năm gốc 2020, theo Phụ lục số liệu môi trường, mục 7.2.", "evidence": ""}\n{"id": "A-multi-1111-507", "claim": "Phát thải khí nhà kính của Công ty Cổ phần Vận tải Cánh Cò giảm 6% so với năm gốc 2020 (Báo cáo phát triển bền vững, trang 52).", "evidence": ""}\n{"id": "A-multi-1111-508", "claim": "Lượng nhiên liệu hóa thạch sử dụng của Công ty Cổ phần Địa ốc An Cư giảm 9,5% so với năm gốc 2021 (Báo cáo phát triển bền vững, trang 102; số liệu đã được Viện Chứng nhận Chất lượng Trường Sơn bảo đảm độc lập).", "evidence": ""}\n{"id": "A-multi-1111-509", "claim": "Công ty Cổ phần Hóa chất Đức Giang Nam ghi nhận lượng nước tiêu thụ giảm 11% so với năm gốc 2019, theo Báo cáo thường niên, trang 91; số liệu đã được Công ty TNHH Kiểm định Sao Việt bảo đảm độc lập.", "evidence": ""}\n',
    'gsi_d_edit.jsonl': '{"id": "A-b25d14627d32a467", "claim": "Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn.", "evidence": "Quản lý môi trường Xét theo lĩnh vực hoạt động, ngành sản xuất thép là lĩnh vực phát thải lớn nhất, đóng góp chủ yếu vào tổng phát thải của toàn tập đoàn. Nhóm nông nghiệp, bất động sản và điện máy gia dụng có mức phát thải thấp hơn đáng kể. Nhóm nguồn phát thải trực tiếp từ quá trình sản xuất chiếm tỷ trọng rõ rệt, phản ánh đặc thù ngành nghề và quy mô sản xuất của tập đoàn. Tập đoàn Hòa Phát xác định giảm thiểu biến đổi khí hậu là một trong những ưu tiên chiến lược và đã xây dựng hệ thống quản lý khí nhà kính (KNK) toàn diện, tập trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang thép, chiếm hơn 99% tổng lượng phát thải KNK của Tập đoàn. Trong đó, hai Khu liên h"}\n{"id": "B-35e48f695c325894", "claim": "advanced technologies and optimizing production processes, member companies have achieved signi昀椀cant results in energy savings and operational ef昀椀ciency gains.", "evidence": ""}\n{"id": "B-0252ab47096fb1d5", "claim": "(2) trong đó 291.295.250 m3 nước biển dùng cho hệ thống FGD (hệ thống khử lưu huỳnh khí thải) do Nhà máy nhiệt điện Vũng Áng 1 sử dụng nước biển để khử lưu huỳnh thay cho hệ thống dùng đá vôi Tất cả khối lượng chất thải rắn thông thường và nguy hại phát sinh đều được các nhà máy thuê đơn vị đủ năng lực xử lý, nước thải trước khi xả vào môi trường đều được xử lý đảm bảo tiêu chuẩn.", "evidence": ""}\n{"id": "A-582b4ff27fd7e469", "claim": "Thông qua các hoạt động tri ân, chăm sóc sức khỏe cộng đồng, đồng hành cùng người cao tuổi, trẻ em và thế hệ trẻ, Traphaco không ngừng lan tỏa các giá trị nhân văn, góp phần nâng cao chất lượng sống và kiến tạo một xã hội khỏe mạnh, bền vững cho hôm nay và các thế hệ mai sau.", "evidence": "Công ty Cổ phần Traphaco xác định trách nhiệm xã hội là một trụ cột quan trọng song hành cùng hoạt động sản xuất – kinh doanh. Trong năm 2025, Traphaco tiếp tục triển khai nhiều chương trình cộng đồng có quy mô, chiều sâu và sức lan tỏa rộng khắp, tập trung vào chăm sóc sức khỏe cộng đồng, tri ân các thế hệ đi trước, đồng hành cùng người cao tuổi, trẻ em có hoàn cảnh khó khăn và thế hệ trẻ. Thông qua chương trình, Traphaco tiếp tục lan tỏa tinh thần sẻ chia vì trẻ em vùng cao, đồng hành cùng cộng đồng trong các hoạt động chăm lo cho những nhóm đối tượng yếu thế trong xã hội. Năm 2025 đánh dấu năm thứ 19 liên tiếp Traphaco đồng hành cùng chương trình “Mùa xuân cho em” do Quỹ Bảo trợ Trẻ em Vi"}\n{"id": "B-3be87c8859aaf12f", "claim": "Triển khai chương trình 5S nhằm nâng cao năng suất lao động, giảm tác động tới môi trường Gắn kết cán bộ trong các hoạt động xanh hoá hoạt đông kinh doanh Ý thức bảo vệ môi trường cần xuất phát từ việc xây dựng môi trường sạch sẽ, không gian thông thoáng tại nơi làm việc.", "evidence": ""}\n{"id": "A-5f40e1a8b472095a", "claim": "At the same time, Bibica has implemented numerous packaging innovations to reduce weight, optimize packing, and use environmentally friendly packaging.", "evidence": "has implemented several measures to optimize packaging size and weight, such as producing/packing larger- format packaging for use in areas with larger coverage, thereby reducing the number of packages. Notably, the company avoids using glass and metal packaging, which helps lower disposal costs and ensures safety for farmers during use and disposal. Group ecosystem have proactively implemented solutions to meet EPR requirements. In addition, Bibica has replaced PS material liners (environmentally unfriendly) with recycled PET material, with an approximate packaging weight of 14 tons."}\n{"id": "A-b997dfb647c53cf7", "claim": "Hệ thống EHS được triển khai phù hợp với mô hình hoạt động tại các công ty thành viên của Tập đoàn, thể hiện cam kết phát triển có trách nhiệm và bền vững dựa trên nền tảng minh bạch và cải tiến liên tục dựa trên các nguyên tắc cốt lõi", "evidence": "Thành tựu này thể hiện cam kết mạnh mẽ của doanh nghiệp trong việc quản lý năng lượng một cách hệ thống, cải tiến liên tục và giảm phát thải carbon. Dựa trên kết quả này, Tập đoàn định hướng mở rộng việc áp dụng tiêu chuẩn ISO 50001 các đơn vị thành viên sản xuất đạo đức kinh doanh có liên quan. Tính liêm chính, minh bạch và công bằng là những nguyên tắc cốt lõi định hướng toàn bộ hoạt động sản xuất kinh doanh của Tập đoàn. BÁO CÁO PHÁT TRIỂN BỀN VỮNG"}\n{"id": "A-18cb2c52524594a8", "claim": "Mục tiêu hàng đầu của Coteccons: Thực hành tốt nguyên tắc 3R (Reduce – Reuse – Recycle) do Công ty thiết lập, giảm thiểu những tác động không cần thiết đến môi trường, sử dụng hiệu quả nguồn nguyên vật liệu và có các giải pháp khoa học kỹ thuật để tối ưu hóa các nguồn lực hiện có.", "evidence": "Reuse - Tái sử dụng; Recycle - Tái chế) trong chính sách giảm thiểu tác động đến môi trường của Công ty. Ecopark cũng là một công trường điển hình về thực hành nâng cao tiêu chuẩn ESG của Coteccons. Tại đây, các tiêu chuẩn về môi trường, an toàn và quyền lợi cho người lao động đều được thiết lập ở mức cao nhất. Trong nỗ lực thúc đẩy các giải pháp về kinh tế tuần hoàn, Coteccons nghiêm khắc yêu cầu áp dụng nguyên tắc 3R trong mọi phương diện xuyên suốt phát triển dự án, tối ưu hiệu quả sử dụng nguyên vật liệu trong một vòng đời khép kín để phát huy giá trị tối đa của chúng. Coteccons đang nỗ lực triển khai Mô hình hóa Dữ liệu Công trình (BIM) cho tất cả các dự án của mình."}\n{"id": "B-9a2a234da633e4ee", "claim": "Ngân hàng Tiêu biểu về Tín dụng Xanh Do Tập đoàn Dữ liệu Quốc tế IDG trao tặng.", "evidence": ""}\n{"id": "A-60c641654a20072f", "claim": "We have established a comprehensive waste management system, covering the collection, segregation, treatment, and disposal of municipal solid waste, general industrial solid waste, and hazardous waste.", "evidence": "Equipment, systems, and facilities for storing hazardous waste, general industrial solid waste, and municipal solid waste are fully compliant with the requirements specified in Circular No. 02/2022/TT-BTNMT dated January 10, 2022. Ï Collaboration with Competent Waste Management Units Companies and factories within the PAN Group have established close partnerships with reputable waste collection and treatment service providers. We conduct thorough evaluations and screenings to select units with the necessary capacity and experience, responsible for the safe and efficient collection, transportation, and treatment of municipal waste, industrial waste, and hazardous waste. The operations of thes"}\n{"id": "A-9792676130b30bd1", "claim": "We believe that sustainability development is integral to our long-term success, and we are proud to take this important step in furthering our journey towards a more sustainable future.", "evidence": "\\u0017 Continuous Improvement \\u0017 Ensure the development towards international integration. \\u0017 \\u0017 Ready – Providing a professional and healthy work environment OUR SUSTAINABILITY STRATEGY We believe that long-term success is built on collaboration, strong governance, inclusive stakeholder engagement, and a strategic focus on sustainability. Our sustainability targets and KPIs based on the 3 pillars Environment, Social and Governance are embedded into our business development strategy, promoting competitives advantages to achieve long-term, inclusive growth and sutainability. Keep the balance of economy, society and environment, including stakeholders who are working towards the implementation of sust"}\n{"id": "B-27152df365d3507f", "claim": "c) Quản lý nguồn nguyên vật liệu Công ty cam kết hoạt động một cách bền vững, bảo vệ môi trường và cộng đồng.", "evidence": ""}\n{"id": "A-19fe9fe913803550", "claim": "đã trở thành một hoạt động thường niên bền vững, liên tục mở rộng phạm vi, tích cực tiếp cận và mang lại cơ hội sống khỏe mạnh cho trẻ em mắc bệnh tim bẩm sinh trên khắp cả nước.", "evidence": "NHỊP ĐẬP YÊU THƯƠNG – TÀI TRỢ PHẪU THUẬT CHO TRẺ EM NGHÈO MẮC BỆNH TIM BẨM SINH Trách nhiệm xã hội - cộng đồng Xuất phát từ mong muốn mang lại cho cho trẻ em Việt Nam một trái tim khỏe Tài trợ phẫu thuật tim cho hơn 50 trẻ em nghèo mắc bệnh tim bẩm sinh; được các bệnh viện lớn và cơ quan xã hội 2025 ghi nhận. Chặng đường nối yêu thương"}\n{"id": "A-ab5fbc72c8e6bc06", "claim": "- Top 10 trong 100 Doanh nghiệp phát triển bền vững", "evidence": "2021- Giải thưởng ngành kim hoàn thế giới - JWA (Jewellery World Awards) - Top 10 trong 100 Doanh nghiệp phát triển bền vững - Top 10 Doanh nghiệp đạt chuẩn văn hóa kinh doanh NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN CƠ CẤU QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG PNJ liên tục 6 năm liền lọt top 10 trong 100 doanh nghiệp phát triển bền vững, bao quát ở 3 lĩnh vực chính: quản trị, môi trường và xã hội."}\n{"id": "B-385acdc04d6b2584", "claim": "Throughout its 33 years of establishment and development, PNJ has consistently updated and improved its production processes, and has demonstrated foresight in creating a sustainable business model that integrates supply chain optimization and promotes environmental protection awareness.", "evidence": ""}\n{"id": "A-d439dfa8bbe8aaae", "claim": "KgCO2e/hectoliter of beer (Scope 1 & Scope 2) compared to 2023.", "evidence": "In 2024, we have invested in and implemented over 50 solutions and continuous improvement initiatives to save energy, reduce greenhouse gas (GHG) emissions and achieve a green transition in energy use. 0.92 KgCO2e Per hectoliter of beer (Scope 1 and Scope 2) in 2024 compared to 2023 Renewable Energy Projects: We installed rooftop solar power systems at several factories within the We started working on internal carbon footprint accounting since 2019 and have consistently improved the scope and quality of data. In 2024, our total scope 1 and scope 2 is 70,888 tonnes CO2 equivalent (compared to 78,161 tonnes CO2"}\n{"id": "A-2e7f492135c148bc", "claim": "Thường xuyên thu gom, phân loại, lưu giữ chất thải trong các thùng chứa riêng biệt, có nhận biết mã CTNH", "evidence": "Duy trì việc cập nhật, đánh giá sự ảnh hưởng của các văn bản pháp luật đến hệ thống của công ty. Phân loại tại nguồn, phổ biển và thực hiện quy định phân loại chất thải tại mỗi bộ phận, phòng ban. Đặt thùng thu gom phân loại tại nơi phát thải, thuận tiện cho nhận biết, lưu trữ. Các khoản mục tiền tệ được phân loại là tài sản được đánh giá lại theo tỷ giá mua của ngân hàng thương mại nơi Công ty và các công ty con thường xuyên có giao dịch; và Các khoản mục tiền tệ được phân loại là nợ phải trả được đánh giá lại theo tỷ giá bán của ngân hàng thương mại nơi Công ty và các công ty con thường xuyên có giao dịch."}\n{"id": "B-694675deeed34b03", "claim": "Bảo vệ và phục hồi môi trường, bảo vệ đất và nguồn nước thông qua các hoạt động quản lý và kinh doanh rừng cao su;", "evidence": ""}\n{"id": "A-43556e82a696abff", "claim": "đào tạo sử dụng các phần mềm quản lý, điều hành và ứng dụng nội bộ (HRM Pro, Base, TXNG Traphacoxanh...);", "evidence": "Phát triển phần mềm Lalalab phần mềm quản lý hóa chất – thuốc thử, ứng dụng quản lý dữ liệu thực nghiệm Labo giúp chuyên biệt phục vụ bộ phận quản lý chất lượng và Labo được tối ưu Hoàn tất triển khai ứng dụng quản lý nhân sự – tiền lương, đưa vào vận hành đồng bộ trên toàn Công ty. Nâng cấp Zalo OA, miniapp và triển khai Chatbot Ai trợ lý bán hàng thông minh giúp nâng cao hiệu quả chăm sóc khách hàng Bắc TV, Nature herb, DLS Ứng dụng hiệu quả phần mềm Base trong quản lí điều hành công việc Xuất bán 1 lô hàng cho Đối tác xuất khẩu vào thị trường Mỹ Làm việc với UBND tỉnh, sở xây dựng và các sở ban ngành về vị trí đất mới tại Bát Xát, Lào Cai (dự án đường sắt cao tốc HP – HN - LC)"}\n{"id": "B-f594a7fe53373516", "claim": "10 Nhà máy sử dụng hệ thống hơi - năng lượng xanh • Tạo việc làm thu nhập cao cho 9.243 lao động.", "evidence": ""}\n{"id": "A-d8bd4734fd20f5b2", "claim": "Chất thải được chuyển hướng sử dụng", "evidence": "Các vụ việc phân biệt đối xử và các hành động khắc phục đã thực hiện GRI 406-1 78,79 Chất thải được chuyển hướng sử dụng GRI 306-4 Chất thải bị loại bỏ GRI 306-5 Chất thải nguy hại được phân loại, thu gom và lưu trữ vào kho chứa chất thải nguy hại đáp ứng đầy đủ yêu cầu của luật. PNJP đã hợp đồng với các đơn vị có chức năng để thu gom, vận chuyển và xử lý các chất thải nguy hại trên theo đúng quy định của luật. Ngoài ra, PNJP đã tiến hành đăng ký Sổ Chủ nguồn thải chất thải nguy hại với Sở Tài nguyên và Môi trường TP."}\n{"id": "A-cd4837ab3092c420", "claim": "(ESG) topics that are most significant to the business, its stakeholders, and the broader societal and environmental context.", "evidence": "topics relevant to the sector and business operations. We mapped out relevant internal and external stakeholders, considering stakeholders that play an important role in the operations of SABECO as well as those who are most impacted by the business’s operations, both positively and negatively. Meanwhile, the importance to stakeholders assesses the degree of stakeholder concern about the topic, as well as the degree to which the company creates positive or negative impacts on the broader environment and society as a result of its operations. By prioritizing the topics accordingly, this ensures that SABECO’s sustainability efforts are focused on areas of greatest importance and influence. Cli"}\n{"id": "A-f69452d9f70748f5", "claim": "Toàn Tiểu ban ESG hiện nay có 10 nhân sự, đại diện cho tất cả phương diện vận hành chính của Công ty.", "evidence": "Trong đó, trách nhiệm điển hình là đưa ESG đến mọi lĩnh vực kinh doanh của công ty, làm cho ESG trở nên hữu hình và có ý nghĩa đối với tất cả nhân viên và các bên liên quan. Các vị trí quản lý cấp cao đã đảm nhiệm 2024 - Hiện tại: Phó Tổng giám đốc, Giám đốc vận hành 2023 - Hiện tại: Thành viên Tiểu ban ESG MỘT DOANH NGHIỆP CHỈ THỰC SỰ VĨ ĐẠI KHI PHÁT TRIỂN VÌ CON NGƯỜI Các hoạt động xã hội của Coteccons mà công chúng thấy được trên truyền thông là phần đại diện cho chiến lược phát triển bền vững và yêu cầu thực hành ESG có chiều sâu mà Coteccons đã triển khai. ESG là một trong những phương diện mà Coteccons tiên phong lan tỏa nhận thức và nâng cao tiêu chuẩn thực hành với tư cách industry l"}\n{"id": "B-5133fce7da1ea93c", "claim": "Đây là hoạt động hợp tác với Bộ Tài nguyên và Môi trường kéo dài trong 5 năm (2023-2027) với ngân sách 15 tỷ Trong thời gian tới, Vinamilk định hướng tiếp tục đẩy mạnh để có thêm nhiều “Cánh rừng Net Zero Vinamilk” tại Việt Nam, đóng góp quan trọng trong việc thực hiện lộ trình tiến đến Net Zero của Vinamilk vào năm 2050.", "evidence": ""}\n{"id": "A-089adfcb41fe031a", "claim": "Dòng máy lọc nước HyperS được phát triển theo hướng tối ưu năng lượng và thân thiện môi trường, nổi bật với công nghệ làm lạnh Block sử dụng gas R600a không gây hại tầng ozone, hệ điện cực titanium phủ Platinum", "evidence": "làm lạnh bằng BLOCK với Gas R600a, giúp tiết kiệm điện năng và giảm thiểu tác động tới tầng ozone. Song song đó, ngành hàng tủ đông đánh dấu bước tiến mới với dòng sản phẩm ứng dụng công nghệ thép ZAM. Công nghệ này giúp tủ đông chống rỉ sét và ăn Cả hai dòng sản phẩm đều có hai lựa chọn màu sắc phổ biến là ghi đậm, phù hợp với đa dạng không gian nội thất. Công nghệ xanh và quy trình sản xuất tối ưu hóa được áp dụng trên nhiều dòng sản phẩm của Điện máy Gia dụng Hòa Phát. Máy lọc nước Hòa Phát sử dụng công nghệ"}\n{"id": "A-954f3a44ebf12d16", "claim": "2 Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO.", "evidence": "CHUNG TAY BẢO VỆ MÔI TRƯỜNG Sử dụng năng lượng hiệu quả thông qua việc sử dụng hợp lý, có ý thức các nguồn năng lượng và tài nguyên thiên nhiên. MỤC TIÊU 6 (SD13) Hành động Ứng phó với biến đổi khí hậu Phủ xanh đất trống đồi núi trọc bằng các vùng trồng dược liệu đạt chuẩn GACP - WHO. 1.000+ Vùng trồng/thu hái dược liệu đạt chuẩn GACP-WHO 13 Khách hàng nhà thuốc trên toàn quốc 30.000+ Nhà máy chiết xuất dược liệu đạt chuẩn GMP-WHO"}\n{"id": "B-153bfb97d777ea1f", "claim": "Spring crop 2024–2025, rice cultivated by 5,473 farmers within Vinaseed’s value chain achieved lower emissions compared to conventional practices.", "evidence": ""}\n{"id": "B-0fed5041b73d6ebe", "claim": "Mỗi CBNV là mắt xích quan trọng, trực tiếp thực hiện các nhiệm vụ cụ thể hàng ngày nhằm hiện thực hóa các mục tiêu phát triển bền vững của Công ty.", "evidence": ""}\n{"id": "A-3dc5c4a83fde8fab", "claim": "S 2 DG 1 DG 6 S Change in score since 2015 (p.p): S 11 DG G 7 SD Missing data: S 0 DG 1 DG 8 S 9 SD G Source: The Sustainable Development Goals Report 2025 - United Nations", "evidence": "S 7 DG G1 Country rank 1 SD S 100 16 DG G SD 2 75 SD 5 1 G DG 3 50 S 25 Country score 4 S 1 DG DG 4 S VIETNAM 3 SD 1 G DG 5 S Regional average 69.5 TABLE | 2 1 G D S : ore since 2015 (p.p): : | :"}\n{"id": "B-6b260f1d20f04021", "claim": "GRI 305] SABECO is committed to raising the proportion of renewable energy usage by increasing investment in innovative low-carbon technologies that contribute to climate change mitigation and adaptation efforts, as well as promoting optimization of overall resource use for maximum efficiency.", "evidence": ""}\n{"id": "B-ee4b9253a749cd8a", "claim": "Trong năm 2025, DHG Pharma phát sinh tổng cộng 159.104 m3 nước thải, với chi phí xử lý khoảng 2,1 tỷ Toàn bộ nước thải từ các hoạt động sản xuất và vận hành được thu gom và dẫn về hệ thống xử lý nước thải tập trung của Công ty trước khi xả thải ra môi trường.", "evidence": ""}\n{"id": "A-e4177240054ac7cd", "claim": "Nhiệt độ luôn nằm ở mức ổn định từ 18 - 25oC nhờ hệ thống điều khiển tự động, đàn vật nuôi ít bị mệt, giảm stress nhiệt giúp tiêu hóa hiệu quả và giảm phát thải", "evidence": "trưởng, từng mùa vụ để đảm bảo nâng cao sức khỏe vật nuôi, giảm tác động của biến đổi khí hậu gây stress tới con vật, tăng tỷ lệ tiêu hóa, giảm tồn dư đạm thải qua phân. Đầu tư, vận hành thực chất, hiệu quả hệ thống xử lý chất thải (rắn, lỏng) động, nâng cao độ chính xác và ổn định của mẫu thép. Ngoài ra, hệ thống cắt phôi theo khối lượng tự động được triển khai tại công đoạn cán thép đã giúp giảm thiểu tiêu hao ngắn dài thép thành phẩm, hạn chế lãng phí vật tư và tăng"}\n{"id": "B-0af90394a1cfbefa", "claim": "recycling is also a goal that the PAN Group focuses on as this is one of the top responsibilities of any manufacturer, closely attached to the United Nations’ 12 Sustainable Development Goals: Responsible consumption and production.", "evidence": ""}\n{"id": "B-079acf7a91109441", "claim": "Tác động tích cực đến môi trường: • Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "evidence": ""}\n{"id": "B-79d11206b3494e8d", "claim": "Sau khi giải được tổ chức thành công, với quãng đường mà các vận động viên tích lũy được, Coteccons đã thực hiện trồng 11.111 cây xanh tại khu vực rừng đặc dụng ở vùng đệm bảo vệ quần thể voọc gáy trắng tại xã Đồng Hóa, huyện Tuyên Hóa, tỉnh Quảng Bình.", "evidence": ""}\n{"id": "A-ad90fa76ab778405", "claim": "Chúng tôi cam kết nỗ lực đồng hành cùng Chính phủ Việt Nam hướng tới mục tiêu Net Zero 2050.", "evidence": "XANH HÓA SẢN XUẤT QUẢN TRỊ VÀ ĐỔI MỚI Trụ cột Xanh hóa Sản xuất hướng tới hài hòa giữa tăng trưởng công nghiệp và bảo vệ môi trường, giảm thiểu biến đổi khí hậu. Chúng tôi nhận thức về trách nhiệm của mình trong việc tham gia lộ trình giảm phát thải quốc gia và đang tích cực thực hành các giải pháp phù hợp với điều kiện kinh doanh, đặc biệt trong lĩnh vực thép, để đồng hành cùng Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050. BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạ"}\n{"id": "B-d8f31db2e705ce80", "claim": "Bùn thải từ cụm xử lý hóa lý hàng ngày được thu gom bằng dàn lọc bùn sau đó được đưa vào kho chứa chất thải nguy hại, định kỳ hàng tháng sẽ bàn giao cho đơn vị chức năng thu gom và xử lý chất thải nguy hại theo đúng quy định.", "evidence": ""}\n{"id": "B-90a62f8691c17017", "claim": "The Lafooco’s wastewater treatment system was upgraded with the plant, increasing the treatment capacity up to 30m3/day and night with better wastewater quality than that required by Loi Binh Nhon Aquatex Ben Tre renovated its wastewater treatment system to handle the specifically high pollutant items of the aquaculture wastewater sources with reasonable price.", "evidence": ""}\n{"id": "A-8c03e86741c139dc", "claim": "thích ứng với biến đổi khí hậu Những cải tiến này đã góp phần nâng cao đáng kể hiệu suất năng lượng, giảm phụ thuộc vào nguồn điện bên ngoài và đóng góp tích cực vào mục tiêu khử carbon dài hạn của Tập đoàn.", "evidence": "Những nỗ lực này nhằm giảm sự phụ thuộc vào nguồn nước ngọt, đặc biệt là nguồn nước khai thác từ cộng đồng, đồng thời góp phần thực hiện các mục tiêu bảo vệ môi trường rộng hơn của Tập đoàn. Việc đầu tư nâng cấp hạ tầng và triển khai các dự án chuyên biệt nhằm nâng cao 103-5 Giảm tiêu thụ năng lượng Hành động giảm thiểu và thích ứng với biến đổi khí hậu 102-4 Mục tiêu và tiến độ giảm phát thải khí nhà kính Đây là báo cáo năm đầu tiên của chúng tôi, lộ trình giảm phát thải chi tiết vẫn đang được"}\n{"id": "A-eac9831108b61efe", "claim": "Phương tiện truyền thông C.7 Trang thông tin điện tử của công ty C.9 Với mục tiêu phát triển bền vững và mang lại lợi ích lâu dài cho cổ đông và nhà đầu tư, DHG Pharma luôn chú trọng minh bạch thông tin và duy trì kênh giao tiếp hiệu quả với các bên liên quan thông qua các hoạt động sau:", "evidence": "DHG Pharma duy trì và cập nhật kịp thời thông tin trên trang web chính thức https://www.dhgpharma.com.vn. Trang web có chuyên mục “Cổ đông” cung cấp thông tin minh bạch, đầy đủ và dễ tiếp cận cho cổ đông và các bên liên quan. Các thông tin công bố được đăng tải song ngữ (tiếng Việt và tiếng Anh), đảm bảo tính minh bạch và thuận tiện trong tra cứu. Sử dụng thông tin có liên quan và có chất lượng Thông tin và truyền thông Truyền thông trong nội bộ Với tinh thần không ngừng cải tiến và phát triển theo các tiêu chuẩn và thông lệ tốt, trong năm 2024, hệ thống kiểm soát nội bộ của DHG Pharma đã triển khai hàng loạt hoạt động quan trọng nhằm nâng cao chất lượng quản trị, tăng cường tính tuân thủ và"}\n{"id": "B-bc7d31cb71a1c7c0", "claim": "Trong Lễ công bố các doanh nghiệp Phát triển Bền vững, Traphaco nằm trong Top 10 các doanh nghiệp phát triển bền vững nhờ những dự án, những hoạt động vì cộng đồng của mình, và Traphaco cũng là doanh nghiệp Dược duy nhất nằm trong Top 10 này.", "evidence": ""}\n{"id": "A-2f03db3f081d085c", "claim": "TRAPHACO NĂM 2025 TRAPHACO - Báo cáo thường niên 2025 Dấu ấn năm 2025 Ngày 19/11/2025, Traphaco được công bố nằm trong Top 10 Doanh nghiệp Bền vững tại Việt Nam (CSI) năm 2025.", "evidence": "TRAPHACO - Báo cáo thường niên 2025 TRAPHACO NĂM 2025 GIẢI THƯỞNG TIÊU BIỂU QUA CÁC NĂM Bằng khen của Thủ tướng Chính phủ Top 10 Doanh nghiệp Sao vàng đất Việt tiêu biểu Trách nhiệm xã hội TRAPHACO NĂM 2025 TRAPHACO - Báo cáo thường niên 2025 Dấu ấn năm 2025 Trong năm 2025, Traphaco triển khai tái cấu trúc toàn diện hệ thống kinh doanh theo hướng chuyên sâu và tinh gọn. Ở kênh ETC, Công ty tách biệt lực lượng bán hàng, kiện toàn nhân sự và nâng cao chất lượng triển khai bán hàng."}\n{"id": "A-7ac0a9b057875618", "claim": "cạnh đó, Các nguồn phát thải xuất phát từ việc sản xuất năng lượng từ bên ngoài", "evidence": "lực trong hoạt động của Tập đoàn Hòa Phát. Đây là cơ sở quan trọng để xây dựng các giải pháp quản lý, sử dụng năng lượng hiệu quả và ứng dụng công nghệ xanh trong thời gian tới. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trư"}\n{"id": "A-070ef4cd6035271a", "claim": "Sử dụng vật liệu xây dựng bền vững và thiết kế tối ưu để giảm lượng khí thải carbon Giảm thiểu rác thải bằng cách thực hiện các chương trình tái chế và giảm thiểu chất thải Bảo vệ và bảo tồn tài nguyên thiên nhiên bằng cách thực hiện các biện pháp quản lý nước bền vững", "evidence": "Thu hút và thiết lập hệ sinh thái ESG trong hoạt động của Công ty. Giảm thiểu rác thải bằng cách thực hiện các chương trình tái chế và tối ưu hóa việc giảm thiểu chất thải. Đảm bảo sức khỏe và an toàn cho người lao động là ưu tiên hàng đầu. 4 TIÊU CHÍ QTCT CỦA COTECCONS Trách nhiệm Đảm bảo tính minh bạch và chính trực. Bảo vệ và bảo tồn tài nguyên thiên nhiên bằng cách thực hiện các biện pháp quản lý nước bền vững. Gắn kết cộng đồng: nền tảng bền vững cho sự phát triển của ngành xây dựng."}\n{"id": "B-994bbcf63b7147a1", "claim": "At PNJ, most of solid waste arises from the operation of PNJP, including: ANNUAL REPORT 2020 | 83", "evidence": ""}\n{"id": "B-946f3af61e51a9f4", "claim": "With the amount of consumed fuel as determined in the Energy section, the amount of CO2 emitted by fuel use in 2018 was 87,481 tons under the following assumptions: • CO2 emissions are calculated based on the average calorific value of fuel and the CO2 emission factor of the Vietnamese electricity grid *(= 0.8154, according to Document No.", "evidence": ""}\n{"id": "B-369f59b47defd51c", "claim": "Additionally, we maintain a responsible approach to water conservation across our operations.", "evidence": ""}\n{"id": "B-190aea657096c52a", "claim": "Non-renewable energy The volume of DO and FO used in 2024 was 2,182,398 liters.", "evidence": ""}\n{"id": "A-c3885e205569328d", "claim": "Công ty đang chủ động kiểm soát và tối ưu hiệu quả vận hành, hướng tới mục tiêu tăng trưởng bền vững.", "evidence": "Kết quả phản ánh nỗ lực đồng bộ trong điều hành, tối ưu hiệu quả và kiểm soát chi phí để tăng trưởng bền vững. Cung với đó công ty tiếp tục củng cố nền tảng vùng trồng dược liệu đạt GACP- WHO, có thêm 03 vùng trồng dược liệu đạt chứng nhận Các giải pháp này giúp Công ty kiểm soát rủi ro tài chính hiệu quả hơn, đồng thời tạo nền tảng cho tăng trưởng bền vững. Năm 2025, Traphaco tiếp tục đẩy mạnh số hóa các hoạt động vận hành cốt lõi như bán hàng, logistics, quản lý kho và chuỗi cung ứng. Việc tích hợp các hệ thống DMS, ERP và BI giúp tăng cường kiểm soát doanh thu, chiết khấu, tồn kho và hiệu quả theo từng kênh, từng địa bàn;"}\n{"id": "B-582e9def86a9d8fe", "claim": "Đặc biệt, chúng tôi quyết liệt triển khai có chiều sâu tiêu chuẩn ESG, lan tỏa rộng và mạnh nhận thức về trách nhiệm với môi trường tự nhiên và môi trường xã hội...", "evidence": ""}\n{"id": "B-10f1adedcaffb78c", "claim": "The percentage of pale meat fish fillet decreased from 9 -12% to 1 – 4% • The percentage of white meat fish fillet increased from 90 – 95% to 98 – 99% (an almost absolute • The percentage of the meat fish filleting wastes dropped from 0.2 – 0.6% to 0.06% - 0.1% • A redundancy of 10 employees / day", "evidence": ""}\n{"id": "B-d5b87b8e7a39b983", "claim": "Tích hợp ESG vào chiến lược quản trị và điều hành, hướng tới mục tiêu tăng trưởng bền vững và trách nhiệm với xã hội.", "evidence": ""}\n{"id": "B-90d1969213bd57ed", "claim": "được triển khai tại Văn miếu Trấn Biên, góp phần cải thiện cảnh quan và lan tỏa ý thức bảo vệ môi trường Gắn kết mục tiêu môi trường Những hoạt động trên thể hiện khát vọng và cam kết đóng góp dài lâu của Bảo Việt cho những hoạt động phát triển bền vững, là lời kêu gọi mạnh mẽ đến cộng đồng, cùng lan tỏa lối sống xanh, bảo vệ môi trường sống, vì một tương lai trong lành và khỏe mạnh cho các thế hệ mai sau.", "evidence": ""}\n{"id": "A-3ddce7ec3da8fcbb", "claim": "Bùn sinh học được tuần hoàn lại các bể hiếu khí, kị khí để tiếp tục xử lý nước và một phần bùn dư được chứa tại bể chứa bùn sinh học sau đó được đơn vị có chức năng thu gom và xử lý chất thải nguy hại theo đúng luật định.", "evidence": "Bùn thải từ bể tự hoại và hệ thống xử lý nước thải tập trung Do hoạt động trong lĩnh vực sản xuất, gia công vàng, bạc, PNJP phát sinh một số chất thải nguy hại, chủ yếu bao gồm: Bùn thải từ hệ thống xử lý nước thải tập trung bao gồm bùn sinh học từ bể lắng sinh học và bùn từ bể lắng hóa lý. Chất thải nguy hại được phân loại, thu gom và lưu trữ vào kho chứa chất thải nguy hại đáp ứng đầy đủ yêu cầu của luật. PNJP đã hợp đồng với các đơn vị có chức năng để thu gom, vận chuyển và xử lý các chất thải nguy hại trên theo đúng quy định của luật. Ngoài ra, PNJP đã tiến hành đăng ký Sổ Chủ nguồn thải chất thải nguy hại với Sở Tài nguyên và Môi trường TP."}\n{"id": "A-90f46dd02fd1760e", "claim": "Cùng với chương trình tham gia trồng rừng đặc dụng bảo vệ môi trường sinh trưởng của loài voọc tại Quảng Bình, Coteccons cũng tham gia hoạt động hỗ trợ sinh kế cho người dân địa phương.", "evidence": "Bên cạnh trồng rừng đặc dụng, Coteccons còn tham gia hoạt động hỗ trợ sinh kế cho người dân địa phương nhằm góp phần gia tăng hành lang bảo vệ voọc, hạn chế tình trạng lấn chiếm đất rừng, khai thác gỗ rừng và tài nguyên rừng trái phép hoặc chăn thả gia súc tại những nơi đàn voọc sinh sống. Bác Nguyễn Thanh Tú - công tác tại Hợp tác xã Sản xuất Kinh doanh Nông Lâm nghiệp và tự nguyện bảo tồn voọc gáy trắng tại xã Thạch Hóa, huyện Tuyên Hóa, Quảng Bình, là một trong những người dân địa phương đồng hành sâu sát nhất cùng Coteccons. Coteccons là một trong những đơn vị hỗ trợ cho Hợp tác xã Voọc gáy trắng – nhóm tình nguyện viên bảo vệ loài voọc gáy trắng để xây dựng một hệ sinh thái an toàn hơn."}\n{"id": "B-0395612ccda33716", "claim": "Ngay trong giai đoạn 2 năm biến động vì dịch Covid-19, Vinamilk đã vượt qua những khó khăn thách thức và tiến hành xây dựng hệ thống trang trại sinh thái Vinamilk Green Farm, đây được coi như là sự cam kết mạnh mẽ của Công ty trong việc thực hiện các chiến lược phát triển bền vững, với 03 khía cạnh chủ đạo:", "evidence": ""}\n{"id": "A-4ab4fc0badf3051e", "claim": "Năm 2025 là năm đầu tiên Tập Đoàn Hòa Phát lập báo cáo phát thải KNK toàn", "evidence": "ện năng tiêu thụ so với phương pháp truyền thống. • Thu hồi nhiệt từ quá trình thiêu kết, sử dụng nhiệt dư để phát điện, góp phần tối ưu hóa hiệu quả sử dụng năng lượng tổng thể. • Công nghệ đúc – cán liên tục cho phép nạp trực tiếp phôi nóng (750–900°C) vào dây chuyền cán. giúp giảm nhu cầu gia nhiệt, tiết kiệm năng lượng và giảm phát thải CO2 trên mỗi tấn thép. • Tận dụng các nguồn hơi nước từ hệ thống làm mát lò thổi, lò nung phôi để tận dụng phát điện. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT Năm 2025 là năm đầu tiên Tập Đoàn lập báo cáo phát thải KNK toàn tập đoàn, do vậy kỳ báo cáo từ ngày 01/01/2025 đến 31/12/2025 được lựa chọn làm năm cơ sở vì đây là kỳ báo cáo đầu t"}\n{"id": "B-ad63422e5dc9bdf8", "claim": "Mục tiêu là góp 11.000 cây xanh vào dự án trồng rừng nguyên sinh tỉnh Quảng Bình, Quảng Trị.", "evidence": ""}\n{"id": "A-25029c890c992b3c", "claim": "Established in 2023, the Sustainable Development Steering Committee plays a leading role in overseeing and ensuring effective sustainability management across the Group.", "evidence": "(ii) The Sustainability Steering Committee, which plays a leading role in overseeing and ensuring effective management of sustainability across the Group. Its members include the Chief Executive Officer and three representatives from the Group’s key business sectors: Agriculture, Aquaculture, and Food Processing; SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Sustainability Committee Established in 2021, the Sustainable Development Committee under the Board of Directors serves as the highest governing body of the Group responsible for overseeing the implementation of its sustainable development strategy, while driving efforts to keep pace with rapid global changes."}\n{"id": "B-a251690a0e988938", "claim": "In 2023, PNJ aims to reduce energy consumption in accordance with the roadmap to Net Zero Emissions of the National target, which entails a 1% reduction per year.", "evidence": ""}\n{"id": "A-e8c0bd3acc74d64f", "claim": "The report focuses on presenting SABECO\'s sustainability efforts for the year 2024, addressing the key concerns raised by our stakeholders.", "evidence": "1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas STRATEGIC FRAMEWORK FOR THE FUTURE 2024 KEY PROJECTS Meanwhile, the importance to stakeholders assesses the degree of stakeholder concern about the topic, as well as the degree to which the company creates positive or negative impacts on the broader environment and society as a result of its operations. By prioritizing the topics accordingly, this ensures that SABECO’s sustainability efforts are focused on areas of greatest importance and influence. Climate Change/Greenhouse Gas Emissions and Energy Consumption"}\n{"id": "B-35954e5c519a0019", "claim": "CAM KẾT SỬ DỤNG NĂNG LƯỢNG VÀ QUẢN LÝ NƯỚC BỀN VỮNG Công ty đã chính thức ban hành cam kết sử dụng năng lượng bền vững và đưa vào hoạt động hàng ngày.", "evidence": ""}\n{"id": "B-4d4ec7ebb932e241", "claim": "Some breweries have also achieved ISO 50001 (Energy Management System) in 2023, while all employees have been trained frequently on labor safety, food hygiene and safety as required by the law.", "evidence": ""}\n{"id": "B-da3b456d8aadbcbe", "claim": "Lĩnh vực sản xuất kinh doanh điện Định hướng đến năm 2050, tổng công suất đặt là 25.892 MW, chiếm khoảng 5% công suất đặt toàn hệ Giai đoạn đến 2050, PV Power tập trung phát triển các nguồn năng lượng tái tạo và năng lượng mới phù hợp lộ trình phát triển quy hoạch ngành điện, tổng công suất năng lượng tái tạo và năng lượng mới chiếm khoảng 3% tổng công suất nguồn năng lượng tái tạo và năng lượng mới trong toàn hệ thống;", "evidence": ""}\n{"id": "B-f88e6e56e45ec98f", "claim": "Tại buổi lễ này, Traphaco lần thứ 2 liên tiếp được tôn vinh Top 10 doanh nghiệp bền vững trong bảng xếp hạng các doanh nghiệp bền vững năm 2017 lĩnh vực Giải thưởng này cũng đã khẳng định chiến lược “Con đường sức khỏe xanh” của Traphaco chính là minh chứng rõ nét cho sự phát triển bền vững, gắn lợi ích doanh nghiệp với lợi ích môi trường và xã hội.", "evidence": ""}\n{"id": "A-1a0f6bf6d730e4fe", "claim": "Coteccons hướng tới hợp tác với các đối tác năng lượng tái tạo để mua sắm năng lượng cần thiết cho các dự án.", "evidence": "Mỗi nhà cung cấp đã, đang và sẽ hợp tác đều là các đối tác quan trọng góp phần tạo nên thành công của Coteccons nói chung và dự án nói riêng. Khối Thu mua luôn đặt mục tiêu cân bằng quyền lợi giữa công ty và các đối tác để đảm bảo sự đồng hành lâu dài và bền vững. Hàng năm, Khối Thu mua sẽ chọn các đối tác tiềm năng và phát triển thành đối tác chiến lược thông qua ký kết hợp đồng nguyên tắc để cùng đồng hành trong quá trình phát triển. Cũng tại hội nghị, Coteccons đã tiến hành ký kết thỏa thuận hợp tác chiến lược với một số đối tác để phát triển sản phẩm - dịch vụ theo hướng bền vững. Trong năm qua, Coteccons tích cực tổ chức và tham gia nhiều chương trình, sự kiện, gặp gỡ để thúc đẩy tinh t"}\n{"id": "A-cd33be3daca60c4b", "claim": "PNJP đã hợp đồng với các đơn vị có chức năng để thu gom, vận chuyển và xử lý các chất thải nguy hại trên theo đúng quy định của luật.", "evidence": "- Các loại hóa chất thải: hóa chất phòng thí nghiệm thải, xăng dầu, dầu động cơ hộp số bôi trơn thải, ... - Bùn thải từ hệ thống xử lý nước thải. Bùn sinh học được tuần hoàn lại các bể hiếu khí, kị khí để tiếp tục xử lý nước và một phần bùn dư được chứa tại bể chứa bùn sinh học sau đó được đơn vị có chức năng thu gom và xử lý chất thải nguy hại theo đúng luật định. Bùn thải từ cụm xử lý hóa lý hàng ngày được thu gom bằng dàn lọc bùn sau đó được đưa vào kho chứa chất thải nguy hại, định kỳ hàng tháng sẽ bàn giao cho đơn vị chức năng thu gom và xử lý chất thải nguy hại theo đúng quy định. hợp đồng với Công ty TNHH MTV dịch vụ công ích quận Gò Vấp để thu gom, vận chuyển và xử lý theo quy định c"}\n{"id": "A-63d196387d14d4a9", "claim": "Họ đã chia sẻ những kiến thức và công nghệ hướng đến Net Zero, cung cấp được các chứng nhận trong nước và quốc tế để chứng minh tính thân thiện với môi trường của sản phẩm.", "evidence": "TABLE Phạm Xuân Thu - Giám đốc Khối Thu mua “Chúng tôi nhận thức được việc chọn lựa vật liệu trong xây dựng vô cùng quan trọng, vì vật liệu sẽ có tác động trực tiếp đến môi trường trong suốt vòng đời của một công trình. Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt khe và có trách nhiệm để lựa chọn những nguyên vật liệu ít tác động xấu đến môi trường, có khả năng tái chế hoặc nguyên vật liệu tái chế, tiết kiệm được năng lượng và tài nguyên từ nguồn gốc sản xuất và trong quá trình sử dụng. Coteccons cũng ưu tiên nguyên vật liệu địa phương, vật liệu có độ bền cao, vật liệu xanh từ công nghệ mới... Chúng tôi đã có cơ hội gặp gỡ những đối tác chuyên nghiệp với những sản phẩm, giải pháp t"}\n{"id": "B-70cd72d5661d61d5", "claim": "Những nỗ lực bền bỉ trong thực hiện trách nhiệm môi trường, xã hội và cộng đồng tiếp tục khẳng định vai trò của DHG Pharma là doanh nghiệp phát triển bền vững, có trách nhiệm và đồng hành cùng sự phát triển chung của đất nước.", "evidence": ""}\n{"id": "A-0c2f2e1941d9d4d9", "claim": "DHG Pharma đang nỗ lực để đáp ứng kỳ vọng của các bên liên quan đối với tính minh bạch thông tin, tăng trưởng bền vững trong kinh doanh, thể hiện trách nhiệm của doanh nghiệp với cộng đồng.", "evidence": "Quản trị hiệu quả, minh bạch gắn liền với trách nhiệm với môi trường và góp phần xây dựng cộng đồng, xã hội. Nội dung báo cáo là minh chứng cho sự cam kết mạnh mẽ của DHG Pharma trên con đường phát triển bền vững trước kỳ vọng của các bên liên quan. Bước 05 Nguyên tắc phát triển bền vững được tích hợp vào chiến lược tổng thể của DHG Pharma. Đây cũng là sự khẳng định mạnh mẽ về cam kết của DHG Pharma trong hành trình hướng tới sự phát triển bền vững toàn diện, hài hòa giữa tăng trưởng kinh tế, trách nhiệm xã hội và bảo vệ môi trường. Đối với các vấn đề trọng yếu, DHG Pharma xác định phương pháp quản trị dựa trên bối cảnh ngành dược trong nước và thế giới, thông tin phản hồi và mối quan tâm gi"}\n{"id": "B-69eb839da0e2f3fb", "claim": "Cam kết chiến lược phát triển bền vững dựa trên khung ESG vững chắc xuyên suốt trong quá trình vận hành của công ty.", "evidence": ""}\n{"id": "A-1d3f1ed3c85e31ed", "claim": "Trẻ hóa đội ngũ nhằm tiếp nối giá trị bền vững Cơ cấu nhân sự theo vùng miền Tại TP.HCM, PNJ sở hữu hai nhà máy sản xuất với quy mô lớn đặt ở Gò Vấp (4 triệu sản phẩm) và Long Hậu (2 triệu sản phẩm) đã đáp ứng mạnh mẽ nhu cầu của thị trường trong và ngoài nước.", "evidence": "2020). Đó là do sự phục hồi nền kinh tế sau đợt giãn cách Covid 19, nhu cầu sản xuất tăng lên, sản lượng sản xuất đang trên đà hồi phục với mức tăng trưởng 27% trong năm vừa qua. Đặc biệt trong năm 2021, nhà máy Long Hậu mới đi vào hoạt động, hệ thống sử dụng và xử lý nước ở nhà máy này vẫn chưa hoàn toàn hiệu quả so với nhà máy Gò Vấp, làm bình quân lượng nước sử dụng trên sản phẩm và lượng nước tái chế chưa được tối ưu như 5 năm gần đây. TẠI 02 NHÀ MÁY PNJP (GÒ VẤP VÀ LONG HẬU) TRONG PHẠM VI 1 VÀ 2 ranh giới và phạm vi PNJ tích cực ỨNG PHÓ BIẾN ĐỔI KHÍ HẬU 2022"}\n{"id": "A-af5f6cc7aceb72d1", "claim": "By collecting and treating this water for reuse in applications such as equipment cleaning, supplying water for production processes that do not require high quality, and cooling, our brewery aims to reduce overall water consumption and lower wastewater treatment costs.", "evidence": "The primary purpose of the harvested rainwater is to support the brewery’s cooling and cleaning processes. The system is expected to reduce the brewery’s water usage by 4% during the six-month rainy season in 2024, representing a substantial cost saving of 148 million VND. SABECO is committed to responsibly managing wastewater throughout the beer production process. 5.7% 100% Reduce/ Reuse/ Recycle The rainwater harvesting system at SABECO’s Cu Chi Brewery in Vietnam represents a sustainable initiative to conserve vital natural resources and reduce operational costs. Using a rooftop area of 20,000 square meters, the system was developed with minimal new investment, as it repurposed old tanks"}\n{"id": "B-c7c7cb910c86b7fc", "claim": "Raise awareness and promote sustainable development through communication and promotion of environmental and social programs.", "evidence": ""}\n{"id": "A-fde3830955719c95", "claim": "Song song, 26% dòng vật liệu bền Đặc biệt, hệ thống dữ liệu trên phần mềm được hiển thị trực quan hóa bằng các biểu đồ, từ đó, dễ dàng so sánh, đánh giá các xu hướng phát thải để đưa ra quyết định đúng đắn và chiến lược phù hợp với mức độ phát thải mà", "evidence": "Các thành viên độc lập đã phát huy tốt vai trò giám sát và kiểm soát, song hành cùng HĐQT đưa ra những quyết định công bằng, minh bạch, thận trọng và có trách nhiệm. Hỗ trợ lựa chọn các quyết định chiến lược: Đưa ra ý kiến độc lập và tư vấn cho HĐQT trong việc xây dựng và thực hiện các chiến lược phát triển của Công ty. Các thành viên độc lập đưa ra những định hướng và giải pháp hữu ích, góp phần đem lại lợi ích cao nhất cho Công ty, không gây xung đột về lợi ích giữa các bên liên quan, tạo ra đối trọng để hài hòa lợi ích giữa các nhóm cổ đông. Báo cáo và phân tích dữ liệu lượng phát thải của doanh nghiệp. Đặc biệt, Công ty đã xây dựng được một đội ngũ nhân sự có kỹ năng cơ bản trong kiểm kê"}\n{"id": "B-552e914ece25dd11", "claim": "duy trì việc không sử dụng chai nước bằng nhựa.", "evidence": ""}\n{"id": "B-ed577a2463f7daa5", "claim": "Đối với đơn vị vận hành nhà máy nhiệt điện đều kiểm soát liên tục nguồn nước thải, khí thải đảm bảo tuân thủ quy chuẩn về môi trường;", "evidence": ""}\n{"id": "A-64bae21b7cc0f1f0", "claim": "1,361,826 1,412,120 1,558,439 PAN Group places strong emphasis on sustainable water resource management as a key factor supporting production stability, community well- being, and ecological balance.", "evidence": "2024 2025 ENERGY WATER AND WASTEWATER EVN grid electricity KWh 81,215,939 110,780,751 116,831,283 Municipal Water Supply m3 1,361,826 1,412,120 1,558,439 Solar power plant SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT COMMUNITY VALUE SHARING Spreading the Value of Sharing PAN Group consistently places strong emphasis on community activities, demonstrating its corporate social responsibility toward society and the environment."}\n{"id": "B-34f9626b1d8e265c", "claim": "Với định hướng phát triển mang tính bền vững với hệ sản phẩm được nâng cao chất lượng ngay từ đầu vào và ngày càng thân thiện hơn với môi trường, an toàn hơn cho người tiêu dùng, cùng với nền tảng, năng lượng và tiềm lực trải qua 47 năm hình thành và phát triển, qua đó Công ty tiếp tục “HÀNH ĐỘNG VÀ CHUYỂN ĐỔI” để nâng tầm vị thế, giá trị của thương hiệu Vinamilk không chỉ tại Việt Nam mà còn trên bảng xếp hạng ngành sữa toàn cầu.", "evidence": ""}\n{"id": "A-4a61dd25b56c2eff", "claim": "việc tái chế một tấn thép phế có thể giúp giảm khoảng 1,5 tấn khí CO2.", "evidence": "Việc sử dụng nhiều hơn thép phế giúp tăng sản lượng, giảm tỷ trọng nguyên liệu phát thải cao, giảm tiêu hao than cốc và các nhiên liệu trên một tấn thép, đồng thời tiết kiệm năng lượng trong toàn bộ chuỗi sản xuất. Trong năm 2025, tỷ lệ phối phế cùng gang lỏng tại Khu liên hợp Gang phát triển bền vững. Hoạt động kinh doanh tại Tập đoàn Phế liệu thép sau tiêu dùng 100% có thể tái chế Tái sử dụng & tái sản xuất"}\n{"id": "A-1938fe02b3ba5a9e", "claim": "\uf0fe Công bố chính sách và các thực hành, mô tả nỗ lực của công ty nhằm bảo đảm chuỗi giá trị của công ty thân thiện với môi trường hoặc phù hợp với việc thúc đẩy phát triển bền vững.", "evidence": "\uf0fe Công bố chính sách và các thực hành, mô tả nỗ lực của công ty trong việc tương tác với cộng đồng nơi công ty hoạt động. \uf0fe Traphaco đã công bố chính sách này trong báo cáo phát triển bền vững. C.1.4 B.3 Phải ngăn cấm giao dịch nội gián và lạm dụng mua bán tư lợi cá nhân. D.2.8 Công ty có công bố rõ ràng chính sách và thực hành, nỗ lực về các chương trình đào tạo và phát triển cho nhân viên. \uf0fe Traphaco đã thực hiện tốt và nêu đầy đủ trong Báo cáo phát triển bền vững. C.3.2 D.3. Công bố Giao dịch bên liên quan (GDBLQ)"}\n{"id": "B-478d51a2f42c1fac", "claim": "hiệu quả sâu rộng, có giá trị và tác động thực tế, Công ty đang nỗ lực làm cho ESG trở nên hữu hình và có ý nghĩa đối với tất cả nhân viên và các bên liên quan.", "evidence": ""}\n{"id": "B-a2c8c39b965d9a2d", "claim": "Mở rộng không gian xanh gia tăng chuyển đổi khí CO2 thành khí O2 trong các văn phòng Điển hình như 50m2 cây xanh tại Trung tâm Công nghệ thông tin tại 71 Ngô Sỹ Liên, 140m2 tại Tòa nhà Bảo Việt số 8 Lê Thái Tổ và hơn 300m2 mảng xanh tại Tòa nhà Trung tâm tài chính Bảo Việt 233 Đồng Khởi.", "evidence": ""}\n{"id": "B-31ed36f21801c29f", "claim": "chủ động tìm kiếm các cơ hội hợp tác đầu tư hoặc M&A đối với các dự án tiềm năng trong và ngoài nước phù hợp xu thế chuyển dịch năng lượng.", "evidence": ""}\n{"id": "A-6c87c065a2811232", "claim": "Phát triển cộng đồng Hành trình số hóa quản trị nhân sự cũng sẽ được đẩy mạnh, với việc mở rộng thêm các phân hệ mới, hướng tới mục tiêu hình thành hệ sinh thái quản lý nhân sự số hóa toàn diện trên phạm vi toàn Tập đoàn.", "evidence": "CON NGƯỜI Trong năm 2026, sau khi hoàn thành giai đoạn nâng cấp, Tập đoàn sẽ tiếp tục mở rộng sang: Ứng dụng di động cho nhân viên; Phân hệ công lương; Quản lý đào tạo; Đánh giá hiệu quả công việc. Tất cả hướng đến mục tiêu: Mỗi cán bộ nhân viên Hòa Phát có thể chủ động quản lý hồ sơ cá nhân, theo dõi lộ trình phát triển và xử lý các thủ tục nhân sự một cách thuận tiện trên nền tảng số. Công tác đánh giá giá trị công việc trên phạm vi toàn Tập đoàn đã hoàn thành vào cuối năm 2024. Đây là nền tảng cho hệ thống các chính sách nhân sự được hoàn thiện, phù hợp chiến lược sản xuất kinh doanh của Tập đoàn. Việc đánh giá nhân sự theo năng suất - hiệu quả công việc và chế độ lương theo ngạch bậc, qu"}\n{"id": "A-a4269abda71c3cb1", "claim": "The Code of Conduct consists of five main sections outlining policies and rules, including: PAN Group pursues long-term sustainable development commitments, with a strategic orientation aligned with international standards and practices.", "evidence": "Strategy The Group’s tax strategy is developed and implemented in alignment with its sustainable development orientation and good governance practices. Internal tax policies and guidelines are applied consistently across the system to ensure efficient, transparent, and compliant tax practices in accordance with both domestic and international standards. Enhancing and enforcing anti-corruption policies SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE Code of Conduct Recognized Sustainability Standards Issued in 2017 together with the Practice Principles, these two components form a comprehensive framework outlining PAN Group’s perspectives, requirements, and internal as well as external m"}\n{"id": "B-1b4d25f08cb51dde", "claim": "contributed to a safer and more sustainable living environment in rural areas, reflecting our dedication to both Environmental Sustainability and Community Development.", "evidence": ""}\n{"id": "A-558de672076a43c7", "claim": "Năm 2022, chúng tôi sẽ luôn nỗ lực hoàn thành các mục tiêu phát triển bền vững đã đề ra, tiếp tục khai phóng các nội năng, thực thi nhiều sáng kiến đổi mới, sáng tạo để chinh phục những tầm cao mới trên hành trình sắp tới, và chung tay kiến tạo những giá trị tốt đẹp cho cộng đồng, xã hội.", "evidence": "Tiến trình F5 sẽ hành động thần tốc hơn, mãnh liệt và sâu rộng hơn để gặt hái thêm nhiều thành quả, chinh phục những đỉnh cao mới, tiếp tục tạo nên những điều tốt đẹp diệu kỳ cho cộng đồng xã hội. Ở PNJ, chúng tôi đều có một tầm nhìn xa về chiến lược quản trị con người để không gây nên một cú sốc nào khi có một sự biến động không lường trước xảy ra. Bằng sức mạnh niềm tin và năng lực ứng biến linh hoạt, chỉ trong hơn 3 tháng đầu năm 2021, PNJ đã hoàn thành gần 50% kế hoạch năm, lợi nhuận gộp của Tập đoàn trong năm 2021 đạt trên 3.598 tỷ, tăng trưởng 4,8% so với cùng kỳ năm 2020. NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM"}\n{"id": "B-ce37eac4c2780590", "claim": "Từ các kết quả đánh giá của Dow Jones - CSA Ratings, bên cạnh việc doanh nghiệp đạt điểm cao ở các lĩnh vực như minh bạch thông tin, quản trị rủi ro và báo cáo khí nhà kính, Bảo Việt nhận thức rõ và đặt mục tiêu tập trung vào các vấn đề sau để cải thiện hoạt động của doanh nghiệp như: Trong những năm tới, Bảo Việt sẽ tập trung cải thiện các yếu tố trên nhằm nâng cao xếp hạng CSA, tăng tính cạnh tranh trong lĩnh vực tài chính và bảo hiểm bền vững.", "evidence": ""}\n{"id": "A-bf0ee5f6af6237b2", "claim": "Tại PNJ, công cuộc tích hợp cùng các mục tiêu phát triển bền vững.", "evidence": "Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát, đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được tích hợp cùng các mục tiêu phát triển bền vững. Căn cứ nội dung 17 mục tiêu phát triển bền vững của Liên Hợp Quốc, PNJ đã nghiên cứu, lựa chọn trên cở sở đặc thù lĩnh vực hoạt động, bối cảnh kinh tế, xã hội Việt Nam, đồng thời tích hợp vào định hướng và các mục tiêu phát triển bền vững của công ty. Tại PNJ, công cuộc phát triển bền vững được tiến hành đồng thời cả bên trong và bên ngoài doanh nghiệp. kết hợp chặt chẽ, hợp lý và hài hòa giữa phát triển kinh tế với phát triển xã hội và bảo vệ tài nguyên và môi trường, chủ động ứng phó với"}\n{"id": "B-11752145bee65921", "claim": "- Orange bin: containing recyclable waste, we dispose of drinking water bottles, waste paper, plastic and metal wastes that do not contain hazardous ingredients.", "evidence": ""}\n{"id": "A-31fb68eb5cd72322", "claim": "Member companies of PAN Group have also integrated content on energy and resource saving into company regulations, communication activities, and training sessions for more than 10,000 employees on Net Zero and sustainable development.", "evidence": "The PAN Group is committed to strictly complying with state regulations on the environment, labor, and social affairs. We have established an environmental and social management system from the Group level down to all member companies, setting sustainable development targets for environmental protection, and ensuring the safety and well-being of our employees and communities. With nearly 11,000 direct employees, PAN The PAN Group focuses on building a common corporate culture with shared values, close connections between member companies, while respecting the differences and traditions of each company. between the parent company and member companies through environmental activities, sustaina"}\n{"id": "A-6f0aee54665f1406", "claim": "Coteccons đã triển khai công tác kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons TRIỂN KHAI HỆ THỐNG KIỂM KÊ KHÍ NHÀ KÍNH Coteccons đã triển khai các biện pháp bền vững và thân thiện với môi trường trong suốt quá trình xây dựng nhà máy LEGO tại Bình Dương. Reporting company Upstream activities Downstream activities Hệ thống kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 Dự án Thung lũng Thanh Xuân (tỉnh Vĩnh Phúc)"}\n{"id": "A-c424f0ae8d2eeaae", "claim": "Nguyên tắc 3R (Reduce – Reuse – Recycle) được Coteccons thiết lập và triển khai từ năm", "evidence": "Reuse - Tái sử dụng; Recycle - Tái chế) trong chính sách giảm thiểu tác động đến môi trường của Công ty. Ecopark cũng là một công trường điển hình về thực hành nâng cao tiêu chuẩn ESG của Coteccons. Tại đây, các tiêu chuẩn về môi trường, an toàn và quyền lợi cho người lao động đều được thiết lập ở mức cao nhất. Kết quả trong năm đã có 411kg rác giấy khô, 507kg rác giấy thùng, 174kg rác nhựa khô đã được phân loại. Khối Kỹ thuật và Tuân thủ đã dùng phương châm “REUSE đồ cũ để REDUCE phát thải rác” trong quá trình trang trí cho khu vực chung của tầng 11 – tòa nhà Coteccons với tác phẩm “Đẹp không đụng hàng” trong dịp chào mừng Tết Nguyên đán. Khi nhắc đến cây thông chúng ta ngay lập tức nghĩ đế"}\n{"id": "B-70f2db796478a3ee", "claim": "Đồng thời, Công ty cũng nhận thấy tầm quan trọng của việc truyền thông về ESG, lan tỏa tinh thần và nhận thức về phát triển xanh, thúc đẩy một cách thiết thực thực hành ESG trong cộng đồng.", "evidence": ""}\n{"id": "B-84620b28fe2bf4cd", "claim": "lượng rác thải khoảng 36.642.000 kg/năm được quản lý bằng các biện pháp: • Phân loại tại nguồn: phổ biến và thực hiện quy định phân loại thành chất thải phế liệu và chất thải thải bỏ đến mỗi bộ phận, phòng ban.", "evidence": ""}\n{"id": "B-946207b84d231c57", "claim": "Đến nay PV Power REC đã và đang sở hữu 5 dự án/ nhà máy điện mặt trời mái nhà và 02 hệ thống tiết kiệm năng lượng sử dụng biến tần tại Nhà máy nhiệt điện Vũng Áng 1.", "evidence": ""}\n{"id": "B-b84b39b14e07e582", "claim": "Áp dụng công nghệ, khoa học: Tiếp tục thực hiện Kế hoạch chủ động tham gia Cách mạng Công nghiệp lần thứ 4, tập trung vào việc số hóa các quy trình sản xuất và quản lý để nâng cao hiệu suất và linh hoạt trong hoạt Đưa ra các giải pháp tiết kiệm năng lượng, thúc đẩy sử dụng nguồn năng lượng tái tạo như điện mặt trời và ánh sáng tự nhiên để giảm thiểu ảnh hưởng tiêu cực đối với môi trường;", "evidence": ""}\n{"id": "A-e0cb26254ce82889", "claim": "Rủi ro phát sinh ô nhiễm từ hoạt động sản xuất và xử lý chất thải Ban hành hướng dẫn công việc quản lý môi trường về thu gom, phân loại, xử lý chất thải.", "evidence": "Để kiểm soát chất thải và hạn chế tác động đến môi trường, DHG Pharma triển khai nghiêm túc công tác quản lý chất thải theo đúng quy định. Hằng năm, Công ty ký hợp đồng với các đơn vị có chức năng thu gom và xử lý chất thải. Toàn bộ chất thải từ hoạt động sản xuất và sinh hoạt được thu gom, phân loại và xử lý theo đúng quy định, đảm bảo tuân thủ các tiêu chuẩn môi trường hiện hành. PHÁT TRIỂN Đạo đức kinh doanh BỀN VỮNG MÔI TRƯỜNG XÃ HỘI Tuân thủ các quy định Vận hành hệ thống xử lý nước thải đảm bảo chất lượng nước sau xử lý đạt tiêu chuẩn trước khi thải ra môi trường hoặc tái sử dụng. Thực hiện phân loại, thu gom và xử lý chất thải (đặc biệt là chất thải nguy hại) theo quy định nghiêm ngặt"}\n{"id": "B-8bf42ae2d76cf26a", "claim": "Trong quá trình hoạt động, DHG Pharma chủ động triển khai nhiều giải pháp nhằm cắt giảm phát thải khí nhà kính, nâng cao tỷ lệ tái chế và tái sử dụng chất thải, cũng như từng bước đầu tư đổi mới công nghệ theo hướng sản xuất sạch hơn, thân thiện với môi trường.", "evidence": ""}\n{"id": "B-56751234aa2257b5", "claim": "Sử dụng nước thải sau xử lý đạt tiêu chuẩn để phục vụ mục đích tưới cây và vệ sinh khuôn viên: khoảng 2% lượng nước đầu vào.", "evidence": ""}\n{"id": "B-4e7aa30250058035", "claim": "Bảo Việt là doanh nghiệp Việt đầu tiên 4 năm liên tiếp được công nhận và xếp hạng doanh nghiệp bền vững trong nhóm Chỉ số Dow Jones Sustainability Indices - DJSI, cũng như được công nhận trong Bảng xếp hạng tính bền vững doanh nghiệp toàn cầu (CSA) do S&P Global thực hiện.", "evidence": ""}\n{"id": "B-d2c1ce4ffefdc181", "claim": "nước thải sản xuất phát sinh khoảng 50~60 m3/ngày x đêm.", "evidence": ""}\n{"id": "B-4a5a1ec671b18b74", "claim": "Đồng hành cùng mục tiêu quốc gia về ứng phó với biến đổi khí hậu và hướng tới phát thải ròng bằng 0, DHG Pharma đã và đang triển khai nhiều giải", "evidence": ""}\n{"id": "B-8a12299d98291537", "claim": "Trong bối cảnh năng suất tăng hơn 60%, áp lực đảm bảo chỉ tiêu cường độ tiêu hao năng lượng là Dù chưa được như kỳ vọng nhưng mức giảm tiêu thụ gas 1,4% so với năm 2021 là kết quả đáng ghi nhận.", "evidence": ""}\n{"id": "A-e509bd8f7749a39e", "claim": "Coteccons hướng tới mục tiêu trở thành và được công nhận là Industry Leader, là hình mẫu trong cam kết thực hành ESG trong lĩnh vực Xây dựng, mang đến các giá trị bền vững đóng góp vào sự phát triển kinh tế, xã hội và bảo vệ môi trường bằng cách kinh doanh có đạo đức và trách nhiệm thông qua Quản trị Tốt.", "evidence": "Kết quả tăng trưởng Kinh tế Phần VI Thực hành ESG theo 5 mục tiêu của Coteccons Coteccons nhất quán với định hướng phát triển bền vững, phát triển có trách nhiệm bằng cách thực thi cụ thể các nhiệm vụ với môi trường, xã hội và con người xuyên suốt tiến trình thực hiện hoạt động kinh doanh. Chiến lược ESG tại Coteccons được thực hiện từ cấp lãnh đạo cao nhất. Chúng tôi nhận thức sâu sắc vai trò và khả năng ảnh hưởng của mình. Do đó, Coteccons muốn là và được công nhận là industry leader - thủ lĩnh trong ngành với mục tiêu thúc đẩy nâng cao tiêu chuẩn của toàn ngành và đóng góp tích cực hơn cho kinh tế - xã hội. Để hiện thực hóa sự công nhận này, trước tiên, chúng tôi cần trở thành tác nhân tạ"}\n{"id": "A-8394f8db5f4b4d4c", "claim": "We have made great progress in ensuring that our primary packaging materials are recyclable, supporting our efforts to reduce greenhouse gas emissions and minimize environmental impacts.", "evidence": "GRI 2-28] Pillar Our Commitment Our Strategy Continuously seeking innovative solutions and initiatives to minimize environmental impact while optimizing the use of resources, raw materials, packaging, energy, and water. Fully complying with environmental protection laws and actively addressing climate change by reducing greenhouse gas emissions At SABECO, we acknowledge that climate change is a critical global challenge affecting ecosystems and communities. We are committed to minimizing our environmental impact and fostering a sustainable future. ENABLING THE ORGANIZATION PERFORMANCE Our goal is to achieve net-zero Greenhouse gases (GHG) emissions by 2050."}\n{"id": "A-a4bcccb23ce0dfb6", "claim": "We are committed to ensuring that all emission treatment systems at production facilities operate effectively and strictly comply with environmental standards and regulations.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Manufacturing and processing facilities We are committed to ensuring that all PAN factories and production facilities strictly comply with environmental protection and discharge regulations. Wastewater treatment systems are modernly invested, operated stably, and regularly inspected and maintained. Treated wastewater must meet strict quality standards, ensuring no pollution of water sources, protecting ecosystems, and safeguarding public health. We commit to transparency in disclosing wastewater quality information and are ready to cooperate with regulatory authorities for monitoring and evaluation. OF FACTORIES AND PROCESSING FACI"}\n{"id": "B-830cc9c896eea804", "claim": "Đầu năm 2023, Vinamilk đã khởi động trồng hơn 1.000 cây xanh thuộc Hoạt đồng trồng cây Đây là hoạt động hợp tác với Bộ Tài nguyên và Môi trường kéo dài trong 5 năm (2023-2027) với ngân sách 15 tỷ đồng.", "evidence": ""}\n{"id": "A-ff5ad5035d639719", "claim": "Hợp đồng rác thải tái chế ký với Công ty CP xử lý, tái chế chất thải công nghiệp Hòa Bình.", "evidence": "2 Nguyên phụ liệu thuốc thành phẩm hủy 29,375kg Thiêu đốt Công ty CP xử lý, tái chế chất thải công nghiệp Hòa Bình 3 Chất thải nguy hại 638,5kg Làm sạch, Thiêu đốt, chôn lấp. Hợp đồng rác thải sinh hoạt ký với công ty CP xử lý chất thải xây dựng và đầu tư phát triển môi trường Hà Nội. Biện pháp quản lý chất thải nguy hại: Công ty đã thực hiện lập báo cáo Sở tài nguyên môi trường Hà Nội và được cấp Giấy phép môi trường số 102/ GPMT- UBND ngày 05/7/2023. Thường xuyên thu gom, phân loại, lưu giữ chất thải trong các thùng chứa riêng biệt, có nhận biết mã CTNH"}\n{"id": "A-5e83848e77f27675", "claim": "Wastewater treatment systems are modernly invested, operated stably, and regularly inspected and maintained.", "evidence": "In addition, these systems are regularly inspected, maintained, and upgraded to ensure stable and efficient operation. The total biomass consumption (rice husks and corn cobs) at Vinaseed was approximately TONS Emission monitoring results at selected factories (mg/Nm3) Treated wastewater must meet strict quality standards, ensuring no pollution of water sources, protecting ecosystems, and safeguarding public health. We commit to transparency in disclosing wastewater quality information and are ready to cooperate with regulatory authorities for monitoring and evaluation. OF FACTORIES AND PROCESSING FACILITIES have wastewater treatment systems that meet standards and comply with the issued Env"}\n{"id": "A-94b4949361890bd3", "claim": "Ứng dụng công nghệ tiết kiệm năng lượng, sử dụng các thiết bị hiện đại và hiệu suất cao.", "evidence": "TRÁCH NHIỆM VỚI MÔI TRƯỜNG GIẢI PHÁP TIẾT KIỆM ĐIỆN NĂNG Tiết kiệm năng lượng đóng vai trò vô cùng quan trọng trong chiến lược phát triển bền vững của các doanh nghiệp vận hành các nhà máy sản xuất quy mô lớn. Đây không chỉ là giải pháp cấp thiết nhằm tối ưu hóa chi phí vận hành, nâng cao năng lực cạnh tranh mà còn thể hiện tầm nhìn dài hạn trong quản trị và sử dụng nguồn tài nguyên hiệu quả. 2 Sử dụng thiết bị làm mát tiết kiệm năng lượng, hệ thống giám sát, cảnh báo nhiệt độ, độ ẩm tự động và trực tuyến. Đào tạo nhân viên vận hành kho về quy trình bảo quản và ứng phó khi có sự cố. Nâng cấp, đầu tư hệ thống xử lý nước thải, khí thải hiện đại, đảm bảo tuân thủ nghiêm ngặt các tiêu chuẩn môi "}\n{"id": "B-68c0d3356f1fb140", "claim": "Hazardous wastesare collected and stored separately to be compliant with law.", "evidence": ""}\n{"id": "A-30213ac412a2ca5d", "claim": "Our “Go Global – Go Green” strategy continues to serve as our North Star, aimed at elevating the value and position of Vietnamese agricultural products in the global market, embodying the spirit of “embracing difficulties, not avoiding them.” 2025 also marked a year where PAN focused on consolidating internal strength and adapting our models to the new landscape.", "evidence": "TABLE SUSTAINABILITY REPORT 2025 2 | | In 2025, Vietnamese enterprises in general, and The PAN Group in particular, faced a convergence of multifaceted challenges: volatility in U.S. tariff policies, the increasingly extreme impacts of natural disasters and floods, and escalating global demands for quality, environmental stewardship, and social responsibility. These “giant waves” have become an unavoidable reality. With the theme “Steadfast Heart – Bold Step,” The PAN Group’s 2025 Sustainability Report demonstrates a long-term vision - remaining resolute in our commitment to sustainable development despite short-term fluctuations. Our “Go Global – Go Green” strategy continues to serve as our"}\n{"id": "A-63b7d11de34b698b", "claim": "Consumer Health and Safety Water Stewardship Customer Relationship Management Talent Attraction and Development The process to determine material topics is broken down into 5 key steps:", "evidence": "Development and Ethics \\u0017 Occupational Health \\u0017 Supply Chain and Safety Management \\u0017 Water Stewardship \\u0017 Customer Relationship \\u0017 Complying with current Talent Attraction and Development CUSTOMER RELATIONSHIP MANAGEMENT Customer Privacy New employee hires and employee turnover Talent Attraction and Development GRI 401 –"}\n{"id": "B-752eb0e07be3dea9", "claim": "Theo đó, với cam kết cứ mỗi 5km chạy được hoàn thành, Coteccons sẽ thay mỗi vận động viên góp thêm 1 cây xanh.", "evidence": ""}\n{"id": "B-f4fff8163562a169", "claim": "Đối với chất thải rắn thông thường, nhà máy thực hiện phân loại tại nguồn thành các nhóm như chất thải hữu cơ, chất thải tái chế và chất thải không tái chế.", "evidence": ""}\n{"id": "A-252f0865f2e9f714", "claim": "4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas 1 2 3 4 The SRMC is comprised of the following members:", "evidence": "1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas CUSTOMER RELATIONSHIP 1 3. 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas SUSTAINABILITY GOVERNANCE SABECO’s corporate governance is vital to its sustainability efforts."}\n{"id": "A-b54b39cd589f4915", "claim": "Trong năm qua, Coteccons tích cực tổ chức và tham gia nhiều chương trình, sự kiện, gặp gỡ để thúc đẩy tinh thần hợp tác và chia sẻ về phát triển bền vững với đối tác, khách hàng và cộng đồng.", "evidence": "Đồng thời, Công ty cũng nhận thấy tầm quan trọng của việc truyền thông về ESG, lan tỏa tinh thần và nhận thức về phát triển xanh, thúc đẩy một cách thiết thực thực hành ESG trong cộng đồng. Sự tham gia của các bên liên quan và cộng đồng là chìa khóa biến phát triển bền vững thành hiện thực trên quy mô lớn và hiệu quả sâu. Do vậy, Coteccons xem đây là nhiệm vụ thúc đẩy tinh thần ĐỒNG CHÍ HƯỚNG và CÁC BÊN LIÊN QUAN Cuối năm 2023, Coteccons và Unicons đã tổ chức 2 sự kiện “Hội nghị đối tác, nhà thầu phụ” khu vực phía Nam và phía Bắc. Với chủ đề “Chung tay cùng thịnh vượng”, đây là dịp để Ban lãnh đạo Coteccons chia sẻ với hơn 200 đối tác về chiến lược, tinh thần hợp tác và định hướng phát triển"}\n{"id": "B-7bae1fe7cdef2a0c", "claim": "Lượng phát thải khí nhà kính từ hoạt động này ghi nhận 58,72 tấn CO2, tăng so với mức 45,95 tấn CO2 năm 2024, chủ yếu do nhu cầu mở rộng hoạt động kinh doanh và gia tăng tần suất di chuyển phục vụ công tác điều hành và phát triển thị trường.", "evidence": ""}\n{"id": "B-2c582453c837eeb6", "claim": "Thực hiện chương trình phát triển rừng bền vững của Công ty.", "evidence": ""}\n{"id": "B-aec7d3fba37e31cc", "claim": "Consequently, PNJ has implemented energy-saving policies that encompass awareness, action, technology, and In 2021, although PNJ\'s total output increased by 20% to meet the increased market demand, the consumption of gas and gasoline decreased by 39% and 14%", "evidence": ""}\n{"id": "A-ab54c7342b5b28e1", "claim": "Hòa Phát hướng tới duy trì năng lực cạnh tranh bền vững trong quá trình chuyển đổi hướng tới trung hòa carbon, đồng thời tiếp tục khẳng định cam kết về chất lượng sản phẩm dẫn", "evidence": "Bên cạnh đó, Công ty Cổ phần Thép Hòa Phát Dung Quất đã ban hành Sổ tay Quản lý Năng lượng, quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng hợp với định hướng phát triển xanh của Chính phủ Việt Nam và cam kết trung hòa carbon đến năm 2050. Chúng tôi tin rằng với tinh thần sáng tạo, quyết tâm và trách nhiệm, Hòa Phát sẽ không chỉ duy trì vị thế tiên phong trong ngành công nghiệp thép, mà còn đóng"}\n{"id": "A-fe5f0f2fbafd6b5c", "claim": "Nhằm mục tiêu phát triển bền vững và chủ động trước mọi biến động trong môi trường kinh doanh, định kỳ công ty xây dựng kế hoạch để tiến hành rà soát và cập nhật, đánh giá mức độ trưởng thành của hệ thống.", "evidence": "Từ đó, các mục tiêu cụ thể và kế hoạch hành động được xây dựng để hiện thực hóa tầm nhìn này. Đồng thời, HĐQT giám sát chặt chẽ quá trình thực thi, đảm bảo hoạt động tuân thủ luật pháp, Điều lệ và các quy định nội bộ của DHG Pharma. Bước 01 THỰC HIỆN MỤC TIÊU ĐỊNH HƯỚNG Báo cáo phát triển bền vững được lập định kỳ nhằm đánh giá và công khai đầy đủ thông tin về kết quả hoạt động của Công ty dựa trên 3 yếu tố nền tảng: Phát triển cộng đồng Xây dựng hệ thống quản lý rủi ro toàn diện, tích hợp các rủi ro ESG vào quy trình chung. Chủ động nhận diện, đánh giá và xây dựng kế hoạch ứng phó, đảm bảo hoạt động kinh doanh liên tục và bền vững. Quản lý rủi ro DHG Pharma 2 6 1 3 4 Phát triển bền vững"}\n{"id": "B-4b0839ca34a8d6ac", "claim": "Tổng Công ty cũng đã ký thỏa thuận hợp tác với Tập đoàn Vingroup để cung cấp trạm sạc (PV Power hiện đang đàm phán với V-Green về cung cấp thiết bị trạm sạc), lắp đặt điện mặt trời tại các dự án và bán điện sạch cho Vingroup.", "evidence": ""}\n{"id": "B-5f0c7ebfa2aa5551", "claim": "LONG AN HALF MARATHON 2024 Dự án thể thao này còn mang sứ mệnh lớn là lan tỏa rộng rãi ra cộng đồng tinh thần sống hướng về môi trường, thông điệp về phát triển bền vững và gây quỹ cho các chương trình trồng rừng hưởng ứng dự án BÁO CÁO PHÁT TRIỂN BỀN VỮNG theo 5 mục tiêu của Coteccons", "evidence": ""}\n{"id": "B-74398c48aea9f87e", "claim": "Triển khai chương trình phát triển rừng bền vững của Công ty.", "evidence": ""}\n{"id": "A-5fcede451ecd830e", "claim": "Tại PNJ, xuất phát từ triết lý phát triển bền vững: “Đặt lợi ích khách hàng, xã hội vào lợi ích doanh nghiệp”, đến nay chiến lược phát triển bền vững được thực hiện đồng bộ cả trong và ngoài doanh nghiệp với các mục tiêu bám sát quan điểm chỉ đạo của Chính phủ qua Nghị quyết 136/ NQ-CP về phát triển bền vững ban hành ngày 25 tháng 09 năm 2020, đó cũng là nội dung cụ thể của 17 Mục tiêu Phát triển Bền vững (SDGs) của Liên Hiệp Quốc.", "evidence": "MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA PNJ Tuyên bố về ESG: “PNJ hành động có trách nhiệm để tôn vinh vẻ đẹp của con người và cuộc sống một cách bền vững”. THẤP DIỄN GIẢI TẠI PNJ Xuất phát từ triết lý phát triển bền vững đặt lợi ích của khách hàng và xã hội vào lợi ích của doanh nghiệp, PNJG luôn đặt mục tiêu cân bằng lợi ích các bên liên quan thông QUY TRÌNH qua việc thực hiện hoạt động quản trị công ty một cách vững mạnh, không ngừng chọn lọc, áp dụng các thông lệ tốt trong khu vực và quốc tế. Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát, đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được tích hợp cùng các mục tiêu phát triển bền vững."}\n{"id": "B-6e2a279573ee4ff1", "claim": "4,99% Lượng nước trong hoạt động sản xuất được tận thu tuần hoàn và tái sử dụng.", "evidence": ""}\n{"id": "A-1f6055afa1899b4b", "claim": "Năm 2024, DHG Pharma đã đầu tư lắp đặt hệ thống điện mặt trời áp mái cho các khu vực nhà xưởng, góp phần giảm thiểu đáng kể tiêu hao năng lượng, hướng đến mục tiêu sản xuất xanh và nghiêm túc thực thi các chiến lược phát triển bền vững mạnh mẽ hơn.", "evidence": "Nguồn năng lượng sạch từ hệ thống điện mặt trời hiện đang được sử dụng trực tiếp cho hoạt động sản xuất tại nhà máy, góp phần tối ưu chi phí vận hành, giảm đáng kể lượng khí nhà kính phát thải ra môi trường, đồng thời thể hiện cam kết của DHG Pharma trong việc thúc đẩy phát triển xanh và bền vững. Song song với các giải pháp tiết kiệm năng lượng được áp dụng nhiều năm qua như: truyền thông nội bộ để nâng cao ý thức sử dụng điện hiệu quả; Tiếp tục đầu tư lắp đặt hệ Sử dụng điện mặt trời góp phần giảm 100% thành phẩm được kiểm nghiệm tại phòng đạt chuẩn WHO-GMP và thống điện năng lượng 1.907 tấn CO2 HÀNH ĐỘNG ỨNG PHÓ VỚI BIẾN ĐỔI"}\n{"id": "A-36a5aa6f4a86e076", "claim": "At the same time, PAN implements training programs and works closely with agents and farming households to enhance sustainable aquaculture practices, ensure product quality, and strengthen transparency across the supply chain.", "evidence": "30% of shrimp raw material areas and 100% of pangasius raw material areas. The remaining shrimp supply is sourced from farming households under strict evaluation and control criteria aligned with sustainable aquaculture practices, ensuring a stable and sustainable input supply. PAN’s member companies, such as Fimex VN and Aquatex Bentre, actively participate in certification programs, research initiatives, and sustainable aquaculture practices, including: Ï Training Linked to Practical Operations Strengthening training programs at factories, farming areas, and cultivation regions, focusing on safe production processes, quality control, and resource optimization."}\n{"id": "A-54f70fc12ef356c5", "claim": "cường hệ thống kiểm soát nội bộ và hỗ trợ sự phát triển bền vững của Tập đoàn.", "evidence": "độc lập về công tác quản trị doanh nghiệp và kiểm soát rủi ro. • Chuẩn hóa quy trình và chính sách nhằm tăng cường kiểm soát nội bộ và quản lý rủi ro tại các công ty thành viên. • Tạo điều kiện để Ban Kiểm soát hoạt động hiệu quả, phù hợp với chức năng và nhiệm vụ được giao. • Tham mưu, đề xuất việc lựa chọn đơn vị kiểm toán có năng lực và uy tín cho Tập đoàn và các công ty con. BAN KIỂM SOÁT Tập đoàn Hòa Phát đã thành lập Ban Kiểm soát – cơ quan của Đại hội đồng cổ đông, đóng vai trò quan trọng trong việc đảm bảo tính minh bạch, trách nhiệm giải trình và quản trị lành mạnh trong toàn Tập đoàn. Các nhiệm vụ chính của Ban Kiểm soát bao gồm: 75 QUẢN TRỊ DOANH NGHIỆP VÀ QUẢN LÝ RỦI RO 7.1 PHÒNG"}\n{"id": "B-43830478f8e57f45", "claim": "Quản lý và tối ưu hóa năng lượng: Ứng dụng các giải pháp tiết kiệm năng lượng trong hệ thống chiếu sáng, hệ thống sưởi, thông gió và điều hòa không khí (HVAC), cũng như trong dây chuyền sản xuất.", "evidence": ""}\n{"id": "A-8eb13f454ebc4089", "claim": "Đồng thời, Traphaco được tôn vinh là doanh nghiệp niêm yết tiêu biểu nhân kỷ niệm 25 năm Thị trường Chứng khoán Việt Nam, thể hiện cam kết về minh bạch, quản trị tốt và tạo giá trị bền vững cho cổ đông.", "evidence": "Việt Nam. Trong khuôn khổ buổi lễ, Traphaco được HOSE trang trọng vinh danh như một doanh nghiệp tiêu biểu đã đồng hành và đóng góp tích cực cho sự trưởng thành của thị trường trong nhiều năm qua. Đại diện Traphaco đã đón nhận Kỷ niệm chương 25 năm, thể hiện sự ghi nhận của HOSE đối với vai trò và nỗ lực bền bỉ của doanh nghiệp. Traphaco được vinh danh nhân kỷ niệm 25 năm thị trường chứng khoán Việt Nam Chuỗi hội nghị khách hàng năm 2025 Ngày 3/12/2025, tại Khách sạn Sheraton Sài Gòn, Sở Giao dịch Chứng khoán TP.HCM (HOSE) đã tổ chức Hội nghị Doanh nghiệp niêm yết và Lễ trao giải Cuộc bình chọn Doanh nghiệp niêm yết năm 2025 - sự kiện quan trọng đánh dấu cột mốc 25 năm hình thành và phát tri"}\n{"id": "B-295888715cc4a0d7", "claim": "cử, đối với thị trường New Zealand, Vinamilk đang thực hiện cung cấp sản phẩm với những yêu cầu về “tính xanh, bền vững” như không có ống hút nhựa, nắp nhựa theo thỏa thuận giảm rác thải nhựa ra môi trường.", "evidence": ""}\n{"id": "A-b7d10528c4818f45", "claim": "Với nền tảng đã được củng cố, với chiến lược đã được định hình, với những động lực tăng trưởng mới đang ngày càng rõ nét, chúng ta có đầy đủ cơ sở để tin tưởng rằng: Traphaco sẽ tiếp tục đột phá, phát triển nhanh hơn và tạo ra nhiều giá trị bền vững hơn cho cổ đông, nhà đầu tư, người lao động và cộng đồng trong giai đoạn mới.", "evidence": "nhiều nhóm sản phẩm chiến lược và các kênh bán hàng hiện đại tăng trưởng tích cực; năng lực nội tại tiếp tục được củng cố trên các phương diện nghiên cứu phát triển, quản lý chất lượng, chuỗi cung ứng và tổ chức vận hành. Điều này đã tạo dựng nền tảng quan trọng để Traphaco bước vào giai đoạn phát triển mới với mô hình hoạt động tinh gọn hơn, năng lực cạnh tranh cao hơn và dư địa tăng trưởng bền vững hơn trong các năm tiếp theo. Đây là năm cuối của nhiệm kỳ 2021-2025, một giai đoạn nhiều biến động nhưng rất đáng tự hào, khi Traphaco đã giữ vững nội lực, củng cố nền tảng và từng bước hình thành những động lực tăng trưởng mới cho tương lai. Quan trọng hơn, nhiệm kỳ vừa qua đã đánh dấu sự hình "}\n{"id": "A-5fc6c3f2ad459f84", "claim": "Trong đó, Công ty công bố đầy đủ các mục tiêu phát triển bền vững kèm theo chỉ tiêu định lượng và kết quả thực hiện cụ thể trong kỳ.", "evidence": "Báo cáo Phát triển bền vững năm 2024 được trình bày và công bố các thông tin chính thức liên quan đến định hướng chiến lược, mục tiêu và kết quả triển khai các chương trình hành động, sáng kiến phát triển bền vững của DHG Pharma đã và đang thực hiện trong năm 2024, làm cơ sở để đánh giá hiệu quả trong công tác thực hiện phát triển bền vững cho những năm tiếp theo. Quy trình quản trị bền vững tại DHG Pharma được vận hành qua 5 bước: Bước 04 Mỗi CBNV là mắt xích quan trọng, trực tiếp thực hiện các nhiệm vụ cụ thể hàng ngày nhằm hiện thực hóa các mục tiêu phát triển bền vững của Công ty. CBNV cam kết thực hiện các mục tiêu phát triển bền vững đúng với định hướng được xây dựng và ban hành tại"}\n{"id": "B-79f5d41ece1c01e3", "claim": "Gắn kết phong trào 5S với môi trường xanh.", "evidence": ""}\n{"id": "A-1f8b3ec2bd95bda5", "claim": "Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt khe và có trách nhiệm để lựa chọn những nguyên vật liệu ít tác động xấu đến môi trường, có khả năng tái chế hoặc nguyên vật liệu tái chế, tiết kiệm được năng lượng và tài nguyên từ nguồn gốc sản xuất và trong quá trình sử dụng.", "evidence": "TABLE Phạm Xuân Thu - Giám đốc Khối Thu mua “Chúng tôi nhận thức được việc chọn lựa vật liệu trong xây dựng vô cùng quan trọng, vì vật liệu sẽ có tác động trực tiếp đến môi trường trong suốt vòng đời của một công trình. Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt khe và có trách nhiệm để lựa chọn những nguyên vật liệu ít tác động xấu đến môi trường, có khả năng tái chế hoặc nguyên vật liệu tái chế, tiết kiệm được năng lượng và tài nguyên từ nguồn gốc sản xuất và trong quá trình sử dụng. Coteccons cũng ưu tiên nguyên vật liệu địa phương, vật liệu có độ bền cao, vật liệu xanh từ công nghệ mới... Chúng tôi đã có cơ hội gặp gỡ những đối tác chuyên nghiệp với những sản phẩm, giải pháp t"}\n{"id": "A-825d4be195f4826f", "claim": "Kết quả phản ánh nỗ lực đồng bộ trong điều hành, tối ưu hiệu quả và kiểm soát chi phí để tăng trưởng bền vững.", "evidence": "(năm 2025: 24,83 ngày), qua đó khẳng định năng lực quản trị tài sản ngắn hạn hiệu quả và đảm bảo dòng tiền phục vụ hoạt động kinh doanh ổn định. Công ty đang chủ động kiểm soát và tối ưu hiệu quả vận hành, hướng tới mục tiêu tăng trưởng bền vững. Đối với hàng tồn kho, Công ty tiếp tục tối ưu hóa cơ cấu tồn kho và nâng cao khả năng đáp ứng thị trường, thể hiện qua việc vòng quay tồn kho duy trì ổn định và số ngày tồn kho được kiểm soát ở mức hợp lý, đảm bảo cân đối giữa dự trữ an toàn và hiệu quả sử dụng vốn. Thành công này bắt nguồn từ việc tinh gọn hóa quy trình sản xuất, tối ưu hóa chuỗi cung ứng nguyên liệu đầu vào và nâng cao năng suất lao động trên toàn hệ thống. Tổng 2.257 2.654 2.633 "}\n{"id": "B-b65920c85c22730e", "claim": "Với kết quả đạt được nêu trên năm 2025 PV Power đã vinh dự được nhận các danh hiệu, giải thưởng: Được Thủ tướng Chính phủ/ Bộ Tài chính/ Tập đoàn tặng bằng khen/ giấy khen trong công tác Đầu tư Xây dựng Nhà máy điện Nhơn Trạch 3 và Nhơn Trạch 4 cho Tổng công ty/ các Ban chức năng và tập thể/cá nhân có thành tích xuất sắc trong quá trình đầu tư xây dựng Lần thứ 6 liên tiếp được vinh danh trong Top 100 Doanh nghiệp bền vững Việt Nam.", "evidence": ""}\n{"id": "A-a891f683e94e8dfa", "claim": "Với kinh nghiệm chuyên môn sâu rộng trong lĩnh vực kiểm nghiệm, nghiên cứu phát triển sản phẩm, quản lý chất lượng và điều hành sản xuất, bà đã có những đóng góp tích cực trong việc nâng cao tiêu chuẩn chất lượng sản phẩm và tối ưu hoá quy trình sản xuất, góp phần đáng kể vào sự phát triển bền vững và uy tín thương hiệu", "evidence": "Chất lượng Sản xuất Nội địa. Kinh nghiệm điều hành sâu rộng và kiến thức chuyên môn vững vàng của ông đã góp phần quan trọng trong việc nâng cao hiệu quả vận hành, đảm bảo chất lượng sản phẩm và thúc đẩy sự phát triển bền vững tại DHG Pharma. Bà sinh năm 1968, được bổ nhiệm giữ chức Phó Tổng Giám đốc CTCP Dược Hậu Giang từ tháng 05/2012. Với mục tiêu gia tăng doanh thu, mở rộng xuất khẩu góp phần gia tăng lợi nhuận công ty bằng các hoạt động đầu tư tập trung vào phát triển sản phẩm chất lượng cao, đáp ứng các tiêu chuẩn quốc tế, giúp tối ưu hóa chi phí và nâng cao hiệu quả hoạt động. Nâng cấp tiêu chuẩn GMP quốc tế và duy trì đáp ứng tiêu chuẩn trong nước cho các dây chuyền sản xuất: Đầu tư "}\n{"id": "A-77ce5b81d5d7d878", "claim": "Tổng quan báo cáo phát triển bền vững 154 Báo cáo của Ban Tổng Giám đốc 119 Định hướng chiến lược phát triển bền vững 155 Báo cáo kiểm toán độc lập 122 Quản trị phát triển bền vững", "evidence": "BỀN VỮNG PHÁT TRIỂN 5 118 Tổng quan báo cáo phát triển bền vững 119 Định hướng chiến lược phát triển bền vững 122 Quản trị phát triển bền vững 124 Gắn kết các bên liên quan TỔNG QUAN BÁO CÁO PHÁT TRIỂN BỀN VỮNG ĐỊNH HƯỚNG CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG NGUYÊN TẮC XÂY DỰNG BÁO CÁO MỤC TIÊU CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG DHG PHARMA HƯỚNG TỚI"}\n{"id": "B-e122238676c0cdd7", "claim": "Với chủ đề này, chúng tôi báo cáo trong phạm vi công Các nhà cung ứng đều tham gia vào chuỗi cung ứng xanh của Traphaco, là mắt xích đầu tiên \\"Nguyên liệu xanh\\" trong 4 giá trị xanh \\"Nguyên liệu - Công nghệ - Sản phẩm - Dịch vụ\\" của công ty.", "evidence": ""}\n{"id": "A-42caab287b97ba2a", "claim": "Bên cạnh trồng rừng đặc dụng, Coteccons còn tham gia hoạt động hỗ trợ sinh kế cho người dân địa phương nhằm góp phần gia tăng hành lang bảo vệ voọc, hạn chế tình trạng lấn chiếm đất rừng, khai thác gỗ rừng và tài nguyên rừng trái phép hoặc chăn thả gia súc tại những nơi đàn voọc sinh sống.", "evidence": "Đến tháng 06/2024, bà con đã có đợt thu hoạch đầu tiên và được doanh nghiệp địa phương bao tiêu đầu ra. Cùng với chương trình tham gia trồng rừng đặc dụng bảo vệ môi trường sinh trưởng của loài voọc tại Quảng Bình, Coteccons cũng tham gia hoạt động hỗ trợ sinh kế cho người dân địa phương. Đây là hoạt động song hành nhằm góp phần gia tăng hành lang bảo vệ cho voọc, hạn chế nguy cơ bị đe dọa do các hoạt động khai thác kinh tế và mưu sinh của người dân. Tỉnh Quảng Bình đã có chính sách tạo điều kiện, khuyến khích và hỗ trợ những hộ gia đình canh tác nương rẫy gần Khu Bảo tồn chuyển dịch sang trồng các loài cây lâu năm."}\n{"id": "B-4636db4a0629d967", "claim": "Qua 40 năm lao động vinh quang, TRAPHACO đã tạo nên một con đường, đưa con người trở về với cuộc sống Xanh thông qua việc nghiên cứu cỏ cây thiên nhiên để tạo ra những sản phẩm chăm sóc cho sức khỏe cộng đồng.", "evidence": ""}\n{"id": "B-d44fa5ca576bf02a", "claim": "By setting out to meet Environmental-Social-Governance (ESG) targets, we are committed to acting beyond simply environmental stewardship but are also taking greater social and governance accountability.", "evidence": ""}\n{"id": "A-76b98deabc362764", "claim": "Với kinh nghiệm chuyên sâu trong lĩnh vực tài chính, kiểm toán, chuyển đổi mô hình quản trị hiệu quả, quản trị nhân sự và huấn luyện cán bộ lãnh đạo/ quản lý của ông đã đóng góp tích cực trong việc củng cố và nâng cao hiệu quả quản trị, đảm bảo an toàn hoạt động và thúc đẩy sự phát triển bền vững.", "evidence": "Chương trình kỹ năng cơ bản Phát triển kiến thức chuyên môn cho các phòng ban, giúp nhân viên trở thành chuyên gia trong lĩnh vực của mình. Chương trình chuyên sâu PHÁT TRIỂN NĂNG LỰC NGUỒN NHÂN LỰC Tập trung vào việc phát triển kỹ năng lãnh đạo và quản lý như quản trị sự thay đổi, lập kế hoạch và tổ chức công việc hiệu quả, thuyết trình, đàm phán, huấn luyện và kèm cặp, quản lý con người và tạo động lực. Với hơn 30 năm công tác và gắn bó lâu dài tại DHG Pharma, bà đã trải qua nhiều vị trí quản lý quan trọng như Phó phòng Nghiên Cứu, Trưởng phòng Kiểm nghiệm, Giám Đốc sản xuất và Giám đốc Chất lượng tại DHG Pharma. Với kinh nghiệm chuyên môn sâu rộng trong lĩnh vực kiểm nghiệm, nghiên cứu ph"}\n{"id": "A-22d0c372f5f271ed", "claim": "The area of high-quality, low-emission rice models in the Mekong Delta was expanded to 47,520 hectares—an increase of 47 times compared to 2024.", "evidence": "program across 10 member companies. Ï Increased the cultivated area of Ï Initiated the “Life form Forestry” Development Subcommittee (now representatives of three key business high-quality, low-emission rice forty-sevenfold to 47,520 hectares in the Mekong Delta region. Project, aiming to plant 1 million trees by 2030. Vinaseed – a member of PAN Group – continued to expand its low-emission rice cultivation area to over 47.520 hectares (20 times higher than in 2024). This contributes to the project “Sustainable development of one million hectares of high-quality, low-emission rice cultivation associated with green growth in the Mekong"}\n{"id": "B-772a4c80812604c3", "claim": "Nguồn nước sử dụng tại DHG Pharma được cung cấp bởi Công ty Cổ phần Cấp nước Cần Thơ 2 và Công ty TNHH MTV Cấp nước Hậu Giang.", "evidence": ""}\n{"id": "A-d339a40785c7a10a", "claim": "Thêm nữa, việc phủ xanh văn phòng và sử dụng dịch vụ Taxi Xanh đã góp phần thể hiện cam kết của chúng ta với một tương lai xanh hơn.", "evidence": "Coteccons liên kết với Taxi Xanh để chuyển đổi từ phương tiện taxi truyền thống sử dụng xe động cơ xăng sang loại hình taxi từ xe động cơ điện của Vinfast. Với nhu cầu đi lại của lượng lớn nhân sự, Coteccons đánh giá việc góp phần giảm khí thải từ việc ưu tiên sử dụng xe động cơ điện là cần thiết. Thông qua hợp tác với Taxi Xanh, người của Coteccons sẽ chuyển đổi nhu cầu đi lại qua phương tiện này. Kết quả kinh doanh mang tính bứt phá trong năm tài chính vừa qua chính là minh chứng thực tế rằng chúng tôi đã đi đúng hướng. Hơn nữa, không chỉ thành tựu tài chính, Coteccons đã thực sự kiến tạo một văn hóa và tinh thần mới. Người lao động của chúng tôi hạnh phúc hơn, khách hàng tín nhiệm chúng t"}\n{"id": "B-38fe4cebc23e748b", "claim": "assessment framework, implemented since 2023 to guide and promote the PAN’s performance showed in the “Top 100 Sustainable Companies in Vietnam” assessment and announcement program for the period 2023–2025, organized by VBCSD – VCCI.", "evidence": ""}\n{"id": "A-0db65a62a3557ba5", "claim": "Đây là biện pháp cốt lõi trong giảm phát thải trực tiếp tại khu vực lò cao và nâng cao hiệu suất luyện gang.", "evidence": "(Nguồn phát thải gián tiếp từ năng lượng nhập khẩu) để phục vụ quá trình sản xuất của Tập đoàn cũng được xem xét và trình bày trong Báo cáo này do hoạt động này cũng góp phần phản ánh một cách hiệu quả các kết quả mà Tập đoàn đã đạt được và các giải pháp mà Tập đoàn sẽ triển khai trong tương lai. Thu hồi nhiệt và khí thải từ các lò luyện cốc, lò cao và lò chuyển để phát điện và cải thiện hiệu suất sản xuất. Ứng dụng công nghệ dập coke khô, vừa tận dụng nhiệt năng để phát điện, vừa giảm thiểu tác động môi trường. Nâng cao trải nghiệm khách hàng Giảm tiêu hao tài nguyên và phát thải gián tiếp Tăng khả năng thích ứng với biến động thị trường Chiến lược này đồng thời đóng góp trực tiếp vào 3 trụ"}\n{"id": "A-a3152ca063d78443", "claim": "We are committed to minimizing our environmental impact and fostering a sustainable future.", "evidence": "We are committed to continuously improving our practices, measuring our impact and setting aspiring goals for the future. This report is just the beginning, and we look forward to sharing our progress in the years to come. Sustainability remains at the core of SABECO’s strategy, directly linked to our ability to operate responsibly and sustainably. (GRI). This demonstrates our increasing commitment to transparency and accountability in environmental, social and governance (ESG) practices. We believe that sustainability development is integral to our long-term success, and we are proud to take this important step in furthering our journey towards a more sustainable future. communities through"}\n{"id": "B-8541c6d4b5ff03ad", "claim": "NSC and SSC’s agricultural waste has been 100% used for sales, composted into fertilizers or burned, which reduces the burden of waste disposal and benefits the company.", "evidence": ""}\n{"id": "B-d6b4c0bdf9c777d7", "claim": "There was a slight increase of 3% in 2020 electricity consumption while the steam consumption maintained the same level as in 2019.", "evidence": ""}\n{"id": "B-2543069120946f8c", "claim": "Đầu tư hạ tầng chống ngập, hệ thống thoát nước, trữ nước và cấp nước dự phòng tại chỗ.", "evidence": ""}\n{"id": "B-3a00d29a15cacfb8", "claim": "Xây dựng hệ thống thu thập, kiểm kê dữ liệu phát thải khí nhà kính và tiêu thụ năng lượng theo GHG Protocol và PCAF;", "evidence": ""}\n{"id": "A-f7b4c930a3a6f965", "claim": "In terms of environmental performance, total emission reductions reached 78,485.62 tons of CO2 equivalent, as independently verified by Regrow Ag through the MRV system platform.", "evidence": "Natural Resources Ï 2.3% reduction in CO2 emission intensity Ï Energy consumption (tons CO2 per VND billion revenue) Ï Water usage Ï 6% reduction in water intensity (1,000 m3 Guided implementation of legal regulations related to environmental and social responsibility (EPR regulations, Environmental Law 2025, etc.). Evaluated the effectiveness of the internal governance system through compliance reports from the Company and its subsidiaries. RESULTS OF SUPERVISION OF THE CEO In 2024, the CEO demonstrated effective leadership in managing and directing the business operations of both the Company and the Group, ensuring the thorough execution of resolutions adopted by the"}\n{"id": "A-4ec208ffbb4020fa", "claim": "12% 25% 25% Tỷ lệ nước tuần hoàn để tái sử dụng so với năm trước đó Hiện tại, doanh nghiệp vẫn đang trong quá trình trang bị cho nhà máy mới để nâng cao hiệu quả sử dụng nguồn nước, công suất xử lý nước thải và đồng thời đưa ra những biện pháp sử dụng/tái chế hiệu quả lượng nước ở nhà máy này.", "evidence": "Là doanh nghiệp luôn coi trọng yếu tố môi trường trong sản xuất, PNJ vẫn cố gắng xử lý tối ưu nguồn nước đầu ra, đảm bảo tỷ lệ xử lý nước trước khi thải ra môi trường ở mức 87.6% so với 2021. Tổng lượng Nước trung bình Nước được tuần hoàn (m3) Sản lượng NĂM Hiểu rõ và trân trọng giá trị của tài nguyên nước, PNJ Để đạt được tỉ lệ như vậy, đối với sản xuất, nguồn nước luôn được tiết kiệm do PNJ sử dụng những máy móc và công nghệ hàng đầu của Ý, do đó, nhu cầu cũng như tiêu hao khi sử dụng nước thấp. Đối với nước sinh hoạt, PNJ luôn thực hiện tuyên truyền vai trò của nguồn nước và sử dụng nước tiết kiệm cho CBCNV toàn công ty, đồng thời, có những khẩu hiệu tiết kiệm nước ở các điểm sử dụng nước"}\n{"id": "A-70ee1d5569d44cea", "claim": "187 1.5 Warehouse staff Training to update regulations on forklift operation safety 205 1 Forklift driver, Warehouse Staff Near miss Training 39 1 Warehouse staff (1)The three groups are defined under Decree 113/2017/ND-CP, which provides detailed regulations and guidance on the implementation of certain provisions of Vietnam\'s Chemical Law.", "evidence": "TABLE | Managers, staff directly working with chemical belonging to three groups as defined by law1 | 299 | Fire Protection and Rescue | Fire and Rescue Team | 583 | 24 Occupational Safety and Hygiene | All employees, belonging to six groups as defined by law2 | 4,675 | 8 Radiation Safety | Managers, staff directly managing & operating equipment with radioactive sources | 68 | 24 Food Safety | Production employees | 517 | 3 First Aid | First aid team | 970 | 4 Forklift Operation Safety Training - Beer Grade Lifting | Forklift Drivers | 122 | 2 Training on HSE Procedures, Regulations & Safety Supervision in Warehouse Operation | Warehouse staff | 187 | 1.5 Training to update regulations on fo"}\n{"id": "A-94778740bd6d94b1", "claim": "STT Tên chất thải nguy hại Mã số CTNH Năm 2024 (tấn/năm) Tổ chức tiếp nhận và xử lý CTNH % 6,3% 7,9% 14,9% 15,4% 2,3% xử lý (*) tổng lượng nước thải", "evidence": "TABLE STT | | Tên chất thải nguy hại | | Mã số CTNH | | Năm 2024 (tấn/năm) | | Phương pháp xử lý (*) | | Tổ chức tiếp nhận và xử lý CTNH TABLE Địa điểm | | Khối lượng năm 2024 (tấn/năm) | | Tổ chức tiếp nhận và xử lý"}\n{"id": "B-14159e5c9e5f846f", "claim": "Năm 2016, năm mở đầu cho kế hoạch 5 năm 2016 - 2020 Với những lợi thế Traphaco đang nắm giữ: Một chiến lược phát triển bền vững gắn hiệu quả kinh tế với trách nhiệm xã hội và bảo vệ môi trường;", "evidence": ""}\n{"id": "B-d8284b76031ccfff", "claim": "Ý thức được vai trò của mình trong công cuộc tiết kiệm năng lượng chung của cả nước, PNJP luôn chú trọng tối đa hóa hiệu năng sử dụng, tiết kiệm năng lượng, đồng thời góp phần bảo vệ môi trường và phát triển một cách bền vững.", "evidence": ""}\n{"id": "B-b3c30dfcbac71565", "claim": "Recover sparging water at the brew house which help save about 2,000 m3/year.", "evidence": ""}\n{"id": "B-5d74fc3fa628f4a0", "claim": "100% nước thải từ hoạt động sản xuất và sinh hoạt được xử lý đạt quy chuẩn trước khi xả thải vào nguồn tiếp nhận.", "evidence": ""}\n{"id": "A-542bab3f171afc95", "claim": "Human resource policies are implemented consistently across the system, aiming to create a safe, inclusive, and sustainable workplace.", "evidence": "Strategy The Group’s tax strategy is developed and implemented in alignment with its sustainable development orientation and good governance practices. Internal tax policies and guidelines are applied consistently across the system to ensure efficient, transparent, and compliant tax practices in accordance with both domestic and international standards. Enhancing and enforcing anti-corruption policies The company also safeguarded shareholder rights by ensuring access to information, responding to shareholder inquiries, and transparently engaging with investors. an inclusive, safe, and development- driven work environment. The Group also actively collaborates with cooperatives and farmers acr"}\n{"id": "A-926fcdf584f9756f", "claim": "SOCIAL Striving to create jobs nationwide, improve living conditions, and contribute to the creation of thriving sustainable communities through 6 key areas: public health, art and culture, community development, education, sports, and environmental sustainability.", "evidence": "TABLE Pillar ENVIRONMENT | Our Commitment Fully complying with environmental protection laws and actively addressing climate change by reducing greenhouse gas emissions | Our Strategy Continuously seeking innovative solutions and initiatives to minimize environmental impact while optimizing the use of resources, raw materials, packaging, energy, and water. SOCIAL | Human Capital Development: maintaining safe working conditions, taking care of physical health, mental health & welfare of employees, focusing on employee development and talent acquisition as well as respecting diversity in the workplace | WARM workforce strategy: \\u0017 Willing – Attracting and onboarding passionate talent \\u0017 Able – D"}\n{"id": "B-bd4d8e8128711547", "claim": "waste, there are contractors who deal with transport The system of collecting and wastewater treatment system has also been invested to generate electricity and make compost.", "evidence": ""}\n{"id": "A-b805bc73fd85554f", "claim": "The Net Zero strategy is implemented by PAN in a consistent and comprehensive manner across all production and business activities of its member companies, including those equitized from state-owned enterprises.", "evidence": "Comprehensive risk management: PAN’s risk management system is built on two platforms: (1) Adhering to international standards in the investment activities to meet shareholder requirements, and (2) Applying speci昀椀c standards of the agriculture and food industry to strictly control risks in the production and business chain. AQUACULTURE PACKAGED FOOD AGRICULTURE PAN orients its resources towards three key areas, where its member companies are all leading companies. Ï Released the first “Production Ï Adopted the Asian Development Ï Became a member of the development strategy integrated with core business activities, making it a consistent and overarching strategic direction. Sustainability Re"}\n{"id": "A-b3614aaf33efc1aa", "claim": "Nước chứa cặn bẩn, tạp chất trong Khu liên hợp được xử lý thông qua hệ thống xử lý nước trung tâm và tái chế 100%.", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 Tái chế, tái sử dụng chất thải rắn thành nguyên vật liệu quay lại quá trình sản xuất Các giải pháp tái chế, tái sử dụng nội bộ tập trung vào việc phân loại và xử lý các chất thải ngay tại nguồn phát sinh. Theo đó, các chất thải rắn chứa sắt sẽ được thu gom và xử lý, trở thành nguyên vật liệu đưa ngược lại quy trình sản xuất thép. Tối đa hóa tái chế/tái sử dụngchất thải phát sinh thành nguyên vật liệu đưa trở lại quá trình sản xuất hoặc chế biến, xử lý tạo ra các sản phẩm phụ có giá trị kinh tế ; nghiên cứu và tìm kiếm các giải pháp điều chỉnh cơ cấu tiêu thụ nguyên liệu và qua đó giảm phát sinh chất thải. Áp dụng giải pháp sử dụng nước hiệu quả, xây dựng các hệ thống t"}\n{"id": "A-7c6476273e26e837", "claim": "1 Thúc đẩy tăng trưởng kinh tế bền vững, hiệu quả và dài hạn;", "evidence": "Hệ thống Quản trị hiện đại dựa trên nền tảng công nghệ thông tin tích hợp. HIỆU QUẢ VÀ BỀN VỮNG TĂNG TRƯỞNG MỤC TIÊU 1 (SD8) Thúc đẩy tăng trưởng kinh tế bền vững, hiệu quả và dài hạn; SDG 1. Xóa đói giảm nghèo dưới mọi hình thức ở mọi nơi SDG 8. Thúc đẩy tăng trưởng kinh tế bền vững, hiệu quả và dài hạn; tạo công ăn việc làm và nâng cao năng suất lao động cho mọi người"}\n{"id": "B-3126990194391145", "claim": "Việc duy trì môi trường làm việc và đào tạo trực tiếp kết hợp online, kết hợp với số hóa tài liệu và chữ ký điện tử, cùng các giải pháp kiểm soát vận hành tòa nhà, đã góp phần giảm đáng kể Nhờ đó, tổng lượng điện tiêu thụ năm 2025 đạt khoảng 556.240 kWh, giảm 19% so với năm 2024, tương đương giảm phát thải khoảng 108 tấn CO2, cho thấy hiệu quả rõ rệt của các chương trình tiết kiệm năng", "evidence": ""}\n{"id": "A-ab6e951d474ffde6", "claim": "Đây là lần thứ 8 Traphaco nhận danh hiệu uy tín này, khẳng định vị thế tiên phong trong hành trình kinh doanh bền vững và đóng góp tích cực vào sự phát triển của xã hội.", "evidence": "Quá trình liên kết ko chỉ tạo việc làm cho các hộ gia đình mà còn góp phần thay đổi mối quan hệ trong gia đình họ một cách tích cực và bền vững, giúp cho họ có cuộc sống bình đẳng, có Chuỗi hoạt động cộng đồng xã hội năm 2025 tiếp tục khẳng định định hướng phát triển bền vững của Traphaco – nơi hiệu quả kinh doanh luôn song hành cùng trách nhiệm xã hội. Năm 2025 đánh dấu bước chuyển quan trọng của Traphaco trên hành trình nâng tầm danh mục sản phẩm và khẳng định vị thế doanh nghiệp dược phát triển dựa trên khoa học và chất lượng. Đây không chỉ là nền tảng cho tăng trưởng bền vững, mà còn là cam kết dài hạn của Traphaco đối với sức khỏe cộng đồng và niềm tin của người tiêu dùng Việt Nam. KHÔN"}\n{"id": "A-8fe3f2e6b707ff04", "claim": "\\u0017 Awarded the \\"Environmentally Friendly Product\\" certificate issued by the Vietnam Environment Industry Association.", "evidence": "TABLE SABECO\'S SUSTAINABILITY MILESTONES 2008 2013 2013-2015 2016 \\u0017 Through equitization, Saigon Beer \\u0017 The first brewery obtained \\u0017 Developed the Environmental \\u0017 Listed on the Ho Chi Minh Stock Exchange (HOSE). – Alcohol – Beverage Corporation certification under ISO 50001:2013 Management System in accordance \\u0017 Awarded the \\"Environmentally Friendly Product\\" (SABECO) was transformed into a for the Energy Management System. with ISO 14001 standards. certificate issued by the Vietnam Environment joint stock company. 2013 – 2015 Industry Association. \\u0017 SABECO was awarded the \\u0017 Began transitioning to biomass boilers to replace title of Labor Hero during the fossil fuel systems at selected brewer"}\n{"id": "B-e9a7cad76fdcde70", "claim": "Certificate of “Environmentally Friendly Products” granted by Vietnam Environmental Industry Association.", "evidence": ""}\n{"id": "A-7ce1d86d60dfa906", "claim": "\\u0017 Named among the Top 100 Sustainable Businesses in Vietnam 2023 by CSI100.", "evidence": "\\u0017 Signed a cooperation program with the Ministry of Culture, Sports and Tourism to elevate national sports and establish Vietnam’s tourism brand. 2023 2024 \\u0017 Named among the Top 100 Sustainable Businesses in Vietnam 2023 by CSI100. \\u0017 Named among the Top 100 Sustainable Businesses in Vietnam 2024 by CSI100. TOP 100 TOP 10 Sustainable Enterprises in Vietnam (Manufacturing) awarded by Vietnam Chamber of Commerce and Industry (VCCI) and Vietnam Business Council for Sustainable Development (VBCSD) – CSI100 Reference for 02 consecutive years (2023-2024)."}\n{"id": "B-4526a945fe696e11", "claim": "Issue the new policies and regulations on The PAN Group’s sustainable development strategies, the common development orientation of its member In particular, the group consistently applied environmentally friendly and high technology in all the agricultural production activities, especially applied to all new projects;", "evidence": ""}\n{"id": "B-7515f955a0976d87", "claim": "In 2023, MWG was honored in the \\"Top 100 Most Valuable Brands in Vietnam 2023,\\" \\"Top 10 Strongest Brands in Vietnam 2023,\\" \\"Top 10 Largest Private Enterprises in Vietnam,\\" \\"Top 10 Reputable Retail Companies in 2023,\\" \\"Top 100 Best Places to Work in Vietnam 2023,\\" and for the second consecutive year in the Top 50 Most Sustainable Development Companies in Vietnam and Top 20 Companies with the Highest VNSI Sustainable Development Index in Vietnam.", "evidence": ""}\n{"id": "A-8f7fea58d379d82e", "claim": "Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để triển khai tại các nhà máy.", "evidence": "Bên cạnh đó, Công ty Cổ phần Thép Hòa Phát Dung Quất đã ban hành Sổ tay Quản lý Năng lượng, quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tới chuyển dịch sang các nguồn năng lượng tái tạo. Bên cạnh đó. Tập đoàn Hòa Phát nghiên cứu triển khai lắp đặt hệ thống điện mặt trời trời áp mái nhằm cung cấp một phần nhu cầu điện năng của các nhà máy. LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi "}\n{"id": "B-66391196e0291a1e", "claim": "Để đảm bảo đạt chất lượng theo các tiêu chuẩn ngành, Traphaco không sử dụng các nguồn nguyên vật liệu tái chế để sản xuất dược phẩm và các sản phẩm chăm sóc sức Với thế mạnh về Đông dược, phần lớn nguyên liệu sản xuất thuốc của Công ty là dược liệu trồng trong nước (90%).", "evidence": ""}\n{"id": "B-05c1f0fcb01d789c", "claim": "The supply chain of clams is strictly supervised and certi昀椀ed for sustainability by MSC.", "evidence": ""}\n{"id": "A-328f8cd501a9b58f", "claim": "SABECO is committed to ensuring the well-being of all employees and fostering a culture of safety, prevention, and continuous improvement.", "evidence": "Active participation of employees in training programs and feedback mechanisms to foster a strong safety culture. Personal Protective Equipment (PPE): Fire prevention: Implementation of fire prevention systems and regular drills to enhance workplace safety. Fostering resilience: Demonstrating corporate citizenship: Actively participating in disaster relief efforts, showcasing Sabeco\'s commitment to social responsibility & contributing to the overall well-being of the nation."}\n{"id": "A-3c81088335410307", "claim": "Phát triển cộng đồng Xây dựng hệ thống quản lý rủi ro toàn diện, tích hợp các rủi ro ESG vào quy trình chung.", "evidence": "QLRR Kiểm tra, kiểm toán nội bộ Xác định mục tiêu rõ ràng đề có thể nhận diện và đánh giá rủi ro liên quan đến mục tiêu Nhận diện và phân tích rủi ro Đánh giá rủi ro Các phòng ban, đơn vị Hội đồng Quản lý rủi ro 1 Liên tục nhận diện rủi ro tiềm ẩn trong hoạt động kinh doanh, cập nhật danh mục và đánh giá mức độ ảnh hưởng, khả năng xảy ra theo bộ tiêu chí – thông số rủi ro đã thiết lập, từ đó xây dựng kế hoạch ứng phó rõ ràng, phù hợp, các biện pháp kiểm soát được triển khai chặt chẽ và giám sát rủi ro liên tục, trong đó chú trọng việc xác định rõ vai trò, trách nhiệm của từng phòng ban dựa vào mô hình 3 tuyến phòng thủ giúp phân công trách nhiệm rõ ràng, đảm bảo tất cả các bộ phận đều tham g"}\n{"id": "A-85f6876d3d6959f5", "claim": "Chất thải Chỉ tiêu tiêu thụ nước được xác định dựa trên tổng hóa đơn thanh toán tiền nước hàng tháng tại 2 cơ sở chính: trụ sở Công ty mẹ tại 288 Bis Nguyễn Văn Cừ, Quận Ninh Kiều, TP.", "evidence": "Số liệu được thống kê từ tổng hóa đơn thanh toán điện tại hai địa điểm chính: trụ sở Công ty mẹ tại 288 Bis Nguyễn Văn Cừ, quận Ninh Kiều, TP. Cần Thơ và Chi nhánh Nhà máy Dược phẩm DHG tại Khu công nghiệp Tân Phú Thạnh, huyện Châu Thành A, tỉnh Hậu Giang. : 288 Bis Nguyễn Văn Cừ, P. An Hòa, Q. Ninh Kiều, TP. Cần Thơ Giấy chứng nhận đăng ký kinh doanh và mã số thuế : 1800156801 DHG Pharma Giới thiệu tổng quan"}\n{"id": "A-70abd120b5af6b8e", "claim": "Hiện chúng tôi đanh tích cực tìm kiếm, trao đổi và hoàn thiện danh mục các đối tác xanh (green profile) cho Coteccons như một phần quan trọng để xây dựng chuỗi cung ứng xanh.”", "evidence": "TABLE Phạm Xuân Thu - Giám đốc Khối Thu mua “Chúng tôi nhận thức được việc chọn lựa vật liệu trong xây dựng vô cùng quan trọng, vì vật liệu sẽ có tác động trực tiếp đến môi trường trong suốt vòng đời của một công trình. Do vậy, chúng tôi tự đòi hỏi chính mình phải rất khắt khe và có trách nhiệm để lựa chọn những nguyên vật liệu ít tác động xấu đến môi trường, có khả năng tái chế hoặc nguyên vật liệu tái chế, tiết kiệm được năng lượng và tài nguyên từ nguồn gốc sản xuất và trong quá trình sử dụng. Coteccons cũng ưu tiên nguyên vật liệu địa phương, vật liệu có độ bền cao, vật liệu xanh từ công nghệ mới... Chúng tôi đã có cơ hội gặp gỡ những đối tác chuyên nghiệp với những sản phẩm, giải pháp t"}\n{"id": "A-bcb68bdc0b407656", "claim": "TC (Tận thu/Tái chế) Tổng cộng 232,9 1.046", "evidence": "Trụ sở CTCP Dược Hậu Giang 97,1 415 Công ty CP Công nghệ Môi trường Trái Đất Xanh Tổng cộng 232,9 Chi Nhánh Nhà Máy Dược Phẩm DHG 135,8 631 - Công ty CP Công nghệ Môi trường Trái Đất Xanh BÁO CÁO CỦA BAN TỔNG GIÁM ĐỐC Đội ngũ nhân sự là nền tảng của sự phát triển bền vững. DHG Pharma tập trung xây dựng đội ngũ chất lượng cao, năng động và tận tâm. Đánh giá trách nhiệm môi trường, xã hội, cộng đồng"}\n{"id": "B-f0f40dec9e3ec30f", "claim": "efficiencies, along with focusing on sustainability via managing raw materials and energy consumption in the In May 2023, SABECO officially signed a Memorandum of Understanding with SP Group (Singapore) to deploy Phase 2 of the Rooftop Solar Project at 9 breweries of SABECO with the highest capacity of up to 10.44 MWp.", "evidence": ""}\n{"id": "A-52c799417eb7e069", "claim": "Đây chính là nền tảng quan trọng để Công ty nắm bắt cơ hội thị trường, củng cố vị thế dẫn đầu và hướng tới tăng trưởng bền vững trong dài hạn.", "evidence": "TRAPHACO trưởng kinh tế trong dài hạn của doanh nghiệp kết hợp với các mục tiêu phát triển xã hội và bảo vệ môi trường, từ đó đóng góp cho sự phát triển chung của cộng đồng. Báo Tăng trưởng luôn gắn liền tăng trưởng kinh tế với bảo vệ môi trường và trách nhiệm xã hội cáo này ghi lại những hoạt động quan trọng của Traphaco và các đơn vị thành viên kinh tế của Traphaco (sau đây gọi tắt là “Traphaco”) liên quan đến các vấn đề về bền vững ba yếu tố đóng vai trò nền tảng cho thành công lâu dài của trong năm 2025. Phạm vi báo cáo: Công ty cổ phần Traphaco Đồng thời, Traphaco đẩy nhanh tiến độ các dự án nhà máy theo tiêu chuẩn cao, hướng tới EU-GMP, nhằm nâng chuẩn quản trị sản xuất, giảm thiểu rủi"}\n{"id": "B-69fcf3dbdd31c222", "claim": "tái chế toàn bộ và được chứng nhận FSC - chứng nhận sản phẩm mang lại lợi ích về môi trường, kinh • Hệ thống nhà máy đạt chứng nhận FSSC 22000, ISO 14001, ISO 50001, Organic EU.", "evidence": ""}\n{"id": "A-ef0ccd06571e91af", "claim": "As part of SABECO’s commitment to sustainability and Net-Zero by 2050, we invested in renewable energy solutions to minimize environmental impacts and improve community well-being.", "evidence": "GRI 2-28] Pillar Our Commitment Our Strategy Continuously seeking innovative solutions and initiatives to minimize environmental impact while optimizing the use of resources, raw materials, packaging, energy, and water. Fully complying with environmental protection laws and actively addressing climate change by reducing greenhouse gas emissions In 2024, we have invested in and implemented over 50 solutions and continuous improvement initiatives to save energy, reduce greenhouse gas (GHG) emissions and achieve a green transition in energy use. 0.92 KgCO2e Per hectoliter of beer (Scope 1 and Scope 2) in 2024 compared to 2023 Renewable Energy Projects: We installed rooftop solar power systems a"}\n{"id": "A-c3a9b24dc075ff3c", "claim": "2 6 1 3 4 Phát triển bền vững 5.", "evidence": "4 3 1 6 2 Phát triển bền vững 5. 118 119 Báo cáo thường niên 2024 BỀN VỮNG PHÁT TRIỂN 5 118 Tổng quan báo cáo phát triển bền vững 119 Định hướng chiến lược phát triển bền vững 122 Quản trị phát triển bền vững 124 Gắn kết các bên liên quan"}\n{"id": "B-d2177452117de1eb", "claim": "trên được duy trì cải tiến, đánh giá liên tục trong suốt quá trình hoạt động của các nhà máy từ đó mang lại nhiều lợi ích cho công ty từ việc sử dụng tài nguyên hiệu quả và góp phần bảo vệ Trong năm 2024, Khối Sản xuất có kế hoạch tiếp tục thực hiện mục tiêu trọng", "evidence": ""}\n{"id": "B-8204380c233d3866", "claim": "cạnh tăng trưởng về cả doanh thu lẫn lợi nhuận, Vinamilk còn đảm bảo nguồn cổ tức ổn định và phân bổ nguồn lực cần thiết để thực hiện các cam kết về phát triển bền vững.", "evidence": ""}\n{"id": "A-37c9322ffa67731f", "claim": "Các biện pháp giảm phát thải 14.", "evidence": "Sử dụng vật liệu xây dựng bền vững và thiết kế tối ưu để giảm lượng khí thải carbon Giảm thiểu rác thải bằng cách thực hiện các chương trình tái chế và giảm thiểu chất thải Bảo vệ và bảo tồn tài nguyên thiên nhiên bằng cách thực hiện các biện pháp quản lý nước bền vững Căn cứ vào thực trạng số liệu phát thải, Coteccons tiếp tục nghiên cứu các biện pháp giảm thiểu phát thải và xây dựng kế hoạch, mục tiêu và lộ trình thực hiện, đặc biệt là các hoạt động can thiệp trong chuỗi cung ứng để giảm thiểu dấu chân carbon, tăng hiệu xuất và phát huy đổi mới sáng tạo trong hoạt động giảm phát thải; Lưu trữ và quản lý dữ liệu lượng phát thải của Coteccons, dành cho cả những công trình, dự án đã hoàn thiệ"}\n{"id": "B-7a2a7a2a8525de2e", "claim": "Việc thành lập Tiểu ban ESG được xác nhận bằng Nghị quyết 13B/2022/NQ-HDQT với 7/7 thành viên HĐQT tán thành.", "evidence": ""}\n{"id": "B-0273e8342476d770", "claim": "Các nhà máy được xây dựng mới dành 20% kinh phí đầu tư cho việc xử lý môi trường, cập nhật công nghệ sản xuất hiện đại cùng với việc áp dụng hệ thống quản trị tiên tiến, giảm thiểu tác động môi trường, và tạo ra năng suất lao động cao, hiệu quả kinh doanh từ tiết kiệm năng lượng và giảm thiểu nguồn rác thải.", "evidence": ""}\n{"id": "B-ac0dcc6352bdadad", "claim": "“Được thành lập năm 2007, với tầm nhìn tiên phong trong sự nghiệp xanh hóa nguồn nhiên liệu tại Việt Nam, Công ty CP CNG Việt Nam hiện là đơn vị hàng đầu về cung cấp khí thiên nhiên nén (CNG – Compressed Natural Gas) và khí thiên nhiên hóa lỏng (LNG – Liquefied Natural Gas), chiếm lĩnh >70% thị phần và có đầy đủ cơ sở hạ tầng vận chuyển – phân phối trên khắp cả nước.", "evidence": ""}\n{"id": "B-32e1a23768aa7860", "claim": "Công ty không ghi nhận bất kỳ sự cố liên quan đến việc không tuân thủ giới hạn xả thải • Trong năm 2022, lượng nước thải sản xuất ghi nhận trung bình 10.625 m3/tháng, chủ yếu phát sinh ở các giai đoạn: sản xuất dược phẩm, rửa máy móc thiết bị, vệ sinh nhà xưởng.", "evidence": ""}\n{"id": "B-2b222a6ddb74a7f1", "claim": "Tác động tích cực đến môi trường: Với trên 9 ngàn ha rừng, hàng năm rừng cao su của Công ty hấp thụ một lượng lớn carbon, góp phần giảm thiểu biến đổi khí hậu và tăng nhiệt độ.", "evidence": ""}\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: INPUTS=gsi_layer_a.jsonl,gsi_d_edit.jsonl python -u multirun', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('INPUTS=gsi_layer_a.jsonl,gsi_d_edit.jsonl python -u multirun.py jobs_small.json issue_worker.py')


In [ ]:
print('[cell] run 2: ls -la out; cat out/*_summary.json', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('ls -la out; cat out/*_summary.json')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)